In [ ]:
# ============================================================
# AI_Y2 — MODULE 1 / CELL 1
# EXTRACT + SOURCE DATASET AUDIT
# READ-ONLY AUDIT OF DATA CONTENT
# ============================================================

from pathlib import Path
import zipfile
import shutil
import hashlib
import yaml
from collections import Counter, defaultdict

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
ZIP_PATH = Path("/content/yolo_dataset-20261001T214553Z-1-001.zip")
RAW_ROOT = Path("/content/AI_Y2_MODULE1_RAW")

# Supported image extensions
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("=" * 80)
print("AI_Y2 MODULE 1 — CELL 1")
print("SOURCE EXTRACTION + AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. CHECK ZIP
# ------------------------------------------------------------
assert ZIP_PATH.exists(), f"❌ ZIP not found: {ZIP_PATH}"

print(f"\nZIP:")
print(f"  {ZIP_PATH}")
print(f"  Size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")

# ------------------------------------------------------------
# 2. FRESH EXTRACTION
# ------------------------------------------------------------
if RAW_ROOT.exists():
    shutil.rmtree(RAW_ROOT)

RAW_ROOT.mkdir(parents=True, exist_ok=True)

print("\nExtracting ZIP...")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(RAW_ROOT)

print(f"✅ Extracted to: {RAW_ROOT}")

# ------------------------------------------------------------
# 3. FIND ALL data.yaml FILES
# ------------------------------------------------------------
yaml_files = sorted(RAW_ROOT.rglob("data.yaml"))

print("\n" + "=" * 80)
print("DATASET DISCOVERY")
print("=" * 80)

print(f"Found data.yaml files: {len(yaml_files)}")

for i, y in enumerate(yaml_files, 1):
    print(f"{i:>2}. {y}")

if len(yaml_files) == 0:
    raise RuntimeError("❌ No data.yaml found inside ZIP.")

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def normalize_names(names):
    """
    Convert YAML names field into {class_id: class_name}
    regardless of whether YAML stores list or dict.
    """
    if isinstance(names, list):
        return {i: str(v).strip() for i, v in enumerate(names)}

    if isinstance(names, dict):
        return {int(k): str(v).strip() for k, v in names.items()}

    return {}


def resolve_split_dir(dataset_root, yaml_data, split_name):
    """
    Try YAML path first.
    Then common Roboflow folder layout.
    """

    yaml_keys = {
        "train": ["train"],
        "valid": ["val", "valid"],
        "test": ["test"],
    }

    # YAML-declared path
    for key in yaml_keys[split_name]:
        value = yaml_data.get(key)

        if value:
            p = Path(value)

            if not p.is_absolute():
                p = (dataset_root / p).resolve()

            if p.exists():
                return p

    # Standard Roboflow folder candidates
    candidates = []

    if split_name == "train":
        candidates = [
            dataset_root / "train" / "images",
        ]

    elif split_name == "valid":
        candidates = [
            dataset_root / "valid" / "images",
            dataset_root / "val" / "images",
        ]

    elif split_name == "test":
        candidates = [
            dataset_root / "test" / "images",
        ]

    for p in candidates:
        if p.exists():
            return p

    return None


def get_label_dir(image_dir):
    """
    Standard YOLO:
      split/images
      split/labels
    """
    if image_dir is None:
        return None

    if image_dir.name == "images":
        candidate = image_dir.parent / "labels"
        if candidate.exists():
            return candidate

    # fallback
    candidate = image_dir.parent / "labels"
    if candidate.exists():
        return candidate

    return None


def image_files(folder):
    if folder is None or not folder.exists():
        return []

    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    )


def md5_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# 4. DATASET-BY-DATASET AUDIT
# ------------------------------------------------------------
all_records = []
dataset_summaries = []

print("\n" + "=" * 80)
print("SOURCE DATASET DETAILS")
print("=" * 80)

for dataset_index, yaml_path in enumerate(yaml_files, 1):

    dataset_root = yaml_path.parent

    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f) or {}

    names = normalize_names(data.get("names", {}))

    dataset_name = dataset_root.name

    print("\n" + "-" * 80)
    print(f"DATASET #{dataset_index}")
    print("-" * 80)

    print(f"Folder : {dataset_root}")
    print(f"Name   : {dataset_name}")
    print(f"nc     : {data.get('nc', len(names))}")
    print(f"names  : {names}")

    summary = {
        "dataset": dataset_name,
        "root": str(dataset_root),
        "names": names,
        "splits": {}
    }

    for split in ["train", "valid", "test"]:

        img_dir = resolve_split_dir(dataset_root, data, split)
        lbl_dir = get_label_dir(img_dir)

        imgs = image_files(img_dir)

        class_counts = Counter()

        missing_labels = []
        empty_labels = []
        malformed_lines = []

        for img in imgs:

            label_path = None

            if lbl_dir is not None:
                label_path = lbl_dir / f"{img.stem}.txt"

            if label_path is None or not label_path.exists():

                missing_labels.append(img.name)

            else:

                text = label_path.read_text(
                    encoding="utf-8",
                    errors="ignore"
                ).strip()

                if not text:
                    empty_labels.append(img.name)

                else:
                    for line_number, line in enumerate(
                        text.splitlines(), 1
                    ):

                        parts = line.strip().split()

                        if len(parts) < 7:
                            malformed_lines.append(
                                (img.name, line_number, line)
                            )
                            continue

                        try:
                            cls_id = int(float(parts[0]))
                            coords = [float(x) for x in parts[1:]]

                            # Segmentation polygon requires x/y pairs
                            if len(coords) < 6 or len(coords) % 2 != 0:
                                malformed_lines.append(
                                    (img.name, line_number, line)
                                )
                                continue

                            class_counts[cls_id] += 1

                        except Exception:
                            malformed_lines.append(
                                (img.name, line_number, line)
                            )

            all_records.append({
                "dataset": dataset_name,
                "dataset_root": str(dataset_root),
                "split": split,
                "image": img,
                "label": label_path,
            })

        summary["splits"][split] = {
            "images": len(imgs),
            "class_counts": dict(class_counts),
            "missing_labels": len(missing_labels),
            "empty_labels": len(empty_labels),
            "malformed": len(malformed_lines),
        }

        print(f"\n{split.upper():5}")
        print(f"  images          : {len(imgs)}")
        print(f"  label dir       : {lbl_dir}")
        print(f"  object counts   : {dict(class_counts)}")
        print(f"  missing labels  : {len(missing_labels)}")
        print(f"  empty labels    : {len(empty_labels)}")
        print(f"  malformed lines : {len(malformed_lines)}")

        if empty_labels:
            print("  Empty label examples:")
            for x in empty_labels[:10]:
                print(f"    - {x}")

        if missing_labels:
            print("  Missing label examples:")
            for x in missing_labels[:10]:
                print(f"    - {x}")

        if malformed_lines:
            print("  Malformed examples:")
            for x in malformed_lines[:5]:
                print(f"    - {x}")

    dataset_summaries.append(summary)


# ------------------------------------------------------------
# 5. GLOBAL COUNTS
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("GLOBAL IMAGE COUNTS")
print("=" * 80)

global_split_counts = Counter()
global_dataset_counts = Counter()

for r in all_records:
    global_split_counts[r["split"]] += 1
    global_dataset_counts[r["dataset"]] += 1

print("\nBy split:")
for split in ["train", "valid", "test"]:
    print(f"  {split:5}: {global_split_counts[split]}")

print(f"\nTOTAL : {len(all_records)}")

print("\nBy dataset:")
for name, count in global_dataset_counts.items():
    print(f"  {name}: {count}")


# ------------------------------------------------------------
# 6. EXACT DUPLICATE IMAGE AUDIT
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("EXACT DUPLICATE IMAGE AUDIT (MD5)")
print("=" * 80)

hash_groups = defaultdict(list)

for i, r in enumerate(all_records, 1):

    img = r["image"]

    try:
        h = md5_file(img)
        hash_groups[h].append(r)

    except Exception as e:
        print(f"⚠️ Could not hash {img}: {e}")

duplicate_groups = {
    h: rows
    for h, rows in hash_groups.items()
    if len(rows) > 1
}

cross_split_groups = []
cross_dataset_groups = []

for h, rows in duplicate_groups.items():

    splits = {r["split"] for r in rows}
    datasets = {r["dataset"] for r in rows}

    if len(splits) > 1:
        cross_split_groups.append((h, rows))

    if len(datasets) > 1:
        cross_dataset_groups.append((h, rows))


print(f"Duplicate MD5 groups       : {len(duplicate_groups)}")
print(f"Cross-split duplicate groups: {len(cross_split_groups)}")
print(f"Cross-dataset dup groups    : {len(cross_dataset_groups)}")


if duplicate_groups:

    print("\nDuplicate group details:")

    for group_num, (h, rows) in enumerate(
        duplicate_groups.items(), 1
    ):

        print("\n" + "-" * 60)
        print(f"GROUP {group_num}")
        print(f"MD5: {h}")

        for r in rows:
            print(
                f"  dataset={r['dataset']} | "
                f"split={r['split']} | "
                f"image={r['image'].name}"
            )


# ------------------------------------------------------------
# 7. FINAL IMPORTANT SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("CELL 1 FINAL SUMMARY")
print("=" * 80)

total_missing = 0
total_empty = 0
total_malformed = 0

for d in dataset_summaries:
    for s in d["splits"].values():
        total_missing += s["missing_labels"]
        total_empty += s["empty_labels"]
        total_malformed += s["malformed"]

print(f"""
Datasets discovered       : {len(yaml_files)}
Total images              : {len(all_records)}

Train images              : {global_split_counts['train']}
Valid images              : {global_split_counts['valid']}
Test images               : {global_split_counts['test']}

Missing label files       : {total_missing}
Empty label files         : {total_empty}
Malformed polygon lines   : {total_malformed}

Exact duplicate groups    : {len(duplicate_groups)}
Cross-split dup groups    : {len(cross_split_groups)}
Cross-dataset dup groups  : {len(cross_dataset_groups)}

TARGET Y2 FINAL CLASSES:
    0 = processing_area
    1 = lesion

⚠️ No merging/remapping/splitting has been performed yet.
This cell only audited the source ZIP.
""")

print("=" * 80)
print("✅ CELL 1 COMPLETE")
print("=" * 80)

AI_Y2 MODULE 1 — CELL 1
SOURCE EXTRACTION + AUDIT

ZIP:
  /content/yolo_dataset-20261001T214553Z-1-001.zip
  Size: 45.10 MB

Extracting ZIP...
✅ Extracted to: /content/AI_Y2_MODULE1_RAW

DATASET DISCOVERY
Found data.yaml files: 5
 1. /content/AI_Y2_MODULE1_RAW/yolo_dataset/cancer_detection.v2i.yolov11/data.yaml
 2. /content/AI_Y2_MODULE1_RAW/yolo_dataset/leukoplakia_detection.v2i.yolov11/data.yaml
 3. /content/AI_Y2_MODULE1_RAW/yolo_dataset/leukoplakia_like_detection.v3i.yolov11/data.yaml
 4. /content/AI_Y2_MODULE1_RAW/yolo_dataset/processing_area.v3i.yolov11/data.yaml
 5. /content/AI_Y2_MODULE1_RAW/yolo_dataset/ulcer_detection.v4i.yolov11/data.yaml

SOURCE DATASET DETAILS

--------------------------------------------------------------------------------
DATASET #1
--------------------------------------------------------------------------------
Folder : /content/AI_Y2_MODULE1_RAW/yolo_dataset/cancer_detection.v2i.yolov11
Name   : cancer_detection.v2i.yolov11
nc     : 2
names  : {0: 'can

In [ ]:
# ============================================================
# AI_Y2 — MODULE 1 / CELL 2
# CLEAN + SEMANTIC REMAP + LINEAGE/GROUP AUDIT
#
# FINAL Y1-COMPATIBLE CONTRACT:
#   0 = processing_area
#   1 = lesion
# ============================================================

from pathlib import Path
import re
import csv
import json
import yaml
import hashlib
import shutil
from collections import Counter, defaultdict

RAW_ROOT = Path("/content/AI_Y2_MODULE1_RAW")
WORK_ROOT = Path("/content/AI_Y2_MODULE1_WORK")

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

WORK_ROOT.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ------------------------------------------------------------
# APPROVED TARGET MAPPING — Y1 COMPATIBLE
# ------------------------------------------------------------

TARGET_NAMES = {
    0: "processing_area",
    1: "lesion",
}

# Explicit whitelist.
# Unknown classes MUST fail instead of silently becoming lesion.
CLASS_NAME_TO_TARGET = {
    "processing_area": 0,

    "cancer-detection": 1,
    "leukoplakia-detection": 1,
    "leukoplakia-like-detection": 1,
    "ulcer-detection": 1,
}


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def normalize_yaml_names(names):
    if isinstance(names, list):
        return {i: str(v).strip() for i, v in enumerate(names)}

    if isinstance(names, dict):
        return {int(k): str(v).strip() for k, v in names.items()}

    raise ValueError(f"Unsupported YAML names format: {type(names)}")


def resolve_split_image_dir(dataset_root, yaml_data, split):
    yaml_keys = {
        "train": ["train"],
        "valid": ["val", "valid"],
        "test": ["test"],
    }

    for key in yaml_keys[split]:
        value = yaml_data.get(key)

        if value:
            p = Path(value)
            if not p.is_absolute():
                p = (dataset_root / p).resolve()

            if p.exists():
                return p

    candidates = {
        "train": [dataset_root / "train" / "images"],
        "valid": [
            dataset_root / "valid" / "images",
            dataset_root / "val" / "images",
        ],
        "test": [dataset_root / "test" / "images"],
    }

    for p in candidates[split]:
        if p.exists():
            return p

    return None


def get_label_dir(img_dir):
    if img_dir is None:
        return None

    p = img_dir.parent / "labels"

    return p if p.exists() else None


def md5_file(path):
    h = hashlib.md5()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def canonical_source_name(filename):
    """
    Conservative lineage normalization.

    Examples:
    aug_1_C126_jpeg.rf.HASH.jpg
       -> c126

    Copy-of-1M-153-_jpg.rf.HASH.jpg
       -> 1m-153

    Roboflow hash is removed.
    Common augmentation/copy prefixes are removed.

    This is ONLY a grouping candidate, not a diagnosis label.
    """

    name = Path(filename).stem.lower()

    # Remove Roboflow .rf.<hash>
    name = re.sub(r"\.rf\.[0-9a-f]+$", "", name)

    # Remove common repeated copy / augmentation prefixes
    changed = True

    while changed:
        old = name

        name = re.sub(
            r"^(copy[-_ ]?of[-_ ]?|copy[-_ ]+)",
            "",
            name,
            flags=re.I,
        )

        name = re.sub(
            r"^(aug(?:mented)?[-_ ]*\d*[-_ ]+)",
            "",
            name,
            flags=re.I,
        )

        name = re.sub(
            r"^(image[-_ ]copy[-_ ]*)",
            "",
            name,
            flags=re.I,
        )

        changed = (old != name)

    # Remove exported file-type tokens
    name = re.sub(
        r"([_-](jpg|jpeg|png|bmp|webp))+$",
        "",
        name,
        flags=re.I,
    )

    # Normalize separators
    name = re.sub(r"[\s_]+", "-", name)
    name = re.sub(r"-+", "-", name)

    return name.strip("-")


def parse_and_remap_label(label_path, raw_names):
    """
    Converts source classes to:

      0 processing_area
      1 lesion

    Returns remapped YOLO segmentation lines + counts.
    """

    text = label_path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).strip()

    if not text:
        return None, Counter(), "empty"

    output_lines = []
    counts = Counter()

    for line_no, line in enumerate(text.splitlines(), 1):

        parts = line.strip().split()

        if len(parts) < 7:
            return None, Counter(), f"malformed_line_{line_no}"

        try:
            raw_id = int(float(parts[0]))
            coords = [float(x) for x in parts[1:]]

        except Exception:
            return None, Counter(), f"parse_error_line_{line_no}"

        if raw_id not in raw_names:
            return (
                None,
                Counter(),
                f"unknown_raw_class_id_{raw_id}"
            )

        raw_class_name = raw_names[raw_id].strip()

        if raw_class_name not in CLASS_NAME_TO_TARGET:
            return (
                None,
                Counter(),
                f"unknown_class_name_{raw_class_name}"
            )

        if len(coords) < 6 or len(coords) % 2 != 0:
            return (
                None,
                Counter(),
                f"invalid_polygon_line_{line_no}"
            )

        # Coordinates must be finite and normalized
        if any((x < 0 or x > 1) for x in coords):
            return (
                None,
                Counter(),
                f"coordinate_out_of_range_line_{line_no}"
            )

        target_id = CLASS_NAME_TO_TARGET[raw_class_name]

        # fixed formatting for reproducibility
        coord_txt = " ".join(f"{x:.8f}" for x in coords)

        output_lines.append(
            f"{target_id} {coord_txt}"
        )

        counts[target_id] += 1

    return output_lines, counts, None


def label_signature(lines):
    """
    Compare remapped annotations independently of line order.
    """
    if not lines:
        return None

    return hashlib.md5(
        "\n".join(sorted(lines)).encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# UNION-FIND FOR LINEAGE GROUPS
# ------------------------------------------------------------

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)

        if ra == rb:
            return

        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra

        self.parent[rb] = ra

        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


# ------------------------------------------------------------
# DISCOVER + REMAP ALL ELIGIBLE IMAGES
# ------------------------------------------------------------

yaml_files = sorted(RAW_ROOT.rglob("data.yaml"))

assert len(yaml_files) == 5, \
    f"Expected 5 datasets, found {len(yaml_files)}"


records = []
excluded = []

print("=" * 88)
print("AI_Y2 MODULE 1 — CELL 2")
print("CLEAN + Y1-COMPATIBLE REMAP + LINEAGE AUDIT")
print("=" * 88)


for yaml_path in yaml_files:

    dataset_root = yaml_path.parent
    dataset_name = dataset_root.name

    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f) or {}

    raw_names = normalize_yaml_names(data.get("names", {}))

    print("\n" + "-" * 88)
    print(dataset_name)
    print(f"RAW classes: {raw_names}")

    # Show actual conversion
    conversion = {}

    for cid, cname in raw_names.items():
        if cname not in CLASS_NAME_TO_TARGET:
            raise RuntimeError(
                f"❌ Unknown class '{cname}' in {dataset_name}"
            )

        conversion[cid] = (
            CLASS_NAME_TO_TARGET[cname],
            TARGET_NAMES[CLASS_NAME_TO_TARGET[cname]]
        )

    print(f"REMAPPING : {conversion}")

    for split in ["train", "valid", "test"]:

        img_dir = resolve_split_image_dir(
            dataset_root,
            data,
            split
        )

        assert img_dir is not None, \
            f"Image directory missing: {dataset_name}/{split}"

        lbl_dir = get_label_dir(img_dir)

        assert lbl_dir is not None, \
            f"Label directory missing: {dataset_name}/{split}"

        images = sorted(
            x for x in img_dir.iterdir()
            if x.is_file() and x.suffix.lower() in IMG_EXTS
        )

        for img in images:

            label = lbl_dir / f"{img.stem}.txt"

            if not label.exists():
                excluded.append({
                    "dataset": dataset_name,
                    "split": split,
                    "image": img.name,
                    "reason": "missing_label"
                })
                continue

            remapped_lines, counts, error = parse_and_remap_label(
                label,
                raw_names
            )

            if error is not None:
                excluded.append({
                    "dataset": dataset_name,
                    "split": split,
                    "image": img.name,
                    "reason": error
                })
                continue

            rec = {
                "dataset": dataset_name,
                "dataset_root": str(dataset_root),

                "original_split": split,

                "image_path": str(img),
                "label_path": str(label),
                "image_name": img.name,

                "canonical_name": canonical_source_name(img.name),

                "md5": md5_file(img),

                "processing_objects": int(counts[0]),
                "lesion_objects": int(counts[1]),

                "has_processing": int(counts[0] > 0),
                "has_lesion": int(counts[1] > 0),

                "remapped_lines": remapped_lines,
                "label_signature": label_signature(remapped_lines),
            }

            records.append(rec)


# ------------------------------------------------------------
# SUMMARY AFTER CLEANING
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("CLEAN ELIGIBLE DATA")
print("=" * 88)

print(f"Original images : {1166}")
print(f"Eligible images : {len(records)}")
print(f"Excluded images : {len(excluded)}")

print("\nExcluded reasons:")

for reason, n in Counter(
    x["reason"] for x in excluded
).items():
    print(f"  {reason}: {n}")

if excluded:
    print("\nExcluded files:")

    for x in excluded:
        print(
            f"  {x['dataset']} | "
            f"{x['split']} | "
            f"{x['image']} | "
            f"{x['reason']}"
        )


# ------------------------------------------------------------
# CLASS / SOURCE SUMMARY AFTER REMAP
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("POST-REMAP DISTRIBUTION")
print("=" * 88)

by_dataset = defaultdict(lambda: Counter())

for r in records:
    d = by_dataset[r["dataset"]]

    d["images"] += 1
    d["processing_objects"] += r["processing_objects"]
    d["lesion_objects"] += r["lesion_objects"]
    d["lesion_positive_images"] += r["has_lesion"]


for dataset, d in sorted(by_dataset.items()):

    print(f"\n{dataset}")

    print(f"  images                 : {d['images']}")
    print(f"  processing objects     : {d['processing_objects']}")
    print(f"  lesion objects         : {d['lesion_objects']}")
    print(f"  lesion-positive images : {d['lesion_positive_images']}")


total_processing = sum(r["processing_objects"] for r in records)
total_lesions = sum(r["lesion_objects"] for r in records)
total_lesion_positive = sum(r["has_lesion"] for r in records)

print("\nGLOBAL")
print(f"  eligible images        : {len(records)}")
print(f"  processing objects     : {total_processing}")
print(f"  lesion objects         : {total_lesions}")
print(f"  lesion-positive images : {total_lesion_positive}")


# ------------------------------------------------------------
# BUILD LINEAGE GROUPS
#
# Group if:
#   A) exact same image MD5
#   OR
#   B) same normalized source filename
#
# This is intentionally conservative against leakage.
# ------------------------------------------------------------

uf = UnionFind(len(records))

md5_first = {}
name_first = {}

for i, r in enumerate(records):

    h = r["md5"]

    if h in md5_first:
        uf.union(i, md5_first[h])
    else:
        md5_first[h] = i

    cname = r["canonical_name"]

    if cname:
        if cname in name_first:
            uf.union(i, name_first[cname])
        else:
            name_first[cname] = i


root_to_indices = defaultdict(list)

for i in range(len(records)):
    root_to_indices[uf.find(i)].append(i)


groups = list(root_to_indices.values())

# deterministic group IDs
groups = sorted(
    groups,
    key=lambda g: min(records[i]["image_path"] for i in g)
)

for group_number, indices in enumerate(groups, 1):

    gid = f"G{group_number:05d}"

    for i in indices:
        records[i]["group_id"] = gid


# ------------------------------------------------------------
# GROUP STATISTICS
# ------------------------------------------------------------

group_sizes = Counter(len(g) for g in groups)

cross_split_groups = []
cross_dataset_groups = []
cross_category_names = []

for indices in groups:

    rows = [records[i] for i in indices]

    splits = {r["original_split"] for r in rows}
    datasets = {r["dataset"] for r in rows}

    if len(splits) > 1:
        cross_split_groups.append(rows)

    if len(datasets) > 1:
        cross_dataset_groups.append(rows)


print("\n" + "=" * 88)
print("LINEAGE / GROUP AUDIT")
print("=" * 88)

print(f"Eligible images        : {len(records)}")
print(f"Independent groups     : {len(groups)}")
print(f"Cross-original-split   : {len(cross_split_groups)}")
print(f"Cross-source-dataset   : {len(cross_dataset_groups)}")

print("\nGroup size distribution:")

for size, count in sorted(group_sizes.items()):
    print(f"  size {size}: {count} groups")


# ------------------------------------------------------------
# EXACT MD5 DUPLICATE ANALYSIS AFTER REMAP
# ------------------------------------------------------------

md5_groups = defaultdict(list)

for r in records:
    md5_groups[r["md5"]].append(r)

exact_dups = {
    h: rows
    for h, rows in md5_groups.items()
    if len(rows) > 1
}

print("\n" + "=" * 88)
print("EXACT DUPLICATE LABEL CONSISTENCY")
print("=" * 88)

print(f"Exact duplicate groups: {len(exact_dups)}")

for num, (h, rows) in enumerate(exact_dups.items(), 1):

    signatures = {r["label_signature"] for r in rows}

    print("\n" + "-" * 70)
    print(f"Exact duplicate group {num}")
    print(f"MD5: {h}")
    print(f"Remapped labels identical: {len(signatures) == 1}")

    for r in rows:

        print(
            f"  {r['dataset']} | "
            f"{r['original_split']} | "
            f"{r['image_name']} | "
            f"P={r['processing_objects']} "
            f"L={r['lesion_objects']}"
        )


# ------------------------------------------------------------
# SHOW LARGEST LINEAGE GROUPS
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("LARGEST LINEAGE GROUPS")
print("=" * 88)

largest_groups = sorted(
    groups,
    key=len,
    reverse=True
)[:20]

for indices in largest_groups:

    rows = [records[i] for i in indices]

    if len(rows) <= 1:
        break

    print(
        f"\n{rows[0]['group_id']} | "
        f"size={len(rows)} | "
        f"canonical={rows[0]['canonical_name']}"
    )

    for r in rows:

        print(
            f"  {r['dataset']} | "
            f"{r['original_split']} | "
            f"{r['image_name']}"
        )


# ------------------------------------------------------------
# SAVE AUDIT MANIFEST
# ------------------------------------------------------------

manifest_csv = WORK_ROOT / "y2_clean_lineage_manifest.csv"

fieldnames = [
    "group_id",
    "dataset",
    "original_split",
    "image_name",
    "image_path",
    "label_path",
    "canonical_name",
    "md5",
    "processing_objects",
    "lesion_objects",
    "has_processing",
    "has_lesion",
    "label_signature",
]

with open(manifest_csv, "w", newline="", encoding="utf-8") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()

    for r in records:
        writer.writerow({
            k: r[k]
            for k in fieldnames
        })


excluded_csv = WORK_ROOT / "excluded_images.csv"

with open(excluded_csv, "w", newline="", encoding="utf-8") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=["dataset", "split", "image", "reason"]
    )

    writer.writeheader()
    writer.writerows(excluded)


# Save remapped annotations separately for Cell 3
remap_json = WORK_ROOT / "remapped_annotations.json"

serializable = {}

for i, r in enumerate(records):
    serializable[str(i)] = {
        "group_id": r["group_id"],
        "image_path": r["image_path"],
        "dataset": r["dataset"],
        "original_split": r["original_split"],
        "image_name": r["image_name"],
        "canonical_name": r["canonical_name"],
        "md5": r["md5"],
        "processing_objects": r["processing_objects"],
        "lesion_objects": r["lesion_objects"],
        "has_processing": r["has_processing"],
        "has_lesion": r["has_lesion"],
        "remapped_lines": r["remapped_lines"],
    }

with open(remap_json, "w", encoding="utf-8") as f:
    json.dump(serializable, f, indent=2)


# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("CELL 2 FINAL SUMMARY")
print("=" * 88)

print(f"""
Y1-COMPATIBLE FINAL CLASS CONTRACT
----------------------------------
0 = processing_area
1 = lesion

Original images          : 1166
Eligible clean images    : {len(records)}
Excluded                 : {len(excluded)}

Processing objects       : {total_processing}
Lesion objects           : {total_lesions}
Lesion-positive images   : {total_lesion_positive}

Independent lineage groups : {len(groups)}
Cross-original-split groups: {len(cross_split_groups)}
Cross-source groups        : {len(cross_dataset_groups)}

Exact MD5 duplicate groups : {len(exact_dups)}

Manifest:
{manifest_csv}

Remapped annotation cache:
{remap_json}

No final Train / Valid / Test split has been created yet.
""")

print("=" * 88)
print("✅ CELL 2 COMPLETE")
print("=" * 88)

AI_Y2 MODULE 1 — CELL 2
CLEAN + Y1-COMPATIBLE REMAP + LINEAGE AUDIT

----------------------------------------------------------------------------------------
cancer_detection.v2i.yolov11
RAW classes: {0: 'cancer-detection', 1: 'processing_area'}
REMAPPING : {0: (1, 'lesion'), 1: (0, 'processing_area')}

----------------------------------------------------------------------------------------
leukoplakia_detection.v2i.yolov11
RAW classes: {0: 'leukoplakia-detection', 1: 'processing_area'}
REMAPPING : {0: (1, 'lesion'), 1: (0, 'processing_area')}

----------------------------------------------------------------------------------------
leukoplakia_like_detection.v3i.yolov11
RAW classes: {0: 'leukoplakia-like-detection', 1: 'processing_area'}
REMAPPING : {0: (1, 'lesion'), 1: (0, 'processing_area')}

----------------------------------------------------------------------------------------
processing_area.v3i.yolov11
RAW classes: {0: 'processing_area'}
REMAPPING : {0: (0, 'processing_area')}


In [ ]:
# ============================================================
# AI_Y2 — MODULE 1 / CELL 3
#
# FINAL CLEAN DATASET BUILD
# - quality cleanup
# - refined lineage grouping
# - group-aware balanced 75/15/10 split
# - Y1-compatible classes
# - source-prefixed filenames
# - strict leakage verification
#
# FINAL CLASSES:
#   0 = processing_area
#   1 = lesion
# ============================================================

from pathlib import Path
import json
import csv
import re
import shutil
import hashlib
import random
from collections import Counter, defaultdict

import cv2
import numpy as np
import yaml


# ============================================================
# CONFIG
# ============================================================

WORK_ROOT = Path("/content/AI_Y2_MODULE1_WORK")
CACHE_JSON = WORK_ROOT / "remapped_annotations.json"

FINAL_ROOT = Path("/content/AI_Y2_FINAL")

SEED = 42

SPLIT_FRACTIONS = {
    "train": 0.75,
    "valid": 0.15,
    "test":  0.10,
}

# Cross-dataset same-name images are grouped ONLY when visually
# very similar. Lower = stricter.
PHASH_MAX_DISTANCE = 6

TARGET_NAMES = {
    0: "processing_area",
    1: "lesion",
}


# ============================================================
# SOURCE NAMES
# ============================================================

SOURCE_SHORT = {
    "cancer_detection.v2i.yolov11": "cancer",
    "leukoplakia_detection.v2i.yolov11": "leukoplakia",
    "leukoplakia_like_detection.v3i.yolov11": "leukoplakia_like",
    "processing_area.v3i.yolov11": "processing_area",
    "ulcer_detection.v4i.yolov11": "ulcer",
}

DISEASE_SOURCES = {
    "cancer_detection.v2i.yolov11",
    "leukoplakia_detection.v2i.yolov11",
    "leukoplakia_like_detection.v3i.yolov11",
    "ulcer_detection.v4i.yolov11",
}

PROCESSING_SOURCE = "processing_area.v3i.yolov11"


# ============================================================
# HELPERS
# ============================================================

def md5_file(path):
    h = hashlib.md5()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def annotation_signature(lines):
    """
    Order-independent annotation signature.
    """
    return hashlib.md5(
        "\n".join(sorted(lines)).encode("utf-8")
    ).hexdigest()


def phash64(image_path):
    """
    Simple 64-bit perceptual hash using OpenCV DCT.
    """

    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise RuntimeError(f"Unreadable image: {image_path}")

    img = cv2.resize(
        img,
        (32, 32),
        interpolation=cv2.INTER_AREA
    ).astype(np.float32)

    dct = cv2.dct(img)

    low = dct[:8, :8].copy()

    vals = low.flatten()

    # Ignore DC coefficient for threshold
    median = np.median(vals[1:])

    bits = vals > median

    value = 0

    for bit in bits:
        value = (value << 1) | int(bit)

    return int(value)


def hamming64(a, b):
    return (int(a) ^ int(b)).bit_count()


def safe_filename(text):
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text.strip("_")


class UnionFind:

    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):

        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])

        return self.parent[x]

    def union(self, a, b):

        ra = self.find(a)
        rb = self.find(b)

        if ra == rb:
            return

        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra

        self.parent[rb] = ra

        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


# ============================================================
# LOAD CELL 2 CACHE
# ============================================================

assert CACHE_JSON.exists(), f"Missing: {CACHE_JSON}"

with open(CACHE_JSON, "r", encoding="utf-8") as f:
    cache = json.load(f)

records = []

for _, x in cache.items():

    r = dict(x)

    r["image_path"] = Path(r["image_path"])

    r["annotation_signature"] = annotation_signature(
        r["remapped_lines"]
    )

    records.append(r)


print("=" * 92)
print("AI_Y2 MODULE 1 — CELL 3")
print("FINAL CLEAN + BALANCED GROUP-AWARE DATASET BUILD")
print("=" * 92)

print(f"\nInput records from Cell 2: {len(records)}")


# ============================================================
# STEP 1 — LABEL COMPLETENESS QUALITY FILTER
# ============================================================

quality_excluded = []
quality_kept = []

for r in records:

    dataset = r["dataset"]

    # Every retained image should have processing_area annotation
    if not r["has_processing"]:

        quality_excluded.append({
            **r,
            "exclude_reason": "missing_processing_area_annotation"
        })

        continue

    # Disease-source image should also have lesion annotation.
    #
    # Otherwise it may be an incompletely annotated disease image.
    if dataset in DISEASE_SOURCES and not r["has_lesion"]:

        quality_excluded.append({
            **r,
            "exclude_reason": "disease_source_without_lesion_annotation"
        })

        continue

    # Processing-only source should contain no lesion annotation
    if dataset == PROCESSING_SOURCE and r["has_lesion"]:

        quality_excluded.append({
            **r,
            "exclude_reason": "processing_source_contains_lesion"
        })

        continue

    quality_kept.append(r)


records = quality_kept

print("\n" + "=" * 92)
print("QUALITY COMPLETENESS FILTER")
print("=" * 92)

print(f"Kept     : {len(records)}")
print(f"Excluded : {len(quality_excluded)}")

reason_counts = Counter(
    x["exclude_reason"]
    for x in quality_excluded
)

for reason, count in reason_counts.items():
    print(f"  {reason}: {count}")

if quality_excluded:

    print("\nExcluded examples:")

    for x in quality_excluded[:20]:

        print(
            f"  {x['dataset']} | "
            f"{x['original_split']} | "
            f"{x['image_name']} | "
            f"{x['exclude_reason']}"
        )


# ============================================================
# STEP 2 — EXACT DUPLICATE CONFLICT CHECK
# ============================================================

md5_groups = defaultdict(list)

for i, r in enumerate(records):

    # Cell 2 already stored MD5
    md5_groups[r["md5"]].append(i)


ambiguous_indices = set()

print("\n" + "=" * 92)
print("EXACT DUPLICATE ANNOTATION CONFLICT CHECK")
print("=" * 92)

exact_duplicate_groups = 0
exact_conflict_groups = 0

for md5, indices in md5_groups.items():

    if len(indices) <= 1:
        continue

    exact_duplicate_groups += 1

    signatures = {
        records[i]["annotation_signature"]
        for i in indices
    }

    if len(signatures) > 1:

        exact_conflict_groups += 1

        print("\n⚠️ Exact-image annotation conflict:")
        print(f"MD5: {md5}")

        for i in indices:

            r = records[i]

            print(
                f"  {r['dataset']} | "
                f"{r['original_split']} | "
                f"{r['image_name']} | "
                f"P={r['processing_objects']} | "
                f"L={r['lesion_objects']}"
            )

            ambiguous_indices.add(i)


print(f"\nExact duplicate groups : {exact_duplicate_groups}")
print(f"Conflicting groups     : {exact_conflict_groups}")
print(f"Images excluded        : {len(ambiguous_indices)}")


ambiguous_exact = []

clean_after_exact = []

for i, r in enumerate(records):

    if i in ambiguous_indices:

        ambiguous_exact.append({
            **r,
            "exclude_reason": "exact_duplicate_annotation_conflict"
        })

    else:
        clean_after_exact.append(r)

records = clean_after_exact


# ============================================================
# STEP 3 — COMPUTE PERCEPTUAL HASHES
# ============================================================

print("\n" + "=" * 92)
print("PERCEPTUAL IMAGE HASHING")
print("=" * 92)

for idx, r in enumerate(records, 1):

    r["phash"] = phash64(r["image_path"])

    if idx % 200 == 0 or idx == len(records):
        print(f"Hashed {idx}/{len(records)}")


# ============================================================
# STEP 4 — REFINED LINEAGE GROUPING
#
# UNION WHEN:
#
# 1. Exact MD5 same
#
# OR
#
# 2. Same dataset + same canonical filename
#
# OR
#
# 3. Different datasets:
#    same canonical filename
#    AND perceptual hash distance <= threshold
#
# Generic numeric filename alone is NOT enough across datasets.
# ============================================================

uf = UnionFind(len(records))


# ----- exact MD5 -----

first_md5 = {}

for i, r in enumerate(records):

    h = r["md5"]

    if h in first_md5:
        uf.union(i, first_md5[h])

    else:
        first_md5[h] = i


# ----- canonical candidates -----

canonical_groups = defaultdict(list)

for i, r in enumerate(records):

    canonical_groups[r["canonical_name"]].append(i)


cross_dataset_visual_matches = []

for canonical, indices in canonical_groups.items():

    if not canonical:
        continue

    # --------------------------------------------------------
    # SAME DATASET
    # --------------------------------------------------------

    dataset_map = defaultdict(list)

    for i in indices:
        dataset_map[records[i]["dataset"]].append(i)

    for dataset, same_source_indices in dataset_map.items():

        if len(same_source_indices) > 1:

            base = same_source_indices[0]

            for other in same_source_indices[1:]:
                uf.union(base, other)


    # --------------------------------------------------------
    # CROSS DATASET
    # visual confirmation required
    # --------------------------------------------------------

    if len(dataset_map) > 1:

        for a_pos in range(len(indices)):

            for b_pos in range(a_pos + 1, len(indices)):

                ia = indices[a_pos]
                ib = indices[b_pos]

                a = records[ia]
                b = records[ib]

                if a["dataset"] == b["dataset"]:
                    continue

                dist = hamming64(
                    a["phash"],
                    b["phash"]
                )

                if dist <= PHASH_MAX_DISTANCE:

                    uf.union(ia, ib)

                    cross_dataset_visual_matches.append(
                        (
                            canonical,
                            dist,
                            ia,
                            ib
                        )
                    )


# ============================================================
# MATERIALIZE REFINED GROUPS
# ============================================================

root_map = defaultdict(list)

for i in range(len(records)):
    root_map[uf.find(i)].append(i)

groups = list(root_map.values())


# deterministic order
groups = sorted(
    groups,
    key=lambda inds:
        min(str(records[i]["image_path"]) for i in inds)
)


for gid_num, inds in enumerate(groups, 1):

    gid = f"Y2G{gid_num:05d}"

    for i in inds:
        records[i]["refined_group_id"] = gid


print("\n" + "=" * 92)
print("REFINED LINEAGE RESULTS")
print("=" * 92)

print(f"Images                 : {len(records)}")
print(f"Refined groups         : {len(groups)}")
print(
    f"Confirmed cross-dataset visual matches: "
    f"{len(cross_dataset_visual_matches)}"
)


group_size_counts = Counter(len(x) for x in groups)

print("\nGroup-size distribution:")

for size, n in sorted(group_size_counts.items()):
    print(f"  size {size}: {n}")


if cross_dataset_visual_matches:

    print("\nCross-dataset visually-supported matches:")

    shown = set()

    for canonical, dist, ia, ib in cross_dataset_visual_matches:

        key = tuple(
            sorted([
                records[ia]["refined_group_id"],
                records[ib]["refined_group_id"]
            ])
        )

        if key in shown:
            continue

        shown.add(key)

        print(
            f"\n  canonical={canonical} | "
            f"pHash distance={dist}"
        )

        print(
            f"    {records[ia]['dataset']} | "
            f"{records[ia]['image_name']}"
        )

        print(
            f"    {records[ib]['dataset']} | "
            f"{records[ib]['image_name']}"
        )


# ============================================================
# STEP 5 — DETECT TRUE CROSS-SOURCE SEMANTIC CONFLICTS
#
# If visually linked lineage contains:
#
#   disease lesion-positive sample
#   AND
#   processing-only lesion-negative sample
#
# then supervision is contradictory for same lineage.
#
# Exclude entire lineage rather than guess.
# ============================================================

conflict_group_ids = set()

for inds in groups:

    rows = [records[i] for i in inds]

    datasets = {r["dataset"] for r in rows}

    if len(datasets) <= 1:
        continue

    has_disease_source = any(
        r["dataset"] in DISEASE_SOURCES
        for r in rows
    )

    has_processing_source = any(
        r["dataset"] == PROCESSING_SOURCE
        for r in rows
    )

    lesion_states = {
        int(r["has_lesion"])
        for r in rows
    }

    if (
        has_disease_source
        and has_processing_source
        and len(lesion_states) > 1
    ):
        conflict_group_ids.add(
            rows[0]["refined_group_id"]
        )


semantic_conflicts = []

final_clean_records = []

for r in records:

    if r["refined_group_id"] in conflict_group_ids:

        semantic_conflicts.append({
            **r,
            "exclude_reason":
                "cross_source_visual_annotation_conflict"
        })

    else:
        final_clean_records.append(r)

records = final_clean_records


print("\n" + "=" * 92)
print("CROSS-SOURCE SEMANTIC CONFLICT CHECK")
print("=" * 92)

print(f"Conflict lineage groups: {len(conflict_group_ids)}")
print(f"Images excluded        : {len(semantic_conflicts)}")

if semantic_conflicts:

    grouped_conflicts = defaultdict(list)

    for r in semantic_conflicts:
        grouped_conflicts[r["refined_group_id"]].append(r)

    for gid, rows in list(grouped_conflicts.items())[:20]:

        print(f"\n{gid}")

        for r in rows:

            print(
                f"  {r['dataset']} | "
                f"{r['image_name']} | "
                f"lesion={r['has_lesion']}"
            )


# ============================================================
# REBUILD GROUP LIST AFTER CONFLICT REMOVAL
# ============================================================

gid_to_records = defaultdict(list)

for r in records:
    gid_to_records[r["refined_group_id"]].append(r)

group_records = list(gid_to_records.values())


# ============================================================
# STEP 6 — BALANCED GROUP-AWARE SPLIT
#
# Objective:
# 75% train
# 15% valid
# 10% test
#
# Balance source datasets while keeping complete lineage groups
# in one partition.
# ============================================================

random.seed(SEED)

source_names = sorted(
    SOURCE_SHORT.keys()
)

source_totals = Counter()

for r in records:
    source_totals[r["dataset"]] += 1

total_images = len(records)


target_source = {
    split: {
        src: source_totals[src] * fraction
        for src in source_names
    }
    for split, fraction in SPLIT_FRACTIONS.items()
}

target_total = {
    split: total_images * fraction
    for split, fraction in SPLIT_FRACTIONS.items()
}


def group_source_counts(group):
    c = Counter()

    for r in group:
        c[r["dataset"]] += 1

    return c


group_info = []

for g in group_records:

    group_info.append({
        "records": g,
        "size": len(g),
        "counts": group_source_counts(g),
        "gid": g[0]["refined_group_id"],
    })


def split_cost(assignments):

    source_counts = {
        split: Counter()
        for split in SPLIT_FRACTIONS
    }

    total_counts = Counter()

    for g, split in assignments:

        total_counts[split] += g["size"]

        for src, n in g["counts"].items():
            source_counts[split][src] += n

    cost = 0.0

    for split in SPLIT_FRACTIONS:

        # overall distribution
        t = max(target_total[split], 1)

        cost += (
            (total_counts[split] - target_total[split]) / t
        ) ** 2 * 2.0

        # source distributions
        for src in source_names:

            target = max(target_source[split][src], 1)

            cost += (
                (
                    source_counts[split][src]
                    - target_source[split][src]
                ) / target
            ) ** 2

    return cost


best_assignments = None
best_cost = float("inf")


# Multiple deterministic random restarts
for restart in range(250):

    rng = random.Random(SEED + restart)

    # largest groups first, randomized within same size
    ordered = list(group_info)

    rng.shuffle(ordered)

    ordered.sort(
        key=lambda g: g["size"],
        reverse=True
    )

    current_source = {
        split: Counter()
        for split in SPLIT_FRACTIONS
    }

    current_total = Counter()

    assignments = []

    for g in ordered:

        candidate_scores = []

        for split in ["train", "valid", "test"]:

            score = 0.0

            # overall target error after adding group
            new_total = (
                current_total[split]
                + g["size"]
            )

            t_total = max(
                target_total[split],
                1
            )

            score += (
                (new_total - target_total[split])
                / t_total
            ) ** 2 * 2.0

            # source errors
            for src in source_names:

                new_value = (
                    current_source[split][src]
                    + g["counts"].get(src, 0)
                )

                target = max(
                    target_source[split][src],
                    1
                )

                score += (
                    (
                        new_value
                        - target_source[split][src]
                    ) / target
                ) ** 2

            # Strong penalty for substantial overshoot
            if new_total > target_total[split] * 1.04:
                score += 10.0

            candidate_scores.append(
                (score, rng.random(), split)
            )

        candidate_scores.sort()

        chosen = candidate_scores[0][2]

        assignments.append((g, chosen))

        current_total[chosen] += g["size"]

        for src, n in g["counts"].items():
            current_source[chosen][src] += n

    c = split_cost(assignments)

    if c < best_cost:
        best_cost = c
        best_assignments = assignments


assert best_assignments is not None


# ============================================================
# FINAL SPLIT MAP
# ============================================================

group_to_split = {}

for g, split in best_assignments:

    group_to_split[g["gid"]] = split


for r in records:

    r["final_split"] = group_to_split[
        r["refined_group_id"]
    ]


# ============================================================
# SPLIT DISTRIBUTION
# ============================================================

print("\n" + "=" * 92)
print("FINAL GROUP-AWARE SPLIT")
print("=" * 92)

split_counts = Counter(
    r["final_split"]
    for r in records
)

for split in ["train", "valid", "test"]:

    n = split_counts[split]

    print(
        f"{split.upper():5}: "
        f"{n:4d} images "
        f"({100*n/len(records):.2f}%)"
    )


print("\nSource distribution:")

header = (
    f"{'SOURCE':<24}"
    f"{'TRAIN':>8}"
    f"{'VALID':>8}"
    f"{'TEST':>8}"
    f"{'TOTAL':>8}"
)

print(header)
print("-" * len(header))

for src in source_names:

    row = []

    for split in ["train", "valid", "test"]:

        n = sum(
            1
            for r in records
            if r["dataset"] == src
            and r["final_split"] == split
        )

        row.append(n)

    print(
        f"{SOURCE_SHORT[src]:<24}"
        f"{row[0]:>8}"
        f"{row[1]:>8}"
        f"{row[2]:>8}"
        f"{sum(row):>8}"
    )


# ============================================================
# STEP 7 — BUILD FINAL YOLO DATASET
# ============================================================

if FINAL_ROOT.exists():
    shutil.rmtree(FINAL_ROOT)

for split in ["train", "valid", "test"]:

    (FINAL_ROOT / split / "images").mkdir(
        parents=True,
        exist_ok=True
    )

    (FINAL_ROOT / split / "labels").mkdir(
        parents=True,
        exist_ok=True
    )


manifest_rows = []

for idx, r in enumerate(records, 1):

    split = r["final_split"]

    source_prefix = SOURCE_SHORT[r["dataset"]]

    original_name = safe_filename(
        r["image_name"]
    )

    # Guarantee uniqueness even if same source filenames repeat
    unique_prefix = (
        f"{source_prefix}__"
        f"{r['refined_group_id']}__"
        f"{idx:05d}__"
    )

    final_image_name = (
        unique_prefix
        + original_name
    )

    src_image = r["image_path"]

    dst_image = (
        FINAL_ROOT
        / split
        / "images"
        / final_image_name
    )

    shutil.copy2(
        src_image,
        dst_image
    )

    final_label_name = (
        Path(final_image_name).stem
        + ".txt"
    )

    dst_label = (
        FINAL_ROOT
        / split
        / "labels"
        / final_label_name
    )

    dst_label.write_text(
        "\n".join(r["remapped_lines"]) + "\n",
        encoding="utf-8"
    )

    r["final_image_name"] = final_image_name
    r["final_label_name"] = final_label_name

    manifest_rows.append({
        "final_split": split,
        "group_id": r["refined_group_id"],
        "source_dataset": r["dataset"],
        "source_category": source_prefix,
        "original_split": r["original_split"],
        "original_image": r["image_name"],
        "final_image": final_image_name,
        "md5": r["md5"],
        "canonical_name": r["canonical_name"],
        "processing_objects": r["processing_objects"],
        "lesion_objects": r["lesion_objects"],
    })


# ============================================================
# DATA.YAML
# ============================================================

data_yaml = {
    "path": str(FINAL_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 2,
    "names": [
        "processing_area",
        "lesion",
    ],
}

with open(
    FINAL_ROOT / "data.yaml",
    "w",
    encoding="utf-8"
) as f:

    yaml.safe_dump(
        data_yaml,
        f,
        sort_keys=False
    )


# ============================================================
# SAVE MANIFEST
# ============================================================

manifest_path = FINAL_ROOT / "dataset_manifest.csv"

with open(
    manifest_path,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=list(
            manifest_rows[0].keys()
        )
    )

    writer.writeheader()
    writer.writerows(manifest_rows)


# ============================================================
# SAVE ALL EXCLUSIONS
# ============================================================

all_exclusions = []

for x in quality_excluded:

    all_exclusions.append({
        "dataset": x["dataset"],
        "original_split": x["original_split"],
        "image": x["image_name"],
        "reason": x["exclude_reason"],
    })

for x in ambiguous_exact:

    all_exclusions.append({
        "dataset": x["dataset"],
        "original_split": x["original_split"],
        "image": x["image_name"],
        "reason": x["exclude_reason"],
    })

for x in semantic_conflicts:

    all_exclusions.append({
        "dataset": x["dataset"],
        "original_split": x["original_split"],
        "image": x["image_name"],
        "reason": x["exclude_reason"],
    })


exclusion_path = FINAL_ROOT / "quality_exclusions.csv"

with open(
    exclusion_path,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "dataset",
            "original_split",
            "image",
            "reason",
        ]
    )

    writer.writeheader()
    writer.writerows(all_exclusions)


# ============================================================
# STEP 8 — STRICT FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 92)
print("STRICT FINAL VERIFICATION")
print("=" * 92)

final_object_counts = {
    split: Counter()
    for split in ["train", "valid", "test"]
}

split_md5 = defaultdict(set)
group_splits = defaultdict(set)

errors = []

for split in ["train", "valid", "test"]:

    img_dir = FINAL_ROOT / split / "images"
    lbl_dir = FINAL_ROOT / split / "labels"

    images = sorted(
        p for p in img_dir.iterdir()
        if p.is_file()
    )

    labels = sorted(
        p for p in lbl_dir.iterdir()
        if p.suffix == ".txt"
    )

    print(
        f"{split.upper():5} "
        f"images={len(images)} "
        f"labels={len(labels)}"
    )

    if len(images) != len(labels):
        errors.append(
            f"{split}: image/label mismatch"
        )

    for img in images:

        label = lbl_dir / (
            img.stem + ".txt"
        )

        if not label.exists():

            errors.append(
                f"Missing label: {img}"
            )

            continue

        text = label.read_text(
            encoding="utf-8"
        ).strip()

        if not text:

            errors.append(
                f"Empty final label: {label}"
            )

            continue

        for line in text.splitlines():

            parts = line.split()

            cls = int(parts[0])

            if cls not in (0, 1):

                errors.append(
                    f"Invalid class {cls}: {label}"
                )

            final_object_counts[split][cls] += 1

        split_md5[split].add(
            md5_file(img)
        )


for r in records:

    group_splits[
        r["refined_group_id"]
    ].add(
        r["final_split"]
    )


group_leaks = {
    gid: splits
    for gid, splits in group_splits.items()
    if len(splits) > 1
}


md5_leaks = []

split_list = ["train", "valid", "test"]

for i in range(len(split_list)):

    for j in range(i + 1, len(split_list)):

        a = split_list[i]
        b = split_list[j]

        overlap = (
            split_md5[a]
            & split_md5[b]
        )

        if overlap:

            md5_leaks.append(
                (a, b, len(overlap))
            )


print("\nFinal object counts:")

for split in ["train", "valid", "test"]:

    print(
        f"  {split:5} | "
        f"processing_area={final_object_counts[split][0]} | "
        f"lesion={final_object_counts[split][1]}"
    )


print("\nLeakage checks:")
print(f"  lineage groups crossing splits : {len(group_leaks)}")
print(f"  exact MD5 cross-split overlaps : {len(md5_leaks)}")


# ============================================================
# DATA.YAML CHECK
# ============================================================

with open(
    FINAL_ROOT / "data.yaml",
    "r",
    encoding="utf-8"
) as f:

    check_yaml = yaml.safe_load(f)

assert check_yaml["nc"] == 2
assert check_yaml["names"] == [
    "processing_area",
    "lesion",
]


if errors:

    print("\n❌ FINAL VERIFICATION ERRORS:")

    for e in errors[:50]:
        print(" ", e)

    raise RuntimeError(
        f"{len(errors)} verification errors detected."
    )


if group_leaks:

    raise RuntimeError(
        "❌ Group leakage detected!"
    )


if md5_leaks:

    raise RuntimeError(
        f"❌ MD5 leakage detected: {md5_leaks}"
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 92)
print("CELL 3 FINAL SUMMARY")
print("=" * 92)

print(f"""
FINAL AI_Y2 DATASET
-------------------

Path:
{FINAL_ROOT}

Final classes:
0 = processing_area
1 = lesion

Final clean images:
{len(records)}

Split:
Train : {split_counts['train']}
Valid : {split_counts['valid']}
Test  : {split_counts['test']}

Removed in Cell 3:
Quality/incomplete annotation : {len(quality_excluded)}
Exact duplicate conflicts     : {len(ambiguous_exact)}
Cross-source visual conflicts : {len(semantic_conflicts)}

Leakage:
Lineage cross-split groups    : {len(group_leaks)}
Exact MD5 cross-split         : {len(md5_leaks)}

Files:
data.yaml
dataset_manifest.csv
quality_exclusions.csv

✅ Y1-compatible class contract preserved.
✅ Fresh group-aware split created.
✅ Exact duplicate leakage removed.
✅ Source distributions balanced.
✅ Ready for final visual/structural verification.
""")

print("=" * 92)
print("✅ CELL 3 COMPLETE")
print("=" * 92)

AI_Y2 MODULE 1 — CELL 3
FINAL CLEAN + BALANCED GROUP-AWARE DATASET BUILD

Input records from Cell 2: 1158

QUALITY COMPLETENESS FILTER
Kept     : 1155
Excluded : 3
  missing_processing_area_annotation: 2
  disease_source_without_lesion_annotation: 1

Excluded examples:
  leukoplakia_detection.v2i.yolov11 | train | Copy-of-1M-95-_jpg.rf.d849a70279227767b4e5c4968325c796.jpg | missing_processing_area_annotation
  leukoplakia_detection.v2i.yolov11 | train | SMITA00314_R__RB_jpg.rf.bda9c2d88dfe0c58522e723533bae8d5.jpg | disease_source_without_lesion_annotation
  ulcer_detection.v4i.yolov11 | train | 72_jpg.rf.6bf76d7637989aecf18ad972bae3c764.jpg | missing_processing_area_annotation

EXACT DUPLICATE ANNOTATION CONFLICT CHECK

⚠️ Exact-image annotation conflict:
MD5: 52a10bbe5ceb4ecc4bbe0974761f1e8c
  leukoplakia_detection.v2i.yolov11 | test | aug_1_C126_jpeg.rf.3d27d275641baaca747104e4543cb552.jpg | P=1 | L=2
  leukoplakia_like_detection.v3i.yolov11 | valid | aug_1_C126_jpeg.rf.99aa13e626769

In [ ]:
# ============================================================
# AI_Y2 — MODULE 1 / CELL 4 (CORRECTED)
#
# EXACT SOURCE-STRATIFIED + GROUP-AWARE SPLIT
#
# FINAL CONTRACT:
#   0 = processing_area
#   1 = lesion
#
# TARGET:
#   Train = 75%
#   Valid = 15%
#   Test  = 10%
#
# Guarantees:
# - refined lineage group never crosses splits
# - source-wise balance is near exact
# - exact MD5 leakage checked
# - Y1 class IDs preserved
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import random
import shutil
import hashlib
import csv
import yaml
import re
import math

# ------------------------------------------------------------
# REQUIRE CELL 3 STATE
# ------------------------------------------------------------

assert "records" in globals(), (
    "❌ Cell 3 ka clean 'records' variable nahi mila. "
    "Cell 3 run karo, then this cell."
)

assert len(records) == 1153, (
    f"⚠️ Expected 1153 clean records from Cell 3, got {len(records)}"
)

FINAL_ROOT = Path("/content/AI_Y2_FINAL")

SEED = 42

FRACTIONS = {
    "train": 0.75,
    "valid": 0.15,
    "test": 0.10,
}

SOURCE_SHORT = {
    "cancer_detection.v2i.yolov11":
        "cancer",

    "leukoplakia_detection.v2i.yolov11":
        "leukoplakia",

    "leukoplakia_like_detection.v3i.yolov11":
        "leukoplakia_like",

    "processing_area.v3i.yolov11":
        "processing_area",

    "ulcer_detection.v4i.yolov11":
        "ulcer",
}


# ============================================================
# HELPERS
# ============================================================

def md5_file(path):

    h = hashlib.md5()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def safe_filename(text):

    text = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(text)
    )

    return text.strip("_")


def largest_remainder_targets(total):
    """
    Convert 75/15/10 fractions into integer counts
    whose sum is EXACTLY total.

    Example:
      200 -> 150 / 30 / 20
    """

    exact = {
        split: total * frac
        for split, frac in FRACTIONS.items()
    }

    counts = {
        split: math.floor(value)
        for split, value in exact.items()
    }

    remaining = total - sum(counts.values())

    order = sorted(
        FRACTIONS.keys(),
        key=lambda s: (
            exact[s] - counts[s]
        ),
        reverse=True
    )

    for split in order[:remaining]:
        counts[split] += 1

    assert sum(counts.values()) == total

    return counts


# ============================================================
# REBUILD GROUPS FROM CELL 3 RECORDS
# ============================================================

gid_to_records = defaultdict(list)

for r in records:
    gid_to_records[
        r["refined_group_id"]
    ].append(r)


# A refined lineage group must belong to a single source
for gid, rows in gid_to_records.items():

    sources = {
        r["dataset"]
        for r in rows
    }

    if len(sources) != 1:

        raise RuntimeError(
            f"❌ Cross-source refined group detected: "
            f"{gid} -> {sources}"
        )


source_groups = defaultdict(list)

for gid, rows in gid_to_records.items():

    source = rows[0]["dataset"]

    source_groups[source].append({
        "gid": gid,
        "rows": rows,
        "size": len(rows),

        "processing_objects": sum(
            r["processing_objects"]
            for r in rows
        ),

        "lesion_objects": sum(
            r["lesion_objects"]
            for r in rows
        ),
    })


# ============================================================
# EXACT GROUP ALLOCATION
#
# Since current refined groups are only size 1 or 2,
# we can allocate size-2 groups first, then singletons fill
# exact image quotas.
# ============================================================

def allocate_source_groups(
    groups,
    source_name,
    target_counts,
    trials=5000,
):

    pairs = [
        g for g in groups
        if g["size"] == 2
    ]

    singles = [
        g for g in groups
        if g["size"] == 1
    ]

    total_processing = sum(
        g["processing_objects"]
        for g in groups
    )

    total_lesions = sum(
        g["lesion_objects"]
        for g in groups
    )

    best = None
    best_score = float("inf")

    # --------------------------------------------------------
    # Random search only controls which split gets pair groups.
    # Singletons then exactly fill remaining image quotas.
    # --------------------------------------------------------

    for trial in range(trials):

        rng = random.Random(
            SEED
            + trial
            + sum(ord(c) for c in source_name)
        )

        pair_order = list(pairs)
        rng.shuffle(pair_order)

        remaining = dict(target_counts)

        assignments = []

        possible = True

        for g in pair_order:

            candidate_splits = [
                split
                for split in ["train", "valid", "test"]
                if remaining[split] >= 2
            ]

            if not candidate_splits:
                possible = False
                break

            # Prefer split with largest relative remaining room
            scored = []

            for split in candidate_splits:

                target = max(
                    target_counts[split],
                    1
                )

                relative_remaining = (
                    remaining[split]
                    / target
                )

                scored.append(
                    (
                        -relative_remaining,
                        rng.random(),
                        split
                    )
                )

            scored.sort()

            chosen = scored[0][2]

            assignments.append(
                (g, chosen)
            )

            remaining[chosen] -= 2

        if not possible:
            continue

        # Remaining quotas must be exactly fillable
        # using singleton groups.
        if sum(remaining.values()) != len(singles):
            continue

        if any(v < 0 for v in remaining.values()):
            continue

        single_order = list(singles)
        rng.shuffle(single_order)

        pos = 0

        for split in ["train", "valid", "test"]:

            need = remaining[split]

            selected = single_order[
                pos:pos + need
            ]

            for g in selected:
                assignments.append(
                    (g, split)
                )

            pos += need

        if pos != len(single_order):
            continue

        # ----------------------------------------------------
        # Score object balance among all exact-count solutions
        # ----------------------------------------------------

        proc = Counter()
        lesions = Counter()
        images = Counter()

        for g, split in assignments:

            images[split] += g["size"]

            proc[split] += (
                g["processing_objects"]
            )

            lesions[split] += (
                g["lesion_objects"]
            )

        # hard exact image count check
        if any(
            images[s] != target_counts[s]
            for s in target_counts
        ):
            continue

        score = 0.0

        for split, frac in FRACTIONS.items():

            proc_target = (
                total_processing * frac
            )

            lesion_target = (
                total_lesions * frac
            )

            if total_processing > 0:

                score += (
                    abs(
                        proc[split]
                        - proc_target
                    )
                    / max(proc_target, 1)
                )

            if total_lesions > 0:

                score += (
                    abs(
                        lesions[split]
                        - lesion_target
                    )
                    / max(lesion_target, 1)
                )

        if score < best_score:

            best_score = score
            best = assignments

    if best is None:

        raise RuntimeError(
            f"❌ Exact group-aware split not found "
            f"for {source_name}. "
            f"Targets={target_counts}"
        )

    return best


# ============================================================
# BUILD ALL SOURCE-WISE SPLITS
# ============================================================

print("=" * 94)
print("AI_Y2 MODULE 1 — CELL 4 CORRECTED")
print("EXACT SOURCE-STRATIFIED GROUP-AWARE REBALANCE")
print("=" * 94)

group_to_split = {}

source_targets = {}

for source in sorted(source_groups):

    total = sum(
        g["size"]
        for g in source_groups[source]
    )

    targets = largest_remainder_targets(
        total
    )

    source_targets[source] = targets

    print(
        f"\n{SOURCE_SHORT[source]} "
        f"(n={total})"
    )

    print(
        f"  target -> "
        f"train={targets['train']}, "
        f"valid={targets['valid']}, "
        f"test={targets['test']}"
    )

    assignments = allocate_source_groups(
        source_groups[source],
        source,
        targets,
        trials=5000,
    )

    for g, split in assignments:

        group_to_split[
            g["gid"]
        ] = split


# Apply
for r in records:

    r["final_split"] = (
        group_to_split[
            r["refined_group_id"]
        ]
    )


# ============================================================
# DISTRIBUTION CHECK
# ============================================================

print("\n" + "=" * 94)
print("FINAL SOURCE DISTRIBUTION")
print("=" * 94)

header = (
    f"{'SOURCE':<24}"
    f"{'TRAIN':>8}"
    f"{'VALID':>8}"
    f"{'TEST':>8}"
    f"{'TOTAL':>8}"
)

print(header)
print("-" * len(header))


for source in sorted(
    SOURCE_SHORT.keys()
):

    vals = []

    for split in [
        "train",
        "valid",
        "test"
    ]:

        n = sum(
            1
            for r in records
            if r["dataset"] == source
            and r["final_split"] == split
        )

        vals.append(n)

    print(
        f"{SOURCE_SHORT[source]:<24}"
        f"{vals[0]:>8}"
        f"{vals[1]:>8}"
        f"{vals[2]:>8}"
        f"{sum(vals):>8}"
    )


overall = Counter(
    r["final_split"]
    for r in records
)

print("\nOverall:")

for split in [
    "train",
    "valid",
    "test"
]:

    n = overall[split]

    print(
        f"  {split:5}: "
        f"{n:4d} "
        f"({100*n/len(records):.2f}%)"
    )


# ============================================================
# HARD SOURCE-TARGET ASSERTION
# ============================================================

for source, targets in source_targets.items():

    for split, target in targets.items():

        actual = sum(
            1
            for r in records
            if (
                r["dataset"] == source
                and
                r["final_split"] == split
            )
        )

        assert actual == target, (
            f"❌ {source}/{split}: "
            f"expected {target}, got {actual}"
        )


print("\n✅ Every source hit its exact integer target.")


# ============================================================
# REBUILD FINAL DATASET
# ============================================================

print("\n" + "=" * 94)
print("REBUILDING /content/AI_Y2_FINAL")
print("=" * 94)

if FINAL_ROOT.exists():
    shutil.rmtree(FINAL_ROOT)

for split in [
    "train",
    "valid",
    "test"
]:

    (
        FINAL_ROOT
        / split
        / "images"
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    (
        FINAL_ROOT
        / split
        / "labels"
    ).mkdir(
        parents=True,
        exist_ok=True
    )


manifest_rows = []


sorted_records = sorted(
    records,
    key=lambda r: (
        r["final_split"],
        r["dataset"],
        r["refined_group_id"],
        r["image_name"],
    )
)


for idx, r in enumerate(
    sorted_records,
    1
):

    split = r["final_split"]

    prefix = (
        SOURCE_SHORT[
            r["dataset"]
        ]
    )

    final_image_name = (
        f"{prefix}__"
        f"{r['refined_group_id']}__"
        f"{idx:05d}__"
        f"{safe_filename(r['image_name'])}"
    )

    src_image = Path(
        r["image_path"]
    )

    dst_image = (
        FINAL_ROOT
        / split
        / "images"
        / final_image_name
    )

    shutil.copy2(
        src_image,
        dst_image
    )

    final_label_name = (
        Path(final_image_name).stem
        + ".txt"
    )

    dst_label = (
        FINAL_ROOT
        / split
        / "labels"
        / final_label_name
    )

    dst_label.write_text(
        "\n".join(
            r["remapped_lines"]
        ) + "\n",
        encoding="utf-8"
    )

    manifest_rows.append({
        "final_split":
            split,

        "group_id":
            r["refined_group_id"],

        "source_dataset":
            r["dataset"],

        "source_category":
            prefix,

        "original_split":
            r["original_split"],

        "original_image":
            r["image_name"],

        "final_image":
            final_image_name,

        "md5":
            r["md5"],

        "canonical_name":
            r["canonical_name"],

        "processing_objects":
            r["processing_objects"],

        "lesion_objects":
            r["lesion_objects"],
    })


# ============================================================
# DATA.YAML
# ============================================================

data_yaml = {
    "path":
        str(FINAL_ROOT),

    "train":
        "train/images",

    "val":
        "valid/images",

    "test":
        "test/images",

    "nc":
        2,

    "names": [
        "processing_area",
        "lesion",
    ],
}


with open(
    FINAL_ROOT / "data.yaml",
    "w",
    encoding="utf-8"
) as f:

    yaml.safe_dump(
        data_yaml,
        f,
        sort_keys=False
    )


# ============================================================
# MANIFEST
# ============================================================

manifest_path = (
    FINAL_ROOT
    / "dataset_manifest.csv"
)

with open(
    manifest_path,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=list(
            manifest_rows[0].keys()
        )
    )

    writer.writeheader()

    writer.writerows(
        manifest_rows
    )


# ============================================================
# STRICT FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 94)
print("STRICT FINAL VERIFICATION")
print("=" * 94)

errors = []

object_counts = {
    split: Counter()
    for split in [
        "train",
        "valid",
        "test"
    ]
}

split_md5 = {
    split: set()
    for split in [
        "train",
        "valid",
        "test"
    ]
}

group_splits = defaultdict(set)


for r in records:

    group_splits[
        r["refined_group_id"]
    ].add(
        r["final_split"]
    )


for split in [
    "train",
    "valid",
    "test"
]:

    img_dir = (
        FINAL_ROOT
        / split
        / "images"
    )

    lbl_dir = (
        FINAL_ROOT
        / split
        / "labels"
    )

    images = sorted(
        p
        for p in img_dir.iterdir()
        if p.is_file()
    )

    labels = sorted(
        p
        for p in lbl_dir.iterdir()
        if p.suffix == ".txt"
    )

    print(
        f"{split.upper():5} "
        f"images={len(images)} "
        f"labels={len(labels)}"
    )

    if len(images) != len(labels):

        errors.append(
            f"{split}: image-label count mismatch"
        )

    for img in images:

        lbl = (
            lbl_dir
            / f"{img.stem}.txt"
        )

        if not lbl.exists():

            errors.append(
                f"Missing label: {img.name}"
            )

            continue

        text = lbl.read_text(
            encoding="utf-8"
        ).strip()

        if not text:

            errors.append(
                f"Empty final label: {lbl.name}"
            )

            continue

        for line in text.splitlines():

            parts = line.split()

            cls = int(
                parts[0]
            )

            if cls not in (
                0,
                1
            ):

                errors.append(
                    f"Invalid class: {cls}"
                )

            object_counts[
                split
            ][cls] += 1

        split_md5[
            split
        ].add(
            md5_file(img)
        )


# ============================================================
# LEAKAGE
# ============================================================

group_leaks = {
    gid: splits
    for gid, splits in group_splits.items()
    if len(splits) > 1
}


md5_leaks = []

for a, b in [
    ("train", "valid"),
    ("train", "test"),
    ("valid", "test"),
]:

    overlap = (
        split_md5[a]
        &
        split_md5[b]
    )

    if overlap:

        md5_leaks.append(
            (
                a,
                b,
                len(overlap)
            )
        )


print("\nObject distribution:")

for split in [
    "train",
    "valid",
    "test"
]:

    print(
        f"  {split:5} | "
        f"processing_area="
        f"{object_counts[split][0]} | "
        f"lesion="
        f"{object_counts[split][1]}"
    )


print("\nLeakage:")

print(
    f"  lineage groups crossing splits : "
    f"{len(group_leaks)}"
)

print(
    f"  exact MD5 overlaps             : "
    f"{len(md5_leaks)}"
)


# ============================================================
# SOURCE RATIO FINAL
# ============================================================

print("\n" + "=" * 94)
print("SOURCE RATIO FINAL")
print("=" * 94)


for source in sorted(
    SOURCE_SHORT.keys()
):

    rows = [
        r
        for r in records
        if r["dataset"] == source
    ]

    total = len(rows)

    print(
        f"\n{SOURCE_SHORT[source]} "
        f"(n={total})"
    )

    for split in [
        "train",
        "valid",
        "test"
    ]:

        n = sum(
            1
            for r in rows
            if r["final_split"] == split
        )

        print(
            f"  {split:5}: "
            f"{n:3d} "
            f"({100*n/total:5.2f}%)"
        )


# ============================================================
# YAML CONTRACT
# ============================================================

with open(
    FINAL_ROOT / "data.yaml",
    "r",
    encoding="utf-8"
) as f:

    y = yaml.safe_load(f)


assert y["nc"] == 2

assert y["names"] == [
    "processing_area",
    "lesion",
]


if errors:

    raise RuntimeError(
        f"❌ Verification errors: {len(errors)}"
    )

if group_leaks:

    raise RuntimeError(
        "❌ Lineage leakage detected."
    )

if md5_leaks:

    raise RuntimeError(
        f"❌ Exact-image leakage: {md5_leaks}"
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 94)
print("CELL 4 FINAL SUMMARY")
print("=" * 94)

print(f"""
FINAL Y2 DATASET:
{FINAL_ROOT}

CLASS CONTRACT:
0 = processing_area
1 = lesion

TOTAL:
{len(records)}

TRAIN:
{overall['train']}
({100*overall['train']/len(records):.2f}%)

VALID:
{overall['valid']}
({100*overall['valid']/len(records):.2f}%)

TEST:
{overall['test']}
({100*overall['test']/len(records):.2f}%)

LINEAGE LEAKAGE:
{len(group_leaks)}

EXACT MD5 LEAKAGE:
{len(md5_leaks)}

SOURCE-WISE SPLIT:
EXACT INTEGER TARGETS PASSED

DATA YAML:
{FINAL_ROOT / 'data.yaml'}

MANIFEST:
{manifest_path}

✅ Y1 class contract preserved
✅ source-stratified
✅ group-aware
✅ duplicate-safe
✅ ready for final visual audit
""")

print("=" * 94)
print("✅ CELL 4 CORRECTED COMPLETE")
print("=" * 94)

AI_Y2 MODULE 1 — CELL 4 CORRECTED
EXACT SOURCE-STRATIFIED GROUP-AWARE REBALANCE

cancer (n=200)
  target -> train=150, valid=30, test=20

leukoplakia (n=246)
  target -> train=184, valid=37, test=25

leukoplakia_like (n=249)
  target -> train=187, valid=37, test=25

processing_area (n=264)
  target -> train=198, valid=40, test=26

ulcer (n=194)
  target -> train=146, valid=29, test=19

FINAL SOURCE DISTRIBUTION
SOURCE                     TRAIN   VALID    TEST   TOTAL
--------------------------------------------------------
cancer                       150      30      20     200
leukoplakia                  184      37      25     246
leukoplakia_like             187      37      25     249
processing_area              198      40      26     264
ulcer                        146      29      19     194

Overall:
  train:  865 (75.02%)
  valid:  173 (15.00%)
  test :  115 (9.97%)

✅ Every source hit its exact integer target.

REBUILDING /content/AI_Y2_FINAL

STRICT FINAL VERIFICATION
TR

In [ ]:
# ============================================================
# AI_Y2 — MODULE 1 / CELL 5
#
# FINAL VISUAL + STRUCTURAL AUDIT
# DATASET FREEZE / FINGERPRINT
#
# READ-ONLY for images/labels.
#
# FINAL CONTRACT:
#   0 = processing_area
#   1 = lesion
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml


# ============================================================
# PATHS / CONFIG
# ============================================================

FINAL_ROOT = Path("/content/AI_Y2_FINAL")

DATA_YAML = FINAL_ROOT / "data.yaml"
MANIFEST = FINAL_ROOT / "dataset_manifest.csv"

FREEZE_JSON = FINAL_ROOT / "AI_Y2_DATASET_FREEZE.json"

SEED = 42

CLASS_NAMES = {
    0: "processing_area",
    1: "lesion",
}

SOURCE_ORDER = [
    "cancer",
    "leukoplakia",
    "leukoplakia_like",
    "processing_area",
    "ulcer",
]


print("=" * 100)
print("AI_Y2 MODULE 1 — CELL 5")
print("FINAL VISUAL + STRUCTURAL AUDIT + DATASET FREEZE")
print("=" * 100)


# ============================================================
# BASIC FILE CHECKS
# ============================================================

assert FINAL_ROOT.exists(), f"❌ Missing dataset: {FINAL_ROOT}"
assert DATA_YAML.exists(), f"❌ Missing data.yaml"
assert MANIFEST.exists(), f"❌ Missing manifest"


with open(DATA_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)


assert cfg["nc"] == 2, f"❌ nc should be 2, got {cfg['nc']}"

assert cfg["names"] == [
    "processing_area",
    "lesion"
], f"❌ Wrong class order: {cfg['names']}"


print("\n✅ DATA.YAML CONTRACT")
print(f"nc    : {cfg['nc']}")
print(f"names : {cfg['names']}")


# ============================================================
# LOAD MANIFEST
# ============================================================

df = pd.read_csv(MANIFEST)

assert len(df) == 1153, \
    f"❌ Expected 1153 manifest rows, got {len(df)}"


# ============================================================
# STRICT POLYGON VALIDATION
# ============================================================

errors = []

split_stats = {}
all_class_counts = Counter()

polygon_point_counts = Counter()

for split in ["train", "valid", "test"]:

    img_dir = FINAL_ROOT / split / "images"
    lbl_dir = FINAL_ROOT / split / "labels"

    images = sorted([
        p for p in img_dir.iterdir()
        if p.is_file()
    ])

    labels = sorted([
        p for p in lbl_dir.iterdir()
        if p.suffix.lower() == ".txt"
    ])

    class_counts = Counter()

    for img_path in images:

        lbl_path = lbl_dir / f"{img_path.stem}.txt"

        if not lbl_path.exists():
            errors.append(
                f"Missing label: {split}/{img_path.name}"
            )
            continue

        text = lbl_path.read_text(
            encoding="utf-8"
        ).strip()

        if not text:
            errors.append(
                f"Empty label: {split}/{lbl_path.name}"
            )
            continue

        for line_num, line in enumerate(
            text.splitlines(), 1
        ):

            parts = line.split()

            if len(parts) < 7:

                errors.append(
                    f"Too few polygon values: "
                    f"{lbl_path.name}:{line_num}"
                )
                continue

            try:
                cls_id = int(parts[0])
                coords = np.array(
                    [float(x) for x in parts[1:]],
                    dtype=np.float64
                )

            except Exception:

                errors.append(
                    f"Parsing failure: "
                    f"{lbl_path.name}:{line_num}"
                )
                continue

            if cls_id not in (0, 1):

                errors.append(
                    f"Invalid class {cls_id}: "
                    f"{lbl_path.name}:{line_num}"
                )

            if len(coords) < 6:

                errors.append(
                    f"<3 polygon vertices: "
                    f"{lbl_path.name}:{line_num}"
                )

            if len(coords) % 2 != 0:

                errors.append(
                    f"Odd coordinate count: "
                    f"{lbl_path.name}:{line_num}"
                )

            if not np.isfinite(coords).all():

                errors.append(
                    f"NaN/Inf coordinate: "
                    f"{lbl_path.name}:{line_num}"
                )

            if np.any(coords < 0) or np.any(coords > 1):

                errors.append(
                    f"Out-of-range coordinate: "
                    f"{lbl_path.name}:{line_num}"
                )

            pts = coords.reshape(-1, 2)

            # at least 3 unique vertices
            if len(np.unique(
                np.round(pts, 7),
                axis=0
            )) < 3:

                errors.append(
                    f"<3 unique vertices: "
                    f"{lbl_path.name}:{line_num}"
                )

            class_counts[cls_id] += 1
            all_class_counts[cls_id] += 1

            polygon_point_counts[
                len(pts)
            ] += 1


    split_stats[split] = {
        "images": len(images),
        "labels": len(labels),
        "processing_area_objects":
            class_counts[0],
        "lesion_objects":
            class_counts[1],
    }


# ============================================================
# SOURCE / SPLIT SUMMARY
# ============================================================

source_table = pd.crosstab(
    df["source_category"],
    df["final_split"]
)

for c in ["train", "valid", "test"]:
    if c not in source_table.columns:
        source_table[c] = 0

source_table = source_table[
    ["train", "valid", "test"]
]

source_table["total"] = (
    source_table["train"]
    + source_table["valid"]
    + source_table["test"]
)


print("\n" + "=" * 100)
print("FINAL SPLIT / OBJECT SUMMARY")
print("=" * 100)

for split in ["train", "valid", "test"]:

    s = split_stats[split]

    print(
        f"{split.upper():5} | "
        f"images={s['images']:3d} | "
        f"labels={s['labels']:3d} | "
        f"processing_area={s['processing_area_objects']:4d} | "
        f"lesion={s['lesion_objects']:4d}"
    )


print("\nTOTAL OBJECTS")

print(
    f"processing_area : "
    f"{all_class_counts[0]}"
)

print(
    f"lesion          : "
    f"{all_class_counts[1]}"
)


print("\nSOURCE DISTRIBUTION")
print(source_table.to_string())


# ============================================================
# MANIFEST / CONTENT CONSISTENCY
# ============================================================

print("\n" + "=" * 100)
print("MANIFEST CONSISTENCY")
print("=" * 100)

manifest_missing = []

for _, row in df.iterrows():

    split = row["final_split"]
    filename = row["final_image"]

    img_path = (
        FINAL_ROOT
        / split
        / "images"
        / filename
    )

    lbl_path = (
        FINAL_ROOT
        / split
        / "labels"
        / f"{Path(filename).stem}.txt"
    )

    if not img_path.exists():
        manifest_missing.append(
            f"image: {img_path}"
        )

    if not lbl_path.exists():
        manifest_missing.append(
            f"label: {lbl_path}"
        )


print(
    f"Manifest rows               : {len(df)}"
)

print(
    f"Missing referenced files    : "
    f"{len(manifest_missing)}"
)


if manifest_missing:
    errors.extend(manifest_missing)


# ============================================================
# VISUALIZATION HELPERS
# ============================================================

def read_polygons(label_path):

    polygons = []

    text = label_path.read_text(
        encoding="utf-8"
    ).strip()

    for line in text.splitlines():

        parts = line.split()

        cls_id = int(parts[0])

        coords = np.array(
            [float(x) for x in parts[1:]],
            dtype=np.float32
        ).reshape(-1, 2)

        polygons.append(
            (cls_id, coords)
        )

    return polygons


def make_overlay(image_path, label_path):

    image = cv2.imread(str(image_path))

    if image is None:
        raise RuntimeError(
            f"Unreadable image: {image_path}"
        )

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    h, w = image.shape[:2]

    overlay = image.copy()

    polygons = read_polygons(
        label_path
    )

    counts = Counter()

    # Visual colors only:
    # processing_area = green
    # lesion = red
    colors = {
        0: (0, 255, 0),
        1: (255, 0, 0),
    }

    for cls_id, normalized_pts in polygons:

        pts = normalized_pts.copy()

        pts[:, 0] *= w
        pts[:, 1] *= h

        pts = np.round(
            pts
        ).astype(np.int32)

        color = colors[cls_id]

        cv2.polylines(
            overlay,
            [pts],
            isClosed=True,
            color=color,
            thickness=2
        )

        counts[cls_id] += 1

    return overlay, counts


# ============================================================
# SELECT REPRESENTATIVE VISUAL SAMPLES
#
# 5 sources × train/valid/test = 15 examples
# ============================================================

print("\n" + "=" * 100)
print("VISUAL AUDIT")
print("=" * 100)

rng = random.Random(SEED)

visual_rows = []

for source in SOURCE_ORDER:

    for split in [
        "train",
        "valid",
        "test"
    ]:

        subset = df[
            (df["source_category"] == source)
            &
            (df["final_split"] == split)
        ]

        assert len(subset) > 0, (
            f"No {source}/{split} sample"
        )

        # deterministic random sample
        idx = rng.randrange(
            len(subset)
        )

        visual_rows.append(
            subset.iloc[idx]
        )


# ============================================================
# DISPLAY 15 OVERLAYS
# ============================================================

fig, axes = plt.subplots(
    3,
    5,
    figsize=(22, 13)
)

for col, source in enumerate(SOURCE_ORDER):

    for row_idx, split in enumerate(
        ["train", "valid", "test"]
    ):

        row = next(
            x
            for x in visual_rows
            if (
                x["source_category"] == source
                and
                x["final_split"] == split
            )
        )

        image_path = (
            FINAL_ROOT
            / split
            / "images"
            / row["final_image"]
        )

        label_path = (
            FINAL_ROOT
            / split
            / "labels"
            / f"{Path(row['final_image']).stem}.txt"
        )

        overlay, counts = make_overlay(
            image_path,
            label_path
        )

        ax = axes[row_idx, col]

        ax.imshow(overlay)

        ax.set_title(
            f"{source}\n"
            f"{split.upper()} | "
            f"P={counts[0]} L={counts[1]}",
            fontsize=10
        )

        ax.axis("off")


plt.suptitle(
    "AI_Y2 Final Dataset Visual Audit\n"
    "Green = processing_area | Red = lesion",
    fontsize=16
)

plt.tight_layout()
plt.show()


# ============================================================
# DATASET FINGERPRINT
#
# Fingerprint includes every final image MD5 and
# every final label content SHA256.
# ============================================================

print("\n" + "=" * 100)
print("DATASET FINGERPRINT")
print("=" * 100)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


fingerprint_lines = []


for split in [
    "train",
    "valid",
    "test"
]:

    img_dir = (
        FINAL_ROOT
        / split
        / "images"
    )

    lbl_dir = (
        FINAL_ROOT
        / split
        / "labels"
    )

    for img in sorted(
        img_dir.iterdir(),
        key=lambda p: p.name
    ):

        if not img.is_file():
            continue

        lbl = (
            lbl_dir
            / f"{img.stem}.txt"
        )

        fingerprint_lines.append(
            "|".join([
                split,
                img.name,
                sha256_file(img),
                sha256_file(lbl),
            ])
        )


dataset_fingerprint = hashlib.sha256(
    "\n".join(
        fingerprint_lines
    ).encode("utf-8")
).hexdigest()


print(
    f"Dataset SHA256 fingerprint:\n"
    f"{dataset_fingerprint}"
)


# ============================================================
# FREEZE METADATA
# ============================================================

freeze_info = {

    "dataset_name":
        "AI_Y2",

    "module":
        "Module 1 final dataset",

    "root":
        str(FINAL_ROOT),

    "class_contract": {
        "0": "processing_area",
        "1": "lesion",
    },

    "total_images":
        len(df),

    "splits": {
        split: split_stats[split]
        for split in [
            "train",
            "valid",
            "test"
        ]
    },

    "total_objects": {
        "processing_area":
            int(all_class_counts[0]),

        "lesion":
            int(all_class_counts[1]),
    },

    "source_distribution":
        source_table.to_dict(
            orient="index"
        ),

    "verified": {
        "polygon_structure":
            len(errors) == 0,

        "class_mapping":
            True,

        "manifest_consistency":
            len(manifest_missing) == 0,

        "lineage_cross_split":
            0,

        "exact_md5_cross_split":
            0,
    },

    "dataset_sha256":
        dataset_fingerprint,

    "data_yaml_sha256":
        sha256_file(DATA_YAML),

    "manifest_sha256":
        sha256_file(MANIFEST),
}


with open(
    FREEZE_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        freeze_info,
        f,
        indent=2
    )


# ============================================================
# FINAL PASS / FAIL
# ============================================================

print("\n" + "=" * 100)
print("FINAL MODULE 1 AUDIT")
print("=" * 100)

print(
    f"Structural errors       : "
    f"{len(errors)}"
)

print(
    f"Manifest missing files  : "
    f"{len(manifest_missing)}"
)

print(
    f"Freeze metadata         : "
    f"{FREEZE_JSON}"
)


if errors:

    print("\n❌ ERRORS FOUND:")

    for e in errors[:30]:
        print(" -", e)

    raise RuntimeError(
        f"❌ Dataset audit failed "
        f"with {len(errors)} errors."
    )


print("""
✅ STRUCTURAL AUDIT PASSED
✅ CLASS CONTRACT PASSED
✅ MANIFEST CONSISTENCY PASSED
✅ FINAL SPLIT COUNTS PASSED
✅ DATASET FINGERPRINT CREATED

IMPORTANT:
Ab upar displayed 15 overlay images visually inspect karo.

Check:
1. Green processing_area actual useful mouth/mucosal region par hai.
2. Red lesion actual lesion par hai.
3. Red/green polygons shifted nahi hain.
4. Processing-only samples me red lesion nahi hona chahiye.
5. Obvious wrong/tiny/random polygons nahi hone chahiye.

Agar visual overlays clinically/annotation-wise correct dikh rahe hain,
AI_Y2 MODULE 1 ko FROZEN maana ja sakta hai.
""")

print("=" * 100)
print("✅ CELL 5 COMPLETE")
print("=" * 100)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 1
#
# ENVIRONMENT + DATASET FREEZE + COMPATIBILITY GATE
#
# Y1 DROP-IN CONTRACT:
#   model family = YOLO26-M segmentation
#   imgsz        = 640
#   class 0      = processing_area
#   class 1      = lesion
# ============================================================

%pip install -q --no-cache-dir "ultralytics>=8.3.0" "PyYAML>=6.0"

# ============================================================
# IMPORTS
# ============================================================

import os
import gc
import sys
import json
import yaml
import time
import random
import hashlib
import shutil
import warnings

from pathlib import Path
from collections import Counter

import numpy as np
import torch
import ultralytics

from ultralytics import YOLO

warnings.filterwarnings("ignore")


# ============================================================
# PATHS
# ============================================================

DATASET_ROOT = Path(
    "/content/AI_Y2_FINAL"
)

DATA_YAML = (
    DATASET_ROOT /
    "data.yaml"
)

FREEZE_JSON = (
    DATASET_ROOT /
    "AI_Y2_DATASET_FREEZE.json"
)

MANIFEST = (
    DATASET_ROOT /
    "dataset_manifest.csv"
)

# Training workspace
MODULE2_ROOT = Path(
    "/content/AI_Y2_MODULE2"
)

RUNS_ROOT = (
    MODULE2_ROOT /
    "runs"
)

REPORTS_ROOT = (
    MODULE2_ROOT /
    "reports"
)

# Final model later copied here
FINAL_MODEL_PATH = Path(
    "/content/ai_y2.pt"
)


# ============================================================
# HARD COMPATIBILITY CONTRACT
# ============================================================

MODEL_CHECKPOINT = "yolo26m-seg.pt"

IMAGE_SIZE = 640

EXPECTED_CLASSES = {
    0: "processing_area",
    1: "lesion",
}

EXPECTED_DATASET_SHA256 = (
    "fcf15becb5390bc23d79c81d47e0cf45"
    "fe386a74217872c3788ffcf6e3544e6e"
)

SEED = 42


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def dataset_fingerprint(root):

    lines = []

    for split in [
        "train",
        "valid",
        "test"
    ]:

        img_dir = (
            root /
            split /
            "images"
        )

        lbl_dir = (
            root /
            split /
            "labels"
        )

        for img in sorted(
            img_dir.iterdir(),
            key=lambda p: p.name
        ):

            if not img.is_file():
                continue

            lbl = (
                lbl_dir /
                f"{img.stem}.txt"
            )

            if not lbl.exists():
                raise RuntimeError(
                    f"Missing label for: {img}"
                )

            lines.append(
                "|".join([
                    split,
                    img.name,
                    sha256_file(img),
                    sha256_file(lbl),
                ])
            )

    return hashlib.sha256(
        "\n".join(lines).encode("utf-8")
    ).hexdigest()


def count_split(split):

    img_dir = (
        DATASET_ROOT /
        split /
        "images"
    )

    lbl_dir = (
        DATASET_ROOT /
        split /
        "labels"
    )

    images = sorted(
        p for p in img_dir.iterdir()
        if p.is_file()
    )

    labels = sorted(
        p for p in lbl_dir.iterdir()
        if p.suffix == ".txt"
    )

    objects = Counter()

    for lbl in labels:

        text = lbl.read_text(
            encoding="utf-8"
        ).strip()

        if not text:
            raise RuntimeError(
                f"Empty label: {lbl}"
            )

        for line in text.splitlines():

            cls = int(
                line.split()[0]
            )

            objects[cls] += 1

    return (
        len(images),
        len(labels),
        objects
    )


# ============================================================
# BASIC PATH CHECK
# ============================================================

print("=" * 96)
print("AI_Y2 MODULE 2 — CELL 1")
print("ENVIRONMENT + FROZEN DATASET + COMPATIBILITY GATE")
print("=" * 96)

for required in [
    DATASET_ROOT,
    DATA_YAML,
    FREEZE_JSON,
    MANIFEST,
]:

    if not required.exists():

        raise FileNotFoundError(
            f"❌ Required item missing:\n{required}"
        )


# ============================================================
# VERIFY DATA.YAML
# ============================================================

with open(
    DATA_YAML,
    "r",
    encoding="utf-8"
) as f:

    data_cfg = yaml.safe_load(f)


if data_cfg.get("nc") != 2:

    raise RuntimeError(
        f"❌ Expected nc=2, "
        f"found {data_cfg.get('nc')}"
    )


if data_cfg.get("names") != [
    "processing_area",
    "lesion"
]:

    raise RuntimeError(
        "❌ Class mapping changed.\n"
        f"Expected: ['processing_area', 'lesion']\n"
        f"Found   : {data_cfg.get('names')}"
    )


# ============================================================
# VERIFY FREEZE RECORD
# ============================================================

with open(
    FREEZE_JSON,
    "r",
    encoding="utf-8"
) as f:

    freeze = json.load(f)


freeze_class_contract = (
    freeze.get(
        "class_contract",
        {}
    )
)


if freeze_class_contract != {
    "0": "processing_area",
    "1": "lesion",
}:

    raise RuntimeError(
        "❌ Freeze class contract mismatch:\n"
        f"{freeze_class_contract}"
    )


# ============================================================
# RECOMPUTE DATASET FINGERPRINT
# ============================================================

print("\nRecomputing frozen dataset fingerprint...")

current_fingerprint = dataset_fingerprint(
    DATASET_ROOT
)


print(
    f"\nExpected SHA256 : "
    f"{EXPECTED_DATASET_SHA256}"
)

print(
    f"Current SHA256  : "
    f"{current_fingerprint}"
)


if (
    current_fingerprint
    != EXPECTED_DATASET_SHA256
):

    raise RuntimeError(
        "\n❌ DATASET CHANGED AFTER MODULE 1.\n"
        "Training stopped intentionally."
    )


print(
    "✅ Dataset fingerprint exactly matches "
    "Module 1 freeze."
)


# ============================================================
# VERIFY SPLITS + OBJECTS
# ============================================================

print("\n" + "=" * 96)
print("FROZEN DATASET CONTENT")
print("=" * 96)


expected_split_images = {
    "train": 865,
    "valid": 173,
    "test": 115,
}

expected_objects = {
    "train": {
        0: 900,
        1: 787,
    },

    "valid": {
        0: 179,
        1: 157,
    },

    "test": {
        0: 120,
        1: 105,
    },
}


for split in [
    "train",
    "valid",
    "test"
]:

    n_img, n_lbl, obj = count_split(
        split
    )

    print(
        f"{split.upper():5} | "
        f"images={n_img:3d} | "
        f"labels={n_lbl:3d} | "
        f"processing_area={obj[0]:3d} | "
        f"lesion={obj[1]:3d}"
    )

    assert (
        n_img
        == expected_split_images[split]
    )

    assert n_img == n_lbl

    assert (
        obj[0]
        == expected_objects[split][0]
    )

    assert (
        obj[1]
        == expected_objects[split][1]
    )


# ============================================================
# GPU CHECK
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "❌ CUDA GPU not available.\n"
        "Use Colab A100 runtime before training."
    )


gpu_name = torch.cuda.get_device_name(0)

gpu_props = (
    torch.cuda.get_device_properties(0)
)

gpu_memory = (
    gpu_props.total_memory
    / (1024 ** 3)
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)


# ============================================================
# CREATE MODULE 2 WORKSPACE
# ============================================================

MODULE2_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

RUNS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

REPORTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# DOWNLOAD / LOAD SAME MODEL FAMILY AS Y1
# ============================================================

print("\n" + "=" * 96)
print("Y1 → Y2 MODEL COMPATIBILITY")
print("=" * 96)

base_model = YOLO(
    MODEL_CHECKPOINT
)


# pretrained COCO class names are irrelevant here;
# final names will be replaced by dataset training.
#
# But segmentation task must remain segmentation.

task = getattr(
    base_model,
    "task",
    None
)

print(
    f"Checkpoint       : {MODEL_CHECKPOINT}"
)

print(
    f"Ultralytics task : {task}"
)


if str(task) != "segment":

    raise RuntimeError(
        "❌ Base checkpoint is not "
        "a segmentation model."
    )


# ============================================================
# FINAL ENVIRONMENT REPORT
# ============================================================

print("\n" + "=" * 96)
print("MODULE 2 READY")
print("=" * 96)

print(
    f"Python             : "
    f"{sys.version.split()[0]}"
)

print(
    f"PyTorch            : "
    f"{torch.__version__}"
)

print(
    f"Ultralytics        : "
    f"{ultralytics.__version__}"
)

print(
    f"GPU                : "
    f"{gpu_name}"
)

print(
    f"GPU VRAM           : "
    f"{gpu_memory:.2f} GB"
)

print(
    f"Dataset            : "
    f"{DATASET_ROOT}"
)

print(
    f"Dataset fingerprint: "
    f"{current_fingerprint}"
)

print(
    f"Base checkpoint    : "
    f"{MODEL_CHECKPOINT}"
)

print(
    f"Input size         : "
    f"{IMAGE_SIZE}"
)

print(
    f"Class 0            : "
    f"{EXPECTED_CLASSES[0]}"
)

print(
    f"Class 1            : "
    f"{EXPECTED_CLASSES[1]}"
)

print(
    f"Future final model : "
    f"{FINAL_MODEL_PATH}"
)

print("""
COMPATIBILITY CONTRACT
----------------------
Y1:
    input image
        ↓
    YOLO26-M segmentation
        ↓
    class 0 = processing_area
    class 1 = lesion
        ↓
    boxes / masks / polygons / confidence

Y2:
    EXACT SAME EXTERNAL CONTRACT

Therefore downstream software must NOT require
a Y2-specific branch.

C1–C10 should consume detector output through
the same detector interface / ROI adapter.
""")

print("=" * 96)
print("✅ MODULE 2 CELL 1 PASSED")
print("=" * 96)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 356.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 341.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
AI_Y2 MODULE 2 — CELL 1
ENVIRONMENT + FROZEN DATASET + COMPATIBILITY GATE

Recomputing frozen dataset fingerprint...

Expected SHA256 : fcf15becb5390bc23d79c81d47e0cf45fe386a74217872c3788ffcf6e3544e6e
Current SHA256  : fcf15becb5390bc23d79c81d47e0cf45fe386a74217872c3788ffcf6e3544e6e
✅ Dataset fingerprint exactly matches Module 1 freeze.

FROZEN DATASET CONTENT
TRAIN | images=865 | labels=865 | processing_area=900 | lesion=787
VALID | images=173 | labels=173 | proc

In [ ]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 2
#
# TWO-PHASE HIGH-QUALITY SEGMENTATION TRAINING
#
# COMPATIBILITY IS FROZEN:
#   Model family : YOLO26-M Segmentation
#   Input        : 640
#   Class 0      : processing_area
#   Class 1      : lesion
#
# TEST SET IS NOT USED IN THIS CELL.
# ============================================================

import gc
import time
import json
import shutil
import random
from pathlib import Path

import numpy as np
import torch

from ultralytics import YOLO


# ============================================================
# REQUIRE CELL 1 VARIABLES
# ============================================================

required_vars = [
    "DATA_YAML",
    "RUNS_ROOT",
    "REPORTS_ROOT",
    "MODEL_CHECKPOINT",
    "IMAGE_SIZE",
    "EXPECTED_CLASSES",
    "SEED",
]

for v in required_vars:
    if v not in globals():
        raise RuntimeError(
            f"❌ Missing Cell 1 variable: {v}\n"
            "Run Module 2 Cell 1 first."
        )


# ============================================================
# TRAINING PATHS
# ============================================================

PHASE1_NAME = "AI_Y2_phase1_general"
PHASE2_NAME = "AI_Y2_phase2_refine"

PHASE1_DIR = RUNS_ROOT / PHASE1_NAME
PHASE2_DIR = RUNS_ROOT / PHASE2_NAME

PHASE1_CANDIDATE = (
    MODULE2_ROOT /
    "candidate_y2_phase1.pt"
)

PHASE2_CANDIDATE = (
    MODULE2_ROOT /
    "candidate_y2_phase2.pt"
)

TRAINING_INFO_JSON = (
    REPORTS_ROOT /
    "training_candidates.json"
)


# ============================================================
# FRESH TRAINING RUNS ONLY
# ============================================================

for p in [
    PHASE1_DIR,
    PHASE2_DIR,
]:

    if p.exists():
        shutil.rmtree(p)


for p in [
    PHASE1_CANDIDATE,
    PHASE2_CANDIDATE,
]:

    if p.exists():
        p.unlink()


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

gc.collect()
torch.cuda.empty_cache()

# Fast and stable on A100.
torch.backends.cudnn.benchmark = True


# ============================================================
# COMMON SETTINGS
# ============================================================

COMMON = {

    # --------------------------------------------------------
    # Frozen Y1-compatible input contract
    # --------------------------------------------------------
    "imgsz": 640,

    # Small medical dataset:
    # keep moderate batch for gradient diversity.
    "batch": 16,

    "device": 0,
    "workers": 8,

    # --------------------------------------------------------
    # Transfer learning / hardware
    # --------------------------------------------------------
    "pretrained": True,
    "amp": True,

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------
    "optimizer": "AdamW",
    "weight_decay": 0.0005,
    "cos_lr": True,

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------
    "seed": SEED,
    "deterministic": True,

    # --------------------------------------------------------
    # SEGMENTATION QUALITY
    #
    # Y1 effectively used default mask downsampling.
    # Y2 uses denser target-mask supervision.
    # A100 80 GB gives enough headroom.
    # --------------------------------------------------------
    "mask_ratio": 2,
    "overlap_mask": True,

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------
    "plots": True,
    "save": True,
    "save_period": -1,
    "verbose": True,
    "exist_ok": True,

    # --------------------------------------------------------
    # Data loading
    # Dataset is small enough to cache.
    # --------------------------------------------------------
    "cache": "ram",
}


# ============================================================
# PHASE 1
#
# Goal:
# General lesion + processing-area robustness.
#
# Compared with Y1:
# - cleaner/larger dataset
# - less unrealistic mosaic
# - no copy-paste
# - conservative oral-image geometry
# - dense segmentation supervision
# - longer patience
# ============================================================

PHASE1_ARGS = {

    **COMMON,

    "epochs": 350,
    "patience": 55,

    # Fine-tuning LR
    "lr0": 0.0004,
    "lrf": 0.01,

    "warmup_epochs": 4.0,

    # --------------------------------------------------------
    # Geometry
    # Mild only — oral anatomy should remain realistic.
    # --------------------------------------------------------
    "degrees": 8.0,
    "translate": 0.04,
    "scale": 0.12,

    "shear": 0.0,
    "perspective": 0.0,

    "fliplr": 0.50,
    "flipud": 0.0,

    # --------------------------------------------------------
    # Illumination / camera variation
    # --------------------------------------------------------
    "hsv_h": 0.008,
    "hsv_s": 0.16,
    "hsv_v": 0.18,

    # --------------------------------------------------------
    # Composite augmentation
    #
    # Y1:
    # mosaic=0.30
    # copy_paste=0.10
    #
    # For Y2 these are deliberately reduced because
    # clinically unrealistic mouth layouts can hurt
    # processing-area semantics and lesion boundaries.
    # --------------------------------------------------------
    "mosaic": 0.10,
    "close_mosaic": 30,

    "mixup": 0.0,
    "copy_paste": 0.0,
}


# ============================================================
# PHASE 1 TRAINING
# ============================================================

print("=" * 100)
print("AI_Y2 — TRAINING PHASE 1")
print("GENERAL ROBUST SEGMENTATION")
print("=" * 100)

print(f"""
Base checkpoint : {MODEL_CHECKPOINT}
Dataset         : {DATA_YAML}

Input           : 640
Batch           : 16
Mask ratio      : 2

Max epochs      : {PHASE1_ARGS['epochs']}
Patience        : {PHASE1_ARGS['patience']}

Optimizer       : AdamW
Initial LR      : {PHASE1_ARGS['lr0']}
Cosine LR       : True

Mosaic          : {PHASE1_ARGS['mosaic']}
Copy paste      : {PHASE1_ARGS['copy_paste']}

Class 0         : processing_area
Class 1         : lesion
""")

phase1_model = YOLO(
    MODEL_CHECKPOINT
)

p1_start = time.time()

phase1_model.train(
    data=str(DATA_YAML),
    project=str(RUNS_ROOT),
    name=PHASE1_NAME,
    **PHASE1_ARGS
)

p1_minutes = (
    time.time() - p1_start
) / 60.0


PHASE1_BEST = (
    PHASE1_DIR /
    "weights" /
    "best.pt"
)

PHASE1_LAST = (
    PHASE1_DIR /
    "weights" /
    "last.pt"
)


if not PHASE1_BEST.exists():

    raise FileNotFoundError(
        f"❌ Phase 1 best.pt missing:\n"
        f"{PHASE1_BEST}"
    )


# ============================================================
# VERIFY PHASE 1 CLASS CONTRACT
# ============================================================

p1_check = YOLO(
    str(PHASE1_BEST)
)

p1_names = {
    int(k): str(v)
    for k, v
    in p1_check.names.items()
}

if p1_names != EXPECTED_CLASSES:

    raise RuntimeError(
        "❌ PHASE 1 COMPATIBILITY FAILURE\n"
        f"Expected: {EXPECTED_CLASSES}\n"
        f"Found   : {p1_names}"
    )


shutil.copy2(
    PHASE1_BEST,
    PHASE1_CANDIDATE
)


print("\n" + "=" * 100)
print("✅ PHASE 1 COMPLETE")
print("=" * 100)

print(
    f"Time             : "
    f"{p1_minutes:.2f} min"
)

print(
    f"Best checkpoint  : "
    f"{PHASE1_BEST}"
)

print(
    f"Candidate copy   : "
    f"{PHASE1_CANDIDATE}"
)

print(
    f"Classes verified : "
    f"{p1_names}"
)


# ============================================================
# CLEAR GPU BEFORE REFINEMENT
# ============================================================

del phase1_model
del p1_check

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# PHASE 2
#
# Boundary/refinement fine tuning.
#
# Starts from Phase-1 BEST, not last.pt.
#
# Goal:
# - preserve learned generalization
# - improve lesion boundary
# - reduce artificial augmentation influence
# - stabilize processing-area mask
# ============================================================

PHASE2_ARGS = {

    **COMMON,

    "epochs": 120,
    "patience": 30,

    # much lower LR
    "lr0": 0.00010,
    "lrf": 0.05,

    "warmup_epochs": 2.0,

    # --------------------------------------------------------
    # Very mild geometry for refinement
    # --------------------------------------------------------
    "degrees": 3.0,
    "translate": 0.02,
    "scale": 0.05,

    "shear": 0.0,
    "perspective": 0.0,

    "fliplr": 0.50,
    "flipud": 0.0,

    # --------------------------------------------------------
    # Very mild illumination variation
    # --------------------------------------------------------
    "hsv_h": 0.004,
    "hsv_s": 0.08,
    "hsv_v": 0.10,

    # --------------------------------------------------------
    # NO composite augmentation in refinement.
    # Important for exact oral geometry and lesion borders.
    # --------------------------------------------------------
    "mosaic": 0.0,
    "close_mosaic": 0,

    "mixup": 0.0,
    "copy_paste": 0.0,
}


# ============================================================
# PHASE 2 TRAINING
# ============================================================

print("\n" + "=" * 100)
print("AI_Y2 — TRAINING PHASE 2")
print("LOW-LR LESION / MASK REFINEMENT")
print("=" * 100)

print(f"""
Starting weights : Phase-1 BEST
{PHASE1_BEST}

Input            : 640
Mask ratio       : 2

Max epochs       : {PHASE2_ARGS['epochs']}
Patience         : {PHASE2_ARGS['patience']}

Initial LR       : {PHASE2_ARGS['lr0']}

Mosaic           : 0
Copy paste       : 0
MixUp            : 0

Purpose:
- cleaner lesion edges
- stable processing-area boundaries
- less augmentation distortion
""")


phase2_model = YOLO(
    str(PHASE1_BEST)
)

p2_start = time.time()

phase2_model.train(
    data=str(DATA_YAML),
    project=str(RUNS_ROOT),
    name=PHASE2_NAME,
    **PHASE2_ARGS
)

p2_minutes = (
    time.time() - p2_start
) / 60.0


PHASE2_BEST = (
    PHASE2_DIR /
    "weights" /
    "best.pt"
)

PHASE2_LAST = (
    PHASE2_DIR /
    "weights" /
    "last.pt"
)


if not PHASE2_BEST.exists():

    raise FileNotFoundError(
        f"❌ Phase 2 best.pt missing:\n"
        f"{PHASE2_BEST}"
    )


# ============================================================
# VERIFY PHASE 2 CLASS CONTRACT
# ============================================================

p2_check = YOLO(
    str(PHASE2_BEST)
)

p2_names = {
    int(k): str(v)
    for k, v
    in p2_check.names.items()
}

if p2_names != EXPECTED_CLASSES:

    raise RuntimeError(
        "❌ PHASE 2 COMPATIBILITY FAILURE\n"
        f"Expected: {EXPECTED_CLASSES}\n"
        f"Found   : {p2_names}"
    )


shutil.copy2(
    PHASE2_BEST,
    PHASE2_CANDIDATE
)


# ============================================================
# SAVE CANDIDATE METADATA
# ============================================================

training_info = {

    "compatibility_contract": {
        "model_family":
            "YOLO26-M segmentation",

        "imgsz":
            640,

        "classes": {
            "0": "processing_area",
            "1": "lesion",
        },
    },

    "dataset": {
        "path":
            str(DATA_YAML),

        "test_used_for_training":
            False,
    },

    "phase1": {
        "best":
            str(PHASE1_CANDIDATE),

        "minutes":
            p1_minutes,

        "max_epochs":
            PHASE1_ARGS["epochs"],

        "patience":
            PHASE1_ARGS["patience"],

        "lr0":
            PHASE1_ARGS["lr0"],

        "mask_ratio":
            PHASE1_ARGS["mask_ratio"],
    },

    "phase2": {
        "best":
            str(PHASE2_CANDIDATE),

        "minutes":
            p2_minutes,

        "max_epochs":
            PHASE2_ARGS["epochs"],

        "patience":
            PHASE2_ARGS["patience"],

        "lr0":
            PHASE2_ARGS["lr0"],

        "mask_ratio":
            PHASE2_ARGS["mask_ratio"],
    },

    "important":
        "No final model selected yet. "
        "Phase 1 and Phase 2 must be compared "
        "on validation data before final test evaluation.",
}


with open(
    TRAINING_INFO_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        training_info,
        f,
        indent=2
    )


# ============================================================
# FINAL CELL SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("MODULE 2 — CELL 2 COMPLETE")
print("=" * 100)

print(f"""
PHASE 1
-------
Training time:
{p1_minutes:.2f} minutes

Candidate:
{PHASE1_CANDIDATE}

PHASE 2
-------
Training time:
{p2_minutes:.2f} minutes

Candidate:
{PHASE2_CANDIDATE}

CLASS CONTRACT VERIFIED:
0 = processing_area
1 = lesion

INPUT CONTRACT:
640 × 640

TEST SET USED:
NO

FINAL MODEL SELECTED:
NO — intentionally.

Next cell will compare Phase 1 vs Phase 2
on VALIDATION only, including:

- overall mask precision / recall
- overall mask mAP50
- overall mask mAP50-95
- per-class metrics
- processing-area behavior
- lesion behavior
- false lesion predictions on processing-area-only images

Then only the superior validated candidate
will be allowed to touch the TEST set.
""")

print(
    f"Metadata:\n"
    f"{TRAINING_INFO_JSON}"
)

print("=" * 100)
print("✅ CELL 2 PASSED")
print("=" * 100)

AI_Y2 — TRAINING PHASE 1
GENERAL ROBUST SEGMENTATION

Base checkpoint : yolo26m-seg.pt
Dataset         : /content/AI_Y2_FINAL/data.yaml

Input           : 640
Batch           : 16
Mask ratio      : 2

Max epochs      : 350
Patience        : 55

Optimizer       : AdamW
Initial LR      : 0.0004
Cosine LR       : True

Mosaic          : 0.1
Copy paste      : 0.0

Class 0         : processing_area
Class 1         : lesion

Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=30, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/AI_Y2_FINAL/data.yaml, degrees=8.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=Fal

In [ ]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 3
#
# VALIDATION-ONLY CANDIDATE COMPARISON
#
# Compares:
#   Phase 1 vs Phase 2
#
# PRIMARY COMPATIBILITY POINT:
#   imgsz = 640
#   conf  = 0.25
#
# Measures:
# - standard YOLO mask metrics
# - per-class metrics
# - lesion-positive image detection rate
# - false-lesion rate on processing-only images
# - multiple false lesions
#
# TEST SET IS NOT USED.
# ============================================================

from pathlib import Path
from collections import defaultdict, Counter
import json
import math
import gc

import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO


# ============================================================
# REQUIRED PATHS
# ============================================================

PHASE1_CANDIDATE = Path(
    "/content/AI_Y2_MODULE2/candidate_y2_phase1.pt"
)

PHASE2_CANDIDATE = Path(
    "/content/AI_Y2_MODULE2/candidate_y2_phase2.pt"
)

DATASET_ROOT = Path("/content/AI_Y2_FINAL")
DATA_YAML = DATASET_ROOT / "data.yaml"

VAL_IMG_DIR = DATASET_ROOT / "valid" / "images"
VAL_LBL_DIR = DATASET_ROOT / "valid" / "labels"

REPORTS_ROOT = Path(
    "/content/AI_Y2_MODULE2/reports"
)

REPORTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

COMPARISON_JSON = (
    REPORTS_ROOT /
    "validation_candidate_comparison.json"
)


EXPECTED_NAMES = {
    0: "processing_area",
    1: "lesion",
}

IMGSZ = 640

# Keep Y1-compatible primary inference threshold
PRIMARY_CONF = 0.25
PRIMARY_IOU = 0.70

# Analysis only — does not alter final software threshold
CONF_SWEEP = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.50,
]


# ============================================================
# CHECK FILES
# ============================================================

for p in [
    PHASE1_CANDIDATE,
    PHASE2_CANDIDATE,
    DATA_YAML,
    VAL_IMG_DIR,
    VAL_LBL_DIR,
]:
    assert p.exists(), f"❌ Missing: {p}"


# ============================================================
# GROUND-TRUTH IMAGE LABEL TYPE
# ============================================================

def get_gt_image_type(label_path):
    """
    Returns:
        lesion_positive = True/False
        processing_count
        lesion_count
    """

    processing = 0
    lesion = 0

    text = label_path.read_text(
        encoding="utf-8"
    ).strip()

    for line in text.splitlines():

        cls_id = int(
            line.split()[0]
        )

        if cls_id == 0:
            processing += 1

        elif cls_id == 1:
            lesion += 1

    return {
        "lesion_positive": lesion > 0,
        "processing_count": processing,
        "lesion_count": lesion,
    }


# ============================================================
# BUILD VALIDATION IMAGE INDEX
# ============================================================

val_images = sorted(
    p for p in VAL_IMG_DIR.iterdir()
    if p.is_file()
)

gt_by_image = {}

for img in val_images:

    lbl = (
        VAL_LBL_DIR /
        f"{img.stem}.txt"
    )

    assert lbl.exists()

    gt_by_image[img.name] = (
        get_gt_image_type(lbl)
    )


num_val = len(val_images)

num_lesion_positive = sum(
    x["lesion_positive"]
    for x in gt_by_image.values()
)

num_processing_only = (
    num_val - num_lesion_positive
)


print("=" * 100)
print("AI_Y2 MODULE 2 — CELL 3")
print("VALIDATION CANDIDATE COMPARISON")
print("=" * 100)

print(f"""
Validation images         : {num_val}
Lesion-positive images    : {num_lesion_positive}
Processing-only images    : {num_processing_only}

PRIMARY COMPATIBILITY:
imgsz = {IMGSZ}
conf  = {PRIMARY_CONF}
iou   = {PRIMARY_IOU}

TEST SET:
NOT USED
""")


# ============================================================
# ULTRALYTICS METRIC EXTRACTION
# ============================================================

def safe_float(x):

    try:
        return float(x)

    except Exception:
        return None


def run_standard_validation(model_path):

    model = YOLO(
        str(model_path)
    )

    names = {
        int(k): str(v)
        for k, v in model.names.items()
    }

    if names != EXPECTED_NAMES:

        raise RuntimeError(
            f"❌ Compatibility failure:\n"
            f"{model_path}\n"
            f"{names}"
        )

    metrics = model.val(
        data=str(DATA_YAML),
        split="val",

        imgsz=IMGSZ,
        conf=PRIMARY_CONF,
        iou=PRIMARY_IOU,

        device=0,
        batch=16,

        plots=False,
        save_json=False,
        verbose=False,
    )

    # Overall mask metrics
    result = {
        "mask_precision":
            safe_float(metrics.seg.mp),

        "mask_recall":
            safe_float(metrics.seg.mr),

        "mask_map50":
            safe_float(metrics.seg.map50),

        "mask_map50_95":
            safe_float(metrics.seg.map),

        "box_precision":
            safe_float(metrics.box.mp),

        "box_recall":
            safe_float(metrics.box.mr),

        "box_map50":
            safe_float(metrics.box.map50),

        "box_map50_95":
            safe_float(metrics.box.map),
    }

    # Per-class mask metrics
    # Ultralytics exposes maps = mAP50-95 per class.
    if hasattr(metrics.seg, "maps"):

        maps = np.array(
            metrics.seg.maps,
            dtype=float
        )

        if len(maps) >= 2:

            result[
                "processing_mask_map50_95"
            ] = float(maps[0])

            result[
                "lesion_mask_map50_95"
            ] = float(maps[1])

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return result


# ============================================================
# TARGETED IMAGE-LEVEL BEHAVIOR
# ============================================================

def targeted_eval(
    model_path,
    conf,
):

    model = YOLO(
        str(model_path)
    )

    names = {
        int(k): str(v)
        for k, v in model.names.items()
    }

    assert names == EXPECTED_NAMES

    results = model.predict(
        source=[
            str(p)
            for p in val_images
        ],

        imgsz=IMGSZ,
        conf=conf,
        iou=PRIMARY_IOU,

        device=0,
        batch=16,

        retina_masks=True,

        verbose=False,
        save=False,
        stream=False,
    )

    lesion_positive_seen = 0

    processing_only_fp_images = 0
    processing_only_fp_instances = 0

    multiple_false_lesion_images = 0

    processing_detected_images = 0

    total_predicted_lesions = 0
    total_predicted_processing = 0

    lesion_confidences = []
    processing_confidences = []

    for img_path, result in zip(
        val_images,
        results
    ):

        gt = gt_by_image[
            img_path.name
        ]

        predicted_classes = []

        if (
            result.boxes is not None
            and
            len(result.boxes) > 0
        ):

            cls_array = (
                result.boxes.cls
                .detach()
                .cpu()
                .numpy()
                .astype(int)
            )

            conf_array = (
                result.boxes.conf
                .detach()
                .cpu()
                .numpy()
            )

            predicted_classes = (
                cls_array.tolist()
            )

            for cls_id, score in zip(
                cls_array,
                conf_array
            ):

                if cls_id == 0:

                    total_predicted_processing += 1
                    processing_confidences.append(
                        float(score)
                    )

                elif cls_id == 1:

                    total_predicted_lesions += 1
                    lesion_confidences.append(
                        float(score)
                    )

        lesion_preds = sum(
            cls_id == 1
            for cls_id in predicted_classes
        )

        processing_preds = sum(
            cls_id == 0
            for cls_id in predicted_classes
        )

        if processing_preds > 0:
            processing_detected_images += 1

        # lesion-positive GT image:
        # did model detect at least one lesion?
        if gt["lesion_positive"]:

            if lesion_preds > 0:
                lesion_positive_seen += 1

        # processing-only GT image:
        # any lesion is a false lesion event
        else:

            if lesion_preds > 0:

                processing_only_fp_images += 1

                processing_only_fp_instances += (
                    lesion_preds
                )

                if lesion_preds > 1:
                    multiple_false_lesion_images += 1


    lesion_image_recall = (
        lesion_positive_seen
        /
        max(num_lesion_positive, 1)
    )

    false_lesion_image_rate = (
        processing_only_fp_images
        /
        max(num_processing_only, 1)
    )

    processing_presence_rate = (
        processing_detected_images
        /
        num_val
    )


    out = {

        "conf": conf,

        "lesion_positive_images":
            num_lesion_positive,

        "lesion_positive_detected":
            lesion_positive_seen,

        "lesion_image_recall":
            lesion_image_recall,

        "processing_only_images":
            num_processing_only,

        "processing_only_false_lesion_images":
            processing_only_fp_images,

        "false_lesion_image_rate":
            false_lesion_image_rate,

        "false_lesion_instances":
            processing_only_fp_instances,

        "multiple_false_lesion_images":
            multiple_false_lesion_images,

        "processing_presence_rate":
            processing_presence_rate,

        "total_predicted_lesions":
            total_predicted_lesions,

        "total_predicted_processing":
            total_predicted_processing,

        "mean_lesion_conf":
            (
                float(
                    np.mean(
                        lesion_confidences
                    )
                )
                if lesion_confidences
                else None
            ),

        "mean_processing_conf":
            (
                float(
                    np.mean(
                        processing_confidences
                    )
                )
                if processing_confidences
                else None
            ),
    }


    del model
    del results

    gc.collect()
    torch.cuda.empty_cache()

    return out


# ============================================================
# RUN BOTH CANDIDATES
# ============================================================

CANDIDATES = {
    "phase1": PHASE1_CANDIDATE,
    "phase2": PHASE2_CANDIDATE,
}

comparison = {}


for name, path in CANDIDATES.items():

    print("\n" + "=" * 100)
    print(f"{name.upper()} — STANDARD VALIDATION")
    print("=" * 100)

    standard = run_standard_validation(
        path
    )

    print(
        json.dumps(
            standard,
            indent=2
        )
    )

    print(
        f"\n{name.upper()} "
        f"— TARGETED @ conf={PRIMARY_CONF}"
    )

    primary_targeted = targeted_eval(
        path,
        PRIMARY_CONF
    )

    print(
        json.dumps(
            primary_targeted,
            indent=2
        )
    )

    comparison[name] = {
        "path": str(path),

        "standard":
            standard,

        "primary_targeted":
            primary_targeted,

        "threshold_sweep": [],
    }


# ============================================================
# CONFIDENCE SWEEP — ANALYSIS ONLY
# ============================================================

print("\n" + "=" * 100)
print("CONFIDENCE SWEEP — VALIDATION ONLY")
print("ANALYSIS ONLY; Y1-compatible primary threshold remains 0.25")
print("=" * 100)


rows = []


for candidate_name, path in CANDIDATES.items():

    for conf in CONF_SWEEP:

        r = targeted_eval(
            path,
            conf
        )

        comparison[
            candidate_name
        ][
            "threshold_sweep"
        ].append(r)

        rows.append({
            "candidate":
                candidate_name,

            "conf":
                conf,

            "lesion_image_recall":
                r["lesion_image_recall"],

            "false_lesion_image_rate":
                r["false_lesion_image_rate"],

            "false_lesion_images":
                r[
                    "processing_only_false_lesion_images"
                ],

            "false_lesion_instances":
                r[
                    "false_lesion_instances"
                ],

            "processing_presence_rate":
                r[
                    "processing_presence_rate"
                ],
        })


sweep_df = pd.DataFrame(
    rows
)


print(
    sweep_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# PRIMARY CONF=0.25 COMPARISON
# ============================================================

print("\n" + "=" * 100)
print("PRIMARY DROP-IN COMPARISON @ CONF=0.25")
print("=" * 100)


primary_rows = []


for name in [
    "phase1",
    "phase2"
]:

    s = comparison[
        name
    ]["standard"]

    t = comparison[
        name
    ]["primary_targeted"]

    primary_rows.append({

        "candidate":
            name,

        "mask_P":
            s["mask_precision"],

        "mask_R":
            s["mask_recall"],

        "mask_mAP50":
            s["mask_map50"],

        "mask_mAP50_95":
            s["mask_map50_95"],

        "lesion_mAP50_95":
            s.get(
                "lesion_mask_map50_95"
            ),

        "processing_mAP50_95":
            s.get(
                "processing_mask_map50_95"
            ),

        "lesion_image_recall":
            t["lesion_image_recall"],

        "false_lesion_image_rate":
            t[
                "false_lesion_image_rate"
            ],

        "false_lesion_images":
            t[
                "processing_only_false_lesion_images"
            ],

        "processing_presence_rate":
            t[
                "processing_presence_rate"
            ],
    })


primary_df = pd.DataFrame(
    primary_rows
)


print(
    primary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# SAFETY-FIRST SELECTION SCORE
#
# Lesion quality is weighted most heavily.
# False lesions on processing-only images are penalized.
#
# This selection uses VALIDATION ONLY.
# ============================================================

def candidate_score(row):

    # weights intentionally lesion-oriented
    return (

        0.35
        * row[
            "lesion_mAP50_95"
        ]

        +

        0.25
        * row[
            "lesion_image_recall"
        ]

        +

        0.15
        * row[
            "mask_mAP50"
        ]

        +

        0.10
        * row[
            "processing_mAP50_95"
        ]

        +

        0.05
        * row[
            "processing_presence_rate"
        ]

        +

        0.10
        * (
            1.0
            -
            row[
                "false_lesion_image_rate"
            ]
        )
    )


scores = {}


for row in primary_rows:

    scores[
        row["candidate"]
    ] = candidate_score(
        row
    )


print("\nSelection scores:")
for name, score in scores.items():
    print(
        f"  {name}: {score:.6f}"
    )


recommended = max(
    scores,
    key=scores.get
)


print("\n" + "=" * 100)
print("VALIDATION SELECTION")
print("=" * 100)

print(
    f"Recommended candidate: "
    f"{recommended.upper()}"
)

print(
    f"Path:\n"
    f"{comparison[recommended]['path']}"
)

print("""
IMPORTANT:
This is NOT yet the final production model.

The TEST set remains untouched.

Next cell will:
1. lock this validation-selected candidate,
2. evaluate it ONCE on test,
3. compare against Y1 reference numbers where comparison is valid,
4. verify class/input/output compatibility,
5. copy the accepted model to /content/ai_y2.pt.
""")


# ============================================================
# SAVE REPORT
# ============================================================

output = {

    "primary_conf":
        PRIMARY_CONF,

    "primary_iou":
        PRIMARY_IOU,

    "imgsz":
        IMGSZ,

    "test_used":
        False,

    "comparison":
        comparison,

    "selection_scores":
        scores,

    "recommended_candidate":
        recommended,

    "recommended_path":
        comparison[
            recommended
        ]["path"],
}


with open(
    COMPARISON_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2
    )


print(
    f"\nReport saved:\n"
    f"{COMPARISON_JSON}"
)

print("\n" + "=" * 100)
print("✅ MODULE 2 CELL 3 COMPLETE")
print("=" * 100)

AI_Y2 MODULE 2 — CELL 3
VALIDATION CANDIDATE COMPARISON

Validation images         : 173
Lesion-positive images    : 133
Processing-only images    : 40

PRIMARY COMPATIBILITY:
imgsz = 640
conf  = 0.25
iou   = 0.7

TEST SET:
NOT USED


PHASE1 — STANDARD VALIDATION
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLO26m-seg summary (fused): 146 layers, 23,509,010 parameters, 0 gradients, 121.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1395.8±327.3 MB/s, size: 40.8 KB)
val: Scanning /content/AI_Y2_FINAL/valid/labels.cache... 173 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 173/173 48.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.5s/it 16.3s
                   all        173        336      0.889      0.789      0.794      0.512      0.734      0.639      0.566      0.263
Speed: 1.1ms preprocess,

In [ ]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 4
#
# FINAL TEST + DROP-IN COMPATIBILITY GATE + MODEL FREEZE
#
# Selected candidate:
#   PHASE 1
#
# FINAL CONTRACT:
#   YOLO26-M segmentation
#   imgsz = 640
#   class 0 = processing_area
#   class 1 = lesion
#
# TEST SET IS EVALUATED ONCE HERE.
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import json
import shutil
import gc

import numpy as np
import torch
from ultralytics import YOLO


# ============================================================
# PATHS
# ============================================================

SELECTED_MODEL = Path(
    "/content/AI_Y2_MODULE2/candidate_y2_phase1.pt"
)

DATASET_ROOT = Path(
    "/content/AI_Y2_FINAL"
)

DATA_YAML = (
    DATASET_ROOT /
    "data.yaml"
)

TEST_IMG_DIR = (
    DATASET_ROOT /
    "test" /
    "images"
)

TEST_LBL_DIR = (
    DATASET_ROOT /
    "test" /
    "labels"
)

REPORTS_ROOT = Path(
    "/content/AI_Y2_MODULE2/reports"
)

REPORTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_MODEL_PATH = Path(
    "/content/ai_y2.pt"
)

FINAL_REPORT_JSON = (
    REPORTS_ROOT /
    "AI_Y2_final_test_report.json"
)


# ============================================================
# OPTIONAL Y1 MODEL
#
# For TRUE apples-to-apples Y1 vs Y2 comparison,
# place Y1 weights at one of these locations.
#
# If not found, comparison is skipped.
# ============================================================

Y1_CANDIDATE_PATHS = [

    Path(
        "/content/best_oralscan_yolo26m_seg.pt"
    ),

    Path(
        "/content/ai_y1.pt"
    ),

    Path(
        "/content/y1.pt"
    ),
]


Y1_MODEL_PATH = next(
    (
        p
        for p in Y1_CANDIDATE_PATHS
        if p.exists()
    ),
    None
)


# ============================================================
# CONTRACT
# ============================================================

EXPECTED_NAMES = {
    0: "processing_area",
    1: "lesion",
}

IMGSZ = 640

CONF = 0.25

IOU = 0.70


# ============================================================
# HISTORICAL Y1 REFERENCE
#
# These numbers are ONLY historical context.
# They are NOT a strict apples-to-apples comparison because
# Y1 used a different dataset/split.
# ============================================================

Y1_HISTORICAL = {

    "mask_precision":
        0.6125,

    "mask_recall":
        0.5271,

    "mask_map50":
        0.4683,

    "mask_map50_95":
        0.2081,

    "processing_mask_map50_95":
        0.2527,

    "lesion_mask_map50_95":
        0.1636,
}


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


def safe_float(x):

    try:
        return float(x)

    except Exception:
        return None


def get_gt_info(
    label_path
):

    processing = 0
    lesion = 0

    text = label_path.read_text(
        encoding="utf-8"
    ).strip()

    for line in text.splitlines():

        cls = int(
            line.split()[0]
        )

        if cls == 0:
            processing += 1

        elif cls == 1:
            lesion += 1

    return {
        "processing":
            processing,

        "lesion":
            lesion,

        "lesion_positive":
            lesion > 0,
    }


# ============================================================
# PRECHECK
# ============================================================

print("=" * 104)
print("AI_Y2 MODULE 2 — CELL 4")
print("FINAL TEST + COMPATIBILITY GATE + MODEL FREEZE")
print("=" * 104)


for p in [
    SELECTED_MODEL,
    DATA_YAML,
    TEST_IMG_DIR,
    TEST_LBL_DIR,
]:

    if not p.exists():

        raise FileNotFoundError(
            f"❌ Missing:\n{p}"
        )


# ============================================================
# LOAD SELECTED Y2
# ============================================================

y2 = YOLO(
    str(SELECTED_MODEL)
)


y2_names = {
    int(k): str(v)
    for k, v
    in y2.names.items()
}


print("\n" + "=" * 104)
print("Y2 COMPATIBILITY CONTRACT")
print("=" * 104)

print(
    f"Task       : {y2.task}"
)

print(
    f"Classes    : {y2_names}"
)

print(
    f"Input size : {IMGSZ}"
)


if str(y2.task) != "segment":

    raise RuntimeError(
        "❌ Y2 is not segmentation model."
    )


if y2_names != EXPECTED_NAMES:

    raise RuntimeError(
        "\n❌ CLASS CONTRACT FAILURE\n"
        f"Expected: {EXPECTED_NAMES}\n"
        f"Found   : {y2_names}"
    )


print(
    "✅ Y1-compatible class IDs/names verified."
)


# ============================================================
# TEST DATA SUMMARY
# ============================================================

test_images = sorted(
    p
    for p in TEST_IMG_DIR.iterdir()
    if p.is_file()
)


gt = {}


for img in test_images:

    lbl = (
        TEST_LBL_DIR /
        f"{img.stem}.txt"
    )

    assert lbl.exists()

    gt[
        img.name
    ] = get_gt_info(
        lbl
    )


n_test = len(
    test_images
)

n_lesion_positive = sum(
    info[
        "lesion_positive"
    ]
    for info in gt.values()
)

n_processing_only = (
    n_test
    -
    n_lesion_positive
)


print("\n" + "=" * 104)
print("FROZEN TEST SET")
print("=" * 104)

print(
    f"Images                  : {n_test}"
)

print(
    f"Lesion-positive images  : {n_lesion_positive}"
)

print(
    f"Processing-only images  : {n_processing_only}"
)


assert n_test == 115, (
    f"❌ Expected 115 test images, got {n_test}"
)


# ============================================================
# STANDARD FINAL TEST
# ============================================================

print("\n" + "=" * 104)
print("FINAL Y2 TEST — STANDARD METRICS")
print("=" * 104)


y2_metrics = y2.val(

    data=str(
        DATA_YAML
    ),

    split="test",

    imgsz=IMGSZ,

    conf=CONF,

    iou=IOU,

    batch=16,

    device=0,

    plots=True,

    save_json=False,

    verbose=True,
)


y2_standard = {

    "mask_precision":
        safe_float(
            y2_metrics.seg.mp
        ),

    "mask_recall":
        safe_float(
            y2_metrics.seg.mr
        ),

    "mask_map50":
        safe_float(
            y2_metrics.seg.map50
        ),

    "mask_map50_95":
        safe_float(
            y2_metrics.seg.map
        ),

    "box_precision":
        safe_float(
            y2_metrics.box.mp
        ),

    "box_recall":
        safe_float(
            y2_metrics.box.mr
        ),

    "box_map50":
        safe_float(
            y2_metrics.box.map50
        ),

    "box_map50_95":
        safe_float(
            y2_metrics.box.map
        ),
}


if hasattr(
    y2_metrics.seg,
    "maps"
):

    maps = np.array(
        y2_metrics.seg.maps,
        dtype=float
    )

    if len(maps) >= 2:

        y2_standard[
            "processing_mask_map50_95"
        ] = float(
            maps[0]
        )

        y2_standard[
            "lesion_mask_map50_95"
        ] = float(
            maps[1]
        )


print("\nY2 TEST METRICS")

print(
    json.dumps(
        y2_standard,
        indent=2
    )
)


# ============================================================
# TARGETED TEST BEHAVIOR
#
# Same inference contract as Y1:
# imgsz=640 / conf=0.25
# ============================================================

print("\n" + "=" * 104)
print("FINAL Y2 TEST — TARGETED BEHAVIOR @ CONF=0.25")
print("=" * 104)


predictions = y2.predict(

    source=[
        str(p)
        for p in test_images
    ],

    imgsz=IMGSZ,

    conf=CONF,

    iou=IOU,

    device=0,

    batch=16,

    retina_masks=True,

    verbose=False,

    save=False,
)


lesion_positive_detected = 0

false_lesion_images = 0

false_lesion_instances = 0

processing_detected_images = 0

predicted_lesions = 0

predicted_processing = 0


for img_path, result in zip(
    test_images,
    predictions
):

    info = gt[
        img_path.name
    ]

    classes = []

    if (
        result.boxes is not None
        and
        len(result.boxes) > 0
    ):

        classes = (
            result.boxes.cls
            .detach()
            .cpu()
            .numpy()
            .astype(int)
            .tolist()
        )


    lesion_count = sum(
        c == 1
        for c in classes
    )

    processing_count = sum(
        c == 0
        for c in classes
    )


    predicted_lesions += (
        lesion_count
    )

    predicted_processing += (
        processing_count
    )


    if processing_count > 0:

        processing_detected_images += 1


    if info[
        "lesion_positive"
    ]:

        if lesion_count > 0:

            lesion_positive_detected += 1

    else:

        if lesion_count > 0:

            false_lesion_images += 1

            false_lesion_instances += (
                lesion_count
            )


targeted = {

    "lesion_positive_images":
        n_lesion_positive,

    "lesion_positive_detected":
        lesion_positive_detected,

    "lesion_image_recall":
        (
            lesion_positive_detected
            /
            max(
                n_lesion_positive,
                1
            )
        ),

    "processing_only_images":
        n_processing_only,

    "false_lesion_images":
        false_lesion_images,

    "false_lesion_image_rate":
        (
            false_lesion_images
            /
            max(
                n_processing_only,
                1
            )
        ),

    "false_lesion_instances":
        false_lesion_instances,

    "processing_presence_rate":
        (
            processing_detected_images
            /
            n_test
        ),

    "total_predicted_lesions":
        predicted_lesions,

    "total_predicted_processing":
        predicted_processing,
}


print(
    json.dumps(
        targeted,
        indent=2
    )
)


# ============================================================
# OUTPUT-SCHEMA / SOFTWARE COMPATIBILITY TEST
#
# Existing Y1 software expects:
#   result.boxes
#   boxes.cls
#   boxes.conf
#   boxes.xyxy
#   result.masks
#   masks.data
#   masks.xy
# ============================================================

print("\n" + "=" * 104)
print("DROP-IN SOFTWARE COMPATIBILITY TEST")
print("=" * 104)


schema_sample = None


# Find a test image where detections exist
for img_path in test_images:

    r = y2.predict(

        source=str(
            img_path
        ),

        imgsz=IMGSZ,

        conf=CONF,

        iou=IOU,

        device=0,

        retina_masks=True,

        verbose=False,
    )[0]


    if (
        r.boxes is not None
        and
        len(r.boxes) > 0
        and
        r.masks is not None
    ):

        schema_sample = (
            img_path,
            r
        )

        break


if schema_sample is None:

    raise RuntimeError(
        "❌ Could not obtain prediction "
        "for output-schema test."
    )


sample_path, r = schema_sample


# Required Y1-compatible fields
assert r.boxes is not None

assert hasattr(
    r.boxes,
    "cls"
)

assert hasattr(
    r.boxes,
    "conf"
)

assert hasattr(
    r.boxes,
    "xyxy"
)

assert r.masks is not None

assert hasattr(
    r.masks,
    "data"
)

assert hasattr(
    r.masks,
    "xy"
)


# Every class must remain 0/1
sample_classes = (
    r.boxes.cls
    .detach()
    .cpu()
    .numpy()
    .astype(int)
)


assert set(
    sample_classes.tolist()
).issubset(
    {0, 1}
)


# Polygon count should correspond to masks
polygon_count = len(
    r.masks.xy
)

mask_count = int(
    r.masks.data.shape[0]
)

box_count = len(
    r.boxes
)


assert box_count == mask_count, (
    f"❌ boxes={box_count}, masks={mask_count}"
)

assert polygon_count == mask_count, (
    f"❌ polygons={polygon_count}, masks={mask_count}"
)


print(
    f"Sample image     : {sample_path.name}"
)

print(
    f"Boxes            : {box_count}"
)

print(
    f"Masks            : {mask_count}"
)

print(
    f"Polygons         : {polygon_count}"
)

print(
    f"Class IDs        : {sample_classes.tolist()}"
)

print(
    "result.boxes.cls  ✅"
)

print(
    "result.boxes.conf ✅"
)

print(
    "result.boxes.xyxy ✅"
)

print(
    "result.masks.data ✅"
)

print(
    "result.masks.xy   ✅"
)

print(
    "\n✅ Existing Y1 inference code can "
    "consume Y2 through the same Ultralytics API."
)


# ============================================================
# OPTIONAL: TRUE Y1 vs Y2 SAME-TEST COMPARISON
# ============================================================

same_test_y1 = None


if Y1_MODEL_PATH is not None:

    print("\n" + "=" * 104)
    print("Y1 vs Y2 — SAME Y2 TEST SET")
    print("=" * 104)

    print(
        f"Y1 model found:\n"
        f"{Y1_MODEL_PATH}"
    )

    y1 = YOLO(
        str(
            Y1_MODEL_PATH
        )
    )


    y1_names = {
        int(k): str(v)
        for k, v
        in y1.names.items()
    }


    print(
        f"Y1 names: {y1_names}"
    )


    if y1_names != EXPECTED_NAMES:

        print(
            "\n⚠️ Y1 weights found but class "
            "contract does not match."
        )

        print(
            "Skipping same-test comparison."
        )

    else:

        y1_metrics = y1.val(

            data=str(
                DATA_YAML
            ),

            split="test",

            imgsz=IMGSZ,

            conf=CONF,

            iou=IOU,

            batch=16,

            device=0,

            plots=False,

            verbose=False,
        )


        same_test_y1 = {

            "mask_precision":
                safe_float(
                    y1_metrics.seg.mp
                ),

            "mask_recall":
                safe_float(
                    y1_metrics.seg.mr
                ),

            "mask_map50":
                safe_float(
                    y1_metrics.seg.map50
                ),

            "mask_map50_95":
                safe_float(
                    y1_metrics.seg.map
                ),
        }


        if hasattr(
            y1_metrics.seg,
            "maps"
        ):

            m = np.array(
                y1_metrics.seg.maps,
                dtype=float
            )

            if len(m) >= 2:

                same_test_y1[
                    "processing_mask_map50_95"
                ] = float(
                    m[0]
                )

                same_test_y1[
                    "lesion_mask_map50_95"
                ] = float(
                    m[1]
                )


        print("\nSAME-TEST METRICS")

        print(
            f"{'METRIC':<32}"
            f"{'Y1':>12}"
            f"{'Y2':>12}"
            f"{'DELTA':>12}"
        )

        print("-" * 68)


        keys = [
            "mask_precision",
            "mask_recall",
            "mask_map50",
            "mask_map50_95",
            "processing_mask_map50_95",
            "lesion_mask_map50_95",
        ]


        for key in keys:

            if (
                key in same_test_y1
                and
                key in y2_standard
            ):

                a = (
                    same_test_y1[
                        key
                    ]
                )

                b = (
                    y2_standard[
                        key
                    ]
                )

                print(
                    f"{key:<32}"
                    f"{a:>12.4f}"
                    f"{b:>12.4f}"
                    f"{b-a:>+12.4f}"
                )


    del y1

    gc.collect()

    torch.cuda.empty_cache()


else:

    print("\n" + "=" * 104)
    print("Y1 SAME-TEST COMPARISON")
    print("=" * 104)

    print(
        "Y1 .pt weights not found in expected paths."
    )

    print(
        "Skipping apples-to-apples comparison."
    )


# ============================================================
# HISTORICAL Y1 REFERENCE
# ============================================================

print("\n" + "=" * 104)
print("HISTORICAL Y1 REFERENCE")
print("NOT APPLES-TO-APPLES — DIFFERENT TEST DATA")
print("=" * 104)


print(
    f"{'METRIC':<32}"
    f"{'Y1 historical':>16}"
    f"{'Y2 current test':>18}"
)

print("-" * 68)


for key in [
    "mask_precision",
    "mask_recall",
    "mask_map50",
    "mask_map50_95",
    "processing_mask_map50_95",
    "lesion_mask_map50_95",
]:

    if key in y2_standard:

        print(
            f"{key:<32}"
            f"{Y1_HISTORICAL[key]:>16.4f}"
            f"{y2_standard[key]:>18.4f}"
        )


print("""
NOTE:
Historical Y1 numbers are only context.
Because Y1 and Y2 used different test datasets,
they cannot prove improvement by themselves.

If Y1 .pt is available, the SAME-TEST section above
is the valid direct comparison.
""")


# ============================================================
# FINAL MODEL FREEZE
# ============================================================

print("\n" + "=" * 104)
print("FINAL Y2 MODEL FREEZE")
print("=" * 104)


if FINAL_MODEL_PATH.exists():

    FINAL_MODEL_PATH.unlink()


shutil.copy2(
    SELECTED_MODEL,
    FINAL_MODEL_PATH
)


# Reload copied model
final_model = YOLO(
    str(
        FINAL_MODEL_PATH
    )
)


final_names = {
    int(k): str(v)
    for k, v
    in final_model.names.items()
}


if final_names != EXPECTED_NAMES:

    raise RuntimeError(
        "❌ Final copied model class "
        "contract failed."
    )


if str(
    final_model.task
) != "segment":

    raise RuntimeError(
        "❌ Final model task is "
        "not segmentation."
    )


final_sha256 = sha256_file(
    FINAL_MODEL_PATH
)


selected_sha256 = sha256_file(
    SELECTED_MODEL
)


if final_sha256 != selected_sha256:

    raise RuntimeError(
        "❌ Final copied model differs "
        "from selected candidate."
    )


print(
    f"Final model:\n"
    f"{FINAL_MODEL_PATH}"
)

print(
    f"\nSHA256:\n"
    f"{final_sha256}"
)

print(
    f"\nClasses:\n"
    f"{final_names}"
)

print(
    f"\nTask:\n"
    f"{final_model.task}"
)


# ============================================================
# REPORT
# ============================================================

report = {

    "model":
        "AI_Y2",

    "selected_candidate":
        "phase1",

    "selected_candidate_path":
        str(
            SELECTED_MODEL
        ),

    "final_model_path":
        str(
            FINAL_MODEL_PATH
        ),

    "final_model_sha256":
        final_sha256,

    "compatibility": {

        "task":
            "segment",

        "imgsz":
            IMGSZ,

        "class_names": {
            "0":
                "processing_area",

            "1":
                "lesion",
        },

        "boxes_cls":
            True,

        "boxes_conf":
            True,

        "boxes_xyxy":
            True,

        "masks_data":
            True,

        "masks_xy":
            True,
    },

    "test_standard_metrics":
        y2_standard,

    "test_targeted_behavior":
        targeted,

    "same_test_y1_metrics":
        same_test_y1,

    "y1_historical_reference":
        Y1_HISTORICAL,

    "important":
        (
            "Y2 preserves the Y1 detector "
            "input/output class contract. "
            "Numeric predictions are expected "
            "to differ because Y2 is a "
            "different trained model."
        ),
}


with open(
    FINAL_REPORT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 104)
print("AI_Y2 FINAL SUMMARY")
print("=" * 104)


print(f"""
FINAL MODEL:
{FINAL_MODEL_PATH}

SELECTED:
Phase 1 validation-selected candidate

MODEL FAMILY:
YOLO26-M Segmentation

INPUT:
640 × 640

CLASSES:
0 = processing_area
1 = lesion

CONFIDENCE COMPATIBILITY POINT:
0.25

OUTPUT CONTRACT:
boxes.cls   ✅
boxes.conf  ✅
boxes.xyxy  ✅
masks.data  ✅
masks.xy    ✅

FINAL TEST:
Used exactly once in this selection pipeline.

TEST MASK:
Precision  = {y2_standard['mask_precision']:.4f}
Recall     = {y2_standard['mask_recall']:.4f}
mAP50      = {y2_standard['mask_map50']:.4f}
mAP50-95   = {y2_standard['mask_map50_95']:.4f}

PROCESSING mAP50-95:
{y2_standard.get('processing_mask_map50_95', float('nan')):.4f}

LESION mAP50-95:
{y2_standard.get('lesion_mask_map50_95', float('nan')):.4f}

LESION IMAGE DETECTION:
{targeted['lesion_positive_detected']}
/
{targeted['lesion_positive_images']}

Lesion-image recall:
{targeted['lesion_image_recall']:.4f}

FALSE LESION IMAGES:
{targeted['false_lesion_images']}
/
{targeted['processing_only_images']}

False-lesion image rate:
{targeted['false_lesion_image_rate']:.4f}

FINAL SHA256:
{final_sha256}

REPORT:
{FINAL_REPORT_JSON}
""")


print("=" * 104)
print("✅ AI_Y2 MODULE 2 CELL 4 COMPLETE")
print("=" * 104)

AI_Y2 MODULE 2 — CELL 4
FINAL TEST + COMPATIBILITY GATE + MODEL FREEZE

Y2 COMPATIBILITY CONTRACT
Task       : segment
Classes    : {0: 'processing_area', 1: 'lesion'}
Input size : 640
✅ Y1-compatible class IDs/names verified.

FROZEN TEST SET
Images                  : 115
Lesion-positive images  : 89
Processing-only images  : 26

FINAL Y2 TEST — STANDARD METRICS
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLO26m-seg summary (fused): 146 layers, 23,509,010 parameters, 0 gradients, 121.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1490.9±484.8 MB/s, size: 43.1 KB)
val: Scanning /content/AI_Y2_FINAL/test/labels... 115 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 115/115 1.3Kit/s 0.1s
val: New cache created: /content/AI_Y2_FINAL/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.2s/it 9.6s
   

In [ ]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 5
#
# VALIDATION FAILURE ANALYSIS
#
# Uses:
#   /content/ai_y2.pt
#   /content/AI_Y2_FINAL/valid
#
# TEST SET IS NOT USED.
#
# Goal:
# - understand lesion misses
# - quantify mask IoU
# - quantify PA/lesion containment
# - identify worst source categories
# - visualize hardest cases
# ============================================================

from pathlib import Path
from collections import defaultdict, Counter
import math
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO


# ============================================================
# PATHS
# ============================================================

MODEL_PATH = Path("/content/ai_y2.pt")

DATASET_ROOT = Path("/content/AI_Y2_FINAL")

VAL_IMG_DIR = DATASET_ROOT / "valid" / "images"
VAL_LBL_DIR = DATASET_ROOT / "valid" / "labels"

MANIFEST_PATH = DATASET_ROOT / "dataset_manifest.csv"

REPORT_DIR = Path(
    "/content/AI_Y2_MODULE2/reports"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FAILURE_CSV = REPORT_DIR / "validation_failure_analysis.csv"
SUMMARY_JSON = REPORT_DIR / "validation_failure_summary.json"


# ============================================================
# CONFIG
# ============================================================

IMGSZ = 640
CONF = 0.25
IOU = 0.70

EXPECTED_NAMES = {
    0: "processing_area",
    1: "lesion"
}

LOW_IOU_THRESHOLD = 0.50
VERY_LOW_IOU_THRESHOLD = 0.25

SMALL_LESION_AREA_FRAC = 0.02

TOP_WORST = 15


# ============================================================
# CHECKS
# ============================================================

assert MODEL_PATH.exists(), f"Missing model: {MODEL_PATH}"
assert VAL_IMG_DIR.exists()
assert VAL_LBL_DIR.exists()
assert MANIFEST_PATH.exists()


# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO(str(MODEL_PATH))

names = {
    int(k): str(v)
    for k, v in model.names.items()
}

assert names == EXPECTED_NAMES, (
    f"Class contract mismatch: {names}"
)


# ============================================================
# LOAD MANIFEST
# ============================================================

manifest = pd.read_csv(MANIFEST_PATH)

manifest = manifest[
    manifest["final_split"] == "valid"
].copy()

manifest_by_file = {
    row["final_image"]: row
    for _, row in manifest.iterrows()
}


# ============================================================
# HELPERS
# ============================================================

def polygon_to_mask(points, w, h):
    """
    points are normalized Nx2
    """
    mask = np.zeros(
        (h, w),
        dtype=np.uint8
    )

    pts = points.copy()

    pts[:, 0] *= w
    pts[:, 1] *= h

    pts = np.round(
        pts
    ).astype(np.int32)

    cv2.fillPoly(
        mask,
        [pts],
        1
    )

    return mask


def parse_gt_label(label_path, w, h):

    gt = {
        0: [],
        1: []
    }

    text = label_path.read_text(
        encoding="utf-8"
    ).strip()

    for line in text.splitlines():

        parts = line.split()

        cls_id = int(parts[0])

        coords = np.array(
            [float(x) for x in parts[1:]],
            dtype=np.float32
        ).reshape(-1, 2)

        mask = polygon_to_mask(
            coords,
            w,
            h
        )

        gt[cls_id].append(mask)

    return gt


def union_masks(mask_list, h, w):

    if not mask_list:
        return np.zeros(
            (h, w),
            dtype=np.uint8
        )

    out = np.zeros(
        (h, w),
        dtype=np.uint8
    )

    for m in mask_list:
        out = np.maximum(
            out,
            m.astype(np.uint8)
        )

    return out


def resize_pred_mask(mask, w, h):

    if mask.shape[0] == h and mask.shape[1] == w:
        return (
            mask > 0.5
        ).astype(np.uint8)

    resized = cv2.resize(
        mask.astype(np.float32),
        (w, h),
        interpolation=cv2.INTER_NEAREST
    )

    return (
        resized > 0.5
    ).astype(np.uint8)


def mask_iou(a, b):

    inter = np.logical_and(
        a > 0,
        b > 0
    ).sum()

    union = np.logical_or(
        a > 0,
        b > 0
    ).sum()

    if union == 0:
        return 1.0

    return inter / union


def best_match_iou(
    gt_masks,
    pred_masks
):

    if not gt_masks:
        return []

    results = []

    for gt_mask in gt_masks:

        if not pred_masks:
            results.append(0.0)
            continue

        ious = [
            mask_iou(
                gt_mask,
                pred
            )
            for pred in pred_masks
        ]

        results.append(
            max(ious)
        )

    return results


def lesion_inside_processing_ratio(
    lesion_union,
    processing_union
):

    lesion_area = (
        lesion_union > 0
    ).sum()

    if lesion_area == 0:
        return None

    inside = np.logical_and(
        lesion_union > 0,
        processing_union > 0
    ).sum()

    return inside / lesion_area


def source_from_manifest(filename):

    row = manifest_by_file.get(
        filename
    )

    if row is None:
        return "unknown"

    return str(
        row["source_category"]
    )


# ============================================================
# VALIDATION IMAGES
# ============================================================

val_images = sorted(
    p for p in VAL_IMG_DIR.iterdir()
    if p.is_file()
)


# ============================================================
# RUN INFERENCE
# ============================================================

print("=" * 100)
print("AI_Y2 MODULE 2 — CELL 5")
print("VALIDATION FAILURE ANALYSIS")
print("=" * 100)

print(f"""
Model:
{MODEL_PATH}

Validation images:
{len(val_images)}

conf:
{CONF}

imgsz:
{IMGSZ}

TEST SET:
NOT USED
""")


results = model.predict(
    source=[
        str(p)
        for p in val_images
    ],
    imgsz=IMGSZ,
    conf=CONF,
    iou=IOU,
    device=0,
    batch=16,
    retina_masks=True,
    verbose=False,
    save=False
)


# ============================================================
# ANALYZE EACH IMAGE
# ============================================================

rows = []

for img_path, result in zip(
    val_images,
    results
):

    img = cv2.imread(
        str(img_path)
    )

    if img is None:
        raise RuntimeError(
            f"Unreadable image: {img_path}"
        )

    h, w = img.shape[:2]

    label_path = (
        VAL_LBL_DIR /
        f"{img_path.stem}.txt"
    )

    gt = parse_gt_label(
        label_path,
        w,
        h
    )

    gt_pa = gt[0]
    gt_lesions = gt[1]

    gt_pa_union = union_masks(
        gt_pa,
        h,
        w
    )

    gt_lesion_union = union_masks(
        gt_lesions,
        h,
        w
    )


    # --------------------------------------------
    # predictions
    # --------------------------------------------

    pred_pa = []
    pred_lesions = []

    pred_lesion_conf = []
    pred_pa_conf = []

    if (
        result.boxes is not None
        and
        len(result.boxes) > 0
        and
        result.masks is not None
    ):

        classes = (
            result.boxes.cls
            .detach()
            .cpu()
            .numpy()
            .astype(int)
        )

        confs = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
        )

        masks = (
            result.masks.data
            .detach()
            .cpu()
            .numpy()
        )

        for cls_id, conf, mask in zip(
            classes,
            confs,
            masks
        ):

            mask = resize_pred_mask(
                mask,
                w,
                h
            )

            if cls_id == 0:

                pred_pa.append(mask)
                pred_pa_conf.append(
                    float(conf)
                )

            elif cls_id == 1:

                pred_lesions.append(mask)
                pred_lesion_conf.append(
                    float(conf)
                )


    pred_pa_union = union_masks(
        pred_pa,
        h,
        w
    )

    pred_lesion_union = union_masks(
        pred_lesions,
        h,
        w
    )


    # --------------------------------------------
    # class metrics per image
    # --------------------------------------------

    pa_iou = mask_iou(
        gt_pa_union,
        pred_pa_union
    )

    lesion_union_iou = mask_iou(
        gt_lesion_union,
        pred_lesion_union
    )


    lesion_instance_ious = (
        best_match_iou(
            gt_lesions,
            pred_lesions
        )
    )


    if lesion_instance_ious:

        mean_lesion_instance_iou = float(
            np.mean(
                lesion_instance_ious
            )
        )

        min_lesion_instance_iou = float(
            np.min(
                lesion_instance_ious
            )
        )

    else:

        mean_lesion_instance_iou = None
        min_lesion_instance_iou = None


    # --------------------------------------------
    # lesion size
    # --------------------------------------------

    lesion_area_frac = (
        (gt_lesion_union > 0).sum()
        /
        float(w * h)
    )


    is_small_lesion = (
        len(gt_lesions) > 0
        and
        lesion_area_frac
        < SMALL_LESION_AREA_FRAC
    )


    # --------------------------------------------
    # containment
    # --------------------------------------------

    gt_inside_ratio = (
        lesion_inside_processing_ratio(
            gt_lesion_union,
            gt_pa_union
        )
    )

    pred_inside_ratio = (
        lesion_inside_processing_ratio(
            pred_lesion_union,
            pred_pa_union
        )
    )


    # --------------------------------------------
    # failure flags
    # --------------------------------------------

    gt_has_lesion = (
        len(gt_lesions) > 0
    )

    pred_has_lesion = (
        len(pred_lesions) > 0
    )

    missed_lesion = (
        gt_has_lesion
        and
        not pred_has_lesion
    )

    false_lesion = (
        not gt_has_lesion
        and
        pred_has_lesion
    )

    low_lesion_iou = (
        gt_has_lesion
        and
        lesion_union_iou
        < LOW_IOU_THRESHOLD
    )

    very_low_lesion_iou = (
        gt_has_lesion
        and
        lesion_union_iou
        < VERY_LOW_IOU_THRESHOLD
    )

    poor_processing_iou = (
        pa_iou < 0.50
    )

    lesion_outside_processing = (
        pred_inside_ratio is not None
        and
        pred_inside_ratio < 0.90
    )


    rows.append({

        "filename":
            img_path.name,

        "source":
            source_from_manifest(
                img_path.name
            ),

        "gt_pa_instances":
            len(gt_pa),

        "gt_lesion_instances":
            len(gt_lesions),

        "pred_pa_instances":
            len(pred_pa),

        "pred_lesion_instances":
            len(pred_lesions),

        "gt_has_lesion":
            gt_has_lesion,

        "pred_has_lesion":
            pred_has_lesion,

        "missed_lesion":
            missed_lesion,

        "false_lesion":
            false_lesion,

        "pa_iou":
            pa_iou,

        "lesion_union_iou":
            lesion_union_iou,

        "mean_lesion_instance_iou":
            mean_lesion_instance_iou,

        "min_lesion_instance_iou":
            min_lesion_instance_iou,

        "lesion_area_frac":
            lesion_area_frac,

        "is_small_lesion":
            is_small_lesion,

        "low_lesion_iou":
            low_lesion_iou,

        "very_low_lesion_iou":
            very_low_lesion_iou,

        "poor_processing_iou":
            poor_processing_iou,

        "gt_lesion_inside_pa":
            gt_inside_ratio,

        "pred_lesion_inside_pa":
            pred_inside_ratio,

        "lesion_outside_processing":
            lesion_outside_processing,

        "mean_pred_lesion_conf":
            (
                float(
                    np.mean(
                        pred_lesion_conf
                    )
                )
                if pred_lesion_conf
                else None
            ),

        "mean_pred_pa_conf":
            (
                float(
                    np.mean(
                        pred_pa_conf
                    )
                )
                if pred_pa_conf
                else None
            ),
    })


df = pd.DataFrame(rows)


# ============================================================
# SAVE FULL FAILURE TABLE
# ============================================================

df.to_csv(
    FAILURE_CSV,
    index=False
)


# ============================================================
# OVERALL SUMMARY
# ============================================================

lesion_df = df[
    df["gt_has_lesion"] == True
].copy()

processing_only_df = df[
    df["gt_has_lesion"] == False
].copy()


summary = {

    "validation_images":
        int(len(df)),

    "lesion_positive_images":
        int(len(lesion_df)),

    "processing_only_images":
        int(len(processing_only_df)),

    "missed_lesion_images":
        int(
            df["missed_lesion"].sum()
        ),

    "false_lesion_images":
        int(
            df["false_lesion"].sum()
        ),

    "low_lesion_iou_images":
        int(
            df["low_lesion_iou"].sum()
        ),

    "very_low_lesion_iou_images":
        int(
            df["very_low_lesion_iou"].sum()
        ),

    "small_lesion_images":
        int(
            df["is_small_lesion"].sum()
        ),

    "poor_processing_iou_images":
        int(
            df["poor_processing_iou"].sum()
        ),

    "lesion_outside_processing_images":
        int(
            df[
                "lesion_outside_processing"
            ].sum()
        ),

    "mean_processing_iou":
        float(
            df["pa_iou"].mean()
        ),

    "mean_lesion_union_iou":
        float(
            lesion_df[
                "lesion_union_iou"
            ].mean()
        ),

    "median_lesion_union_iou":
        float(
            lesion_df[
                "lesion_union_iou"
            ].median()
        ),
}


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print("\n" + "=" * 100)
print("OVERALL FAILURE SUMMARY")
print("=" * 100)

for k, v in summary.items():
    print(
        f"{k:<38}: {v}"
    )


# ============================================================
# SOURCE-WISE FAILURE SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("SOURCE-WISE LESION PERFORMANCE")
print("=" * 100)


source_rows = []

for source in sorted(
    lesion_df["source"].unique()
):

    s = lesion_df[
        lesion_df["source"] == source
    ]

    source_rows.append({

        "source":
            source,

        "lesion_images":
            len(s),

        "missed":
            int(
                s["missed_lesion"].sum()
            ),

        "miss_rate":
            (
                s["missed_lesion"].mean()
                if len(s)
                else 0
            ),

        "mean_union_iou":
            s[
                "lesion_union_iou"
            ].mean(),

        "median_union_iou":
            s[
                "lesion_union_iou"
            ].median(),

        "low_iou_count":
            int(
                s[
                    "low_lesion_iou"
                ].sum()
            ),

        "very_low_iou_count":
            int(
                s[
                    "very_low_lesion_iou"
                ].sum()
            ),

        "small_lesions":
            int(
                s[
                    "is_small_lesion"
                ].sum()
            ),
    })


source_df = pd.DataFrame(
    source_rows
)

print(
    source_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# SMALL VS NON-SMALL LESION
# ============================================================

print("\n" + "=" * 100)
print("SMALL LESION ANALYSIS")
print("=" * 100)


for flag, name in [
    (True, "SMALL"),
    (False, "NON-SMALL"),
]:

    s = lesion_df[
        lesion_df["is_small_lesion"] == flag
    ]

    if len(s) == 0:
        continue

    print(
        f"{name:<10} "
        f"n={len(s):3d} | "
        f"miss={int(s['missed_lesion'].sum()):3d} | "
        f"mean IoU={s['lesion_union_iou'].mean():.4f} | "
        f"median IoU={s['lesion_union_iou'].median():.4f}"
    )


# ============================================================
# WORST LESION CASES
# ============================================================

worst_lesion = (
    lesion_df
    .sort_values(
        [
            "lesion_union_iou",
            "mean_pred_lesion_conf"
        ],
        ascending=[
            True,
            True
        ]
    )
    .head(
        TOP_WORST
    )
)


print("\n" + "=" * 100)
print("TOP WORST LESION CASES")
print("=" * 100)


cols = [
    "filename",
    "source",
    "gt_lesion_instances",
    "pred_lesion_instances",
    "lesion_union_iou",
    "missed_lesion",
    "is_small_lesion",
    "pred_lesion_inside_pa",
]


print(
    worst_lesion[
        cols
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# VISUAL WORST CASE OVERLAYS
# ============================================================

def draw_gt_pred_overlay(
    image_path,
    label_path,
    prediction_result
):

    img = cv2.imread(
        str(image_path)
    )

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    h, w = img.shape[:2]

    canvas = img.copy()

    # GT:
    # PA = green
    # lesion = red

    gt = parse_gt_label(
        label_path,
        w,
        h
    )

    for cls_id, masks in gt.items():

        color = (
            (0, 255, 0)
            if cls_id == 0
            else
            (255, 0, 0)
        )

        for mask in masks:

            contours, _ = cv2.findContours(
                mask.astype(np.uint8),
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )

            cv2.drawContours(
                canvas,
                contours,
                -1,
                color,
                2
            )


    # Predictions:
    # PA = cyan
    # lesion = yellow

    if (
        prediction_result.boxes is not None
        and
        prediction_result.masks is not None
    ):

        classes = (
            prediction_result.boxes.cls
            .detach()
            .cpu()
            .numpy()
            .astype(int)
        )

        masks = (
            prediction_result.masks.data
            .detach()
            .cpu()
            .numpy()
        )

        for cls_id, mask in zip(
            classes,
            masks
        ):

            mask = resize_pred_mask(
                mask,
                w,
                h
            )

            color = (
                (0, 255, 255)
                if cls_id == 0
                else
                (255, 255, 0)
            )

            contours, _ = cv2.findContours(
                mask.astype(np.uint8),
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )

            cv2.drawContours(
                canvas,
                contours,
                -1,
                color,
                2
            )

    return canvas


# prediction lookup
prediction_map = {
    img.name: result
    for img, result
    in zip(
        val_images,
        results
    )
}


visual_cases = (
    worst_lesion
    .head(12)
)


fig, axes = plt.subplots(
    3,
    4,
    figsize=(20, 15)
)

axes = axes.flatten()


for ax, (_, row) in zip(
    axes,
    visual_cases.iterrows()
):

    filename = row[
        "filename"
    ]

    img_path = (
        VAL_IMG_DIR /
        filename
    )

    lbl_path = (
        VAL_LBL_DIR /
        f"{Path(filename).stem}.txt"
    )

    overlay = draw_gt_pred_overlay(
        img_path,
        lbl_path,
        prediction_map[
            filename
        ]
    )

    ax.imshow(
        overlay
    )

    ax.set_title(
        f"{row['source']}\n"
        f"IoU={row['lesion_union_iou']:.3f} | "
        f"miss={row['missed_lesion']}"
    )

    ax.axis("off")


plt.suptitle(
    "AI_Y2 Worst Validation Cases\n"
    "GT: green PA / red lesion | "
    "Prediction: cyan PA / yellow lesion",
    fontsize=15
)

plt.tight_layout()
plt.show()


# ============================================================
# FINAL DIAGNOSTIC SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("CELL 5 DIAGNOSTIC OUTPUT")
print("=" * 100)

print(f"""
Full per-image failure table:
{FAILURE_CSV}

Summary:
{SUMMARY_JSON}

Interpretation guide:

missed_lesion = True
→ detector failed to emit lesion at conf=0.25

lesion_union_iou < 0.50
→ lesion localization/boundary is weak

is_small_lesion = True
→ lesion occupies <2% of image

poor_processing_iou = True
→ processing-area segmentation itself is weak

lesion_outside_processing = True
→ predicted lesion is not sufficiently contained
  inside predicted processing area

Use this output to decide next training strategy.
""")

print("=" * 100)
print("✅ MODULE 2 CELL 5 COMPLETE")
print("=" * 100)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 6
#
# GT GEOMETRY + HARD-CASE AUDIT
#
# Uses validation only.
# TEST SET NOT USED.
#
# Purpose:
# 1. Verify whether lesion is actually expected
#    inside processing_area in ground truth.
# 2. Separate detection failures from boundary failures.
# 3. Build evidence-based hard-example list.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


# ============================================================
# PATHS
# ============================================================

REPORT_DIR = Path(
    "/content/AI_Y2_MODULE2/reports"
)

FAILURE_CSV = (
    REPORT_DIR /
    "validation_failure_analysis.csv"
)

HARD_CASES_CSV = (
    REPORT_DIR /
    "validation_hard_cases.csv"
)

CELL6_JSON = (
    REPORT_DIR /
    "cell6_geometry_summary.json"
)


assert FAILURE_CSV.exists(), (
    f"❌ Missing Cell 5 output:\n{FAILURE_CSV}"
)


# ============================================================
# LOAD CELL 5 ANALYSIS
# ============================================================

df = pd.read_csv(
    FAILURE_CSV
)


# ============================================================
# HELPERS
# ============================================================

def pct(n, d):
    return (
        100.0 * n / d
        if d > 0
        else 0.0
    )


# ============================================================
# LESION-POSITIVE ONLY
# ============================================================

lesion_df = df[
    df["gt_has_lesion"] == True
].copy()


processing_only_df = df[
    df["gt_has_lesion"] == False
].copy()


print("=" * 100)
print("AI_Y2 MODULE 2 — CELL 6")
print("GROUND-TRUTH GEOMETRY + HARD-CASE AUDIT")
print("=" * 100)

print(f"""
Validation total       : {len(df)}
Lesion-positive        : {len(lesion_df)}
Processing-only        : {len(processing_only_df)}

TEST SET:
NOT USED
""")


# ============================================================
# 1. GROUND-TRUTH CONTAINMENT
# ============================================================

valid_gt_containment = lesion_df[
    lesion_df[
        "gt_lesion_inside_pa"
    ].notna()
].copy()


print("\n" + "=" * 100)
print("GROUND-TRUTH LESION INSIDE PROCESSING-AREA")
print("=" * 100)


thresholds = [
    0.50,
    0.75,
    0.90,
    0.95,
]


gt_containment_summary = {}


for t in thresholds:

    n = int(
        (
            valid_gt_containment[
                "gt_lesion_inside_pa"
            ] >= t
        ).sum()
    )

    gt_containment_summary[
        str(t)
    ] = n

    print(
        f"GT containment >= {t:.2f}: "
        f"{n:3d} / "
        f"{len(valid_gt_containment):3d} "
        f"({pct(n, len(valid_gt_containment)):.2f}%)"
    )


print(
    "\nGT containment stats:"
)

print(
    f"Mean   : "
    f"{valid_gt_containment['gt_lesion_inside_pa'].mean():.4f}"
)

print(
    f"Median : "
    f"{valid_gt_containment['gt_lesion_inside_pa'].median():.4f}"
)

print(
    f"Min    : "
    f"{valid_gt_containment['gt_lesion_inside_pa'].min():.4f}"
)

print(
    f"Max    : "
    f"{valid_gt_containment['gt_lesion_inside_pa'].max():.4f}"
)


# ============================================================
# 2. PREDICTED CONTAINMENT
# ============================================================

pred_containment_df = lesion_df[
    lesion_df[
        "pred_lesion_inside_pa"
    ].notna()
].copy()


print("\n" + "=" * 100)
print("PREDICTED LESION INSIDE PREDICTED PROCESSING-AREA")
print("=" * 100)


for t in thresholds:

    n = int(
        (
            pred_containment_df[
                "pred_lesion_inside_pa"
            ] >= t
        ).sum()
    )

    print(
        f"Pred containment >= {t:.2f}: "
        f"{n:3d} / "
        f"{len(pred_containment_df):3d} "
        f"({pct(n, len(pred_containment_df)):.2f}%)"
    )


print(
    "\nPrediction containment stats:"
)

print(
    f"Mean   : "
    f"{pred_containment_df['pred_lesion_inside_pa'].mean():.4f}"
)

print(
    f"Median : "
    f"{pred_containment_df['pred_lesion_inside_pa'].median():.4f}"
)


# ============================================================
# 3. GT VS PRED CONTAINMENT GAP
# ============================================================

paired = lesion_df[
    lesion_df[
        "gt_lesion_inside_pa"
    ].notna()
    &
    lesion_df[
        "pred_lesion_inside_pa"
    ].notna()
].copy()


paired[
    "containment_gap"
] = (
    paired[
        "pred_lesion_inside_pa"
    ]
    -
    paired[
        "gt_lesion_inside_pa"
    ]
)


print("\n" + "=" * 100)
print("GT vs PRED CONTAINMENT GAP")
print("=" * 100)

print(
    f"Paired lesion images : {len(paired)}"
)

print(
    f"Mean GT containment  : "
    f"{paired['gt_lesion_inside_pa'].mean():.4f}"
)

print(
    f"Mean Pred containment: "
    f"{paired['pred_lesion_inside_pa'].mean():.4f}"
)

print(
    f"Mean gap             : "
    f"{paired['containment_gap'].mean():.4f}"
)


# ============================================================
# 4. DETECTION VS SEGMENTATION FAILURE TYPES
# ============================================================

def classify_failure(row):

    if row["missed_lesion"]:
        return "MISS"

    if row["lesion_union_iou"] < 0.25:
        return "SEVERE_BOUNDARY"

    if row["lesion_union_iou"] < 0.50:
        return "LOW_IOU_BOUNDARY"

    if row["lesion_union_iou"] < 0.70:
        return "MODERATE_BOUNDARY"

    return "GOOD"


lesion_df[
    "failure_type"
] = lesion_df.apply(
    classify_failure,
    axis=1
)


print("\n" + "=" * 100)
print("LESION FAILURE TYPE DISTRIBUTION")
print("=" * 100)


failure_counts = (
    lesion_df[
        "failure_type"
    ]
    .value_counts()
)


for name in [
    "MISS",
    "SEVERE_BOUNDARY",
    "LOW_IOU_BOUNDARY",
    "MODERATE_BOUNDARY",
    "GOOD",
]:

    n = int(
        failure_counts.get(
            name,
            0
        )
    )

    print(
        f"{name:<20}: "
        f"{n:3d} "
        f"({pct(n, len(lesion_df)):.2f}%)"
    )


# ============================================================
# 5. SOURCE × FAILURE TYPE
# ============================================================

print("\n" + "=" * 100)
print("SOURCE × FAILURE TYPE")
print("=" * 100)


source_failure = pd.crosstab(
    lesion_df["source"],
    lesion_df["failure_type"]
)


for col in [
    "MISS",
    "SEVERE_BOUNDARY",
    "LOW_IOU_BOUNDARY",
    "MODERATE_BOUNDARY",
    "GOOD",
]:

    if col not in source_failure.columns:
        source_failure[col] = 0


source_failure = source_failure[
    [
        "MISS",
        "SEVERE_BOUNDARY",
        "LOW_IOU_BOUNDARY",
        "MODERATE_BOUNDARY",
        "GOOD",
    ]
]


print(
    source_failure.to_string()
)


# ============================================================
# 6. PROCESSING AREA QUALITY BY SOURCE
# ============================================================

print("\n" + "=" * 100)
print("PROCESSING-AREA IoU BY SOURCE")
print("=" * 100)


pa_source = (
    df.groupby(
        "source"
    )["pa_iou"]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max"
        ]
    )
    .reset_index()
)


print(
    pa_source.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 7. FALSE LESIONS ON PROCESSING-ONLY
# ============================================================

print("\n" + "=" * 100)
print("FALSE LESIONS ON PROCESSING-ONLY IMAGES")
print("=" * 100)


false_df = processing_only_df[
    processing_only_df[
        "false_lesion"
    ] == True
].copy()


print(
    f"False-lesion images: "
    f"{len(false_df)} / "
    f"{len(processing_only_df)}"
)


if len(false_df):

    print(
        false_df[
            [
                "filename",
                "source",
                "pred_lesion_instances",
                "mean_pred_lesion_conf",
                "pa_iou",
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )


# ============================================================
# 8. BUILD HARD-CASE TABLE
#
# Priority:
# misses > severe IoU > low IoU > poor PA
# ============================================================

def hard_score(row):

    score = 0

    if row["missed_lesion"]:
        score += 100

    if row["gt_has_lesion"]:

        iou = (
            row["lesion_union_iou"]
            if pd.notna(
                row["lesion_union_iou"]
            )
            else 0
        )

        score += (
            50 * (1.0 - iou)
        )

    if row["false_lesion"]:
        score += 45

    if row["poor_processing_iou"]:
        score += (
            25
            * (
                1.0
                -
                row["pa_iou"]
            )
        )

    if row["is_small_lesion"]:
        score += 5

    return score


df[
    "hard_score"
] = df.apply(
    hard_score,
    axis=1
)


hard_df = (
    df.sort_values(
        "hard_score",
        ascending=False
    )
    .copy()
)


hard_df.to_csv(
    HARD_CASES_CSV,
    index=False
)


print("\n" + "=" * 100)
print("TOP 25 HARD CASES")
print("=" * 100)


show_cols = [
    "filename",
    "source",
    "gt_has_lesion",
    "missed_lesion",
    "false_lesion",
    "lesion_union_iou",
    "pa_iou",
    "is_small_lesion",
    "hard_score",
]


print(
    hard_df[
        show_cols
    ]
    .head(25)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 9. SUMMARY DECISION VARIABLES
# ============================================================

gt90_rate = (
    (
        valid_gt_containment[
            "gt_lesion_inside_pa"
        ] >= 0.90
    ).mean()
    if len(valid_gt_containment)
    else 0
)


summary = {

    "validation_images":
        int(len(df)),

    "lesion_images":
        int(len(lesion_df)),

    "gt_containment_mean":
        float(
            valid_gt_containment[
                "gt_lesion_inside_pa"
            ].mean()
        ),

    "gt_containment_median":
        float(
            valid_gt_containment[
                "gt_lesion_inside_pa"
            ].median()
        ),

    "gt_containment_ge_090_rate":
        float(gt90_rate),

    "pred_containment_mean":
        float(
            pred_containment_df[
                "pred_lesion_inside_pa"
            ].mean()
        )
        if len(pred_containment_df)
        else None,

    "failure_types":
        {
            str(k): int(v)
            for k, v
            in failure_counts.items()
        },

    "hard_cases_csv":
        str(HARD_CASES_CSV),
}


with open(
    CELL6_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ============================================================
# DECISION HINT
# ============================================================

print("\n" + "=" * 100)
print("CELL 6 DECISION HINT")
print("=" * 100)


if gt90_rate >= 0.80:

    print("""
GROUND-TRUTH indicates lesion is usually expected
inside processing_area.

→ Processing/lesion consistency can be used safely
  as a future training or post-processing constraint.
""")

else:

    print("""
GROUND-TRUTH does NOT show strong enough evidence that
lesion must always be inside processing_area.

→ Do NOT enforce lesion-inside-processing constraint yet.
  That could suppress valid lesions.
""")


print(f"""
Hard-case table:
{HARD_CASES_CSV}

Summary:
{CELL6_JSON}

✅ TEST SET NOT USED
""")


print("=" * 100)
print("✅ MODULE 2 CELL 6 COMPLETE")
print("=" * 100)

AI_Y2 MODULE 2 — CELL 6
GROUND-TRUTH GEOMETRY + HARD-CASE AUDIT

Validation total       : 173
Lesion-positive        : 133
Processing-only        : 40

TEST SET:
NOT USED


GROUND-TRUTH LESION INSIDE PROCESSING-AREA
GT containment >= 0.50: 133 / 133 (100.00%)
GT containment >= 0.75: 133 / 133 (100.00%)
GT containment >= 0.90: 133 / 133 (100.00%)
GT containment >= 0.95: 132 / 133 (99.25%)

GT containment stats:
Mean   : 0.9934
Median : 0.9967
Min    : 0.9472
Max    : 1.0000

PREDICTED LESION INSIDE PREDICTED PROCESSING-AREA
Pred containment >= 0.50:   3 / 125 (2.40%)
Pred containment >= 0.75:   0 / 125 (0.00%)
Pred containment >= 0.90:   0 / 125 (0.00%)
Pred containment >= 0.95:   0 / 125 (0.00%)

Prediction containment stats:
Mean   : 0.0744
Median : 0.0313

GT vs PRED CONTAINMENT GAP
Paired lesion images : 125
Mean GT containment  : 0.9931
Mean Pred containment: 0.0744
Mean gap             : -0.9187

LESION FAILURE TYPE DISTRIBUTION
MISS                :   8 (6.02%)
SEVERE_BOUNDARY   

In [ ]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 7 FINAL
#
# STANDALONE AFTER COLAB "RESTART SESSION"
#
# TRAIN-ONLY HARD EXAMPLE MINING
# + HARD-MINED DATASET BUILD
#
# Validation/Test are NOT used for mining.
#
# Classes remain:
# 0 = processing_area
# 1 = lesion
# ============================================================

import gc
import json
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch

from ultralytics import YOLO


# ============================================================
# GPU STATUS
# ============================================================

print("=" * 100)
print("AI_Y2 CELL 7 — FRESH-RUNTIME GPU CHECK")
print("=" * 100)

if not torch.cuda.is_available():
    raise RuntimeError("❌ CUDA GPU unavailable.")

print(
    f"GPU       : {torch.cuda.get_device_name(0)}"
)

print(
    f"Allocated : "
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    f"Reserved  : "
    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
)


# Fresh runtime should normally be very low.
if torch.cuda.memory_allocated() / 1024**3 > 5:
    raise RuntimeError(
        "\n❌ GPU still has >5 GB allocated.\n"
        "Colab Runtime → Restart session karo, "
        "then ONLY this cell run karo."
    )


# ============================================================
# PATHS
# ============================================================

MODEL_PATH = Path(
    "/content/ai_y2.pt"
)

ORIGINAL_ROOT = Path(
    "/content/AI_Y2_FINAL"
)

TRAIN_IMG_DIR = (
    ORIGINAL_ROOT /
    "train" /
    "images"
)

TRAIN_LBL_DIR = (
    ORIGINAL_ROOT /
    "train" /
    "labels"
)

MANIFEST_PATH = (
    ORIGINAL_ROOT /
    "dataset_manifest.csv"
)

BOOST_ROOT = Path(
    "/content/AI_Y2_HARDMINED"
)

REPORT_DIR = Path(
    "/content/AI_Y2_MODULE2/reports"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HARD_REPORT = (
    REPORT_DIR /
    "train_hard_mining.csv"
)

SUMMARY_JSON = (
    REPORT_DIR /
    "train_hard_mining_summary.json"
)


# ============================================================
# VERIFY FILES SURVIVED RESTART
# ============================================================

for p in [
    MODEL_PATH,
    ORIGINAL_ROOT,
    TRAIN_IMG_DIR,
    TRAIN_LBL_DIR,
    MANIFEST_PATH,
]:
    if not p.exists():

        raise FileNotFoundError(
            f"\n❌ Missing after restart:\n{p}\n"
            "Do NOT continue until this file/folder exists."
        )


# ============================================================
# CONFIG
# ============================================================

IMGSZ = 640

CONF = 0.25
IOU = 0.70

# Ultra-safe inference
PRED_BATCH = 1

EXPECTED_NAMES = {
    0: "processing_area",
    1: "lesion",
}

LESION_IOU_HARD = 0.50
PA_IOU_HARD = 0.50
CONTAINMENT_HARD = 0.50

BOOST_COPIES = 1


# ============================================================
# HELPERS
# ============================================================

def polygon_to_mask(points, w, h):

    mask = np.zeros(
        (h, w),
        dtype=np.uint8
    )

    pts = points.copy()

    pts[:, 0] *= w
    pts[:, 1] *= h

    pts = np.round(
        pts
    ).astype(np.int32)

    cv2.fillPoly(
        mask,
        [pts],
        1
    )

    return mask


def parse_gt(label_path, w, h):

    out = {
        0: [],
        1: [],
    }

    text = label_path.read_text(
        encoding="utf-8"
    ).strip()

    for line in text.splitlines():

        parts = line.split()

        cls_id = int(
            parts[0]
        )

        coords = np.array(
            [
                float(v)
                for v in parts[1:]
            ],
            dtype=np.float32
        ).reshape(-1, 2)

        out[cls_id].append(
            polygon_to_mask(
                coords,
                w,
                h
            )
        )

    return out


def union_masks(masks, h, w):

    output = np.zeros(
        (h, w),
        dtype=np.uint8
    )

    for mask in masks:

        output = np.maximum(
            output,
            mask.astype(np.uint8)
        )

    return output


def resize_mask(mask, w, h):

    mask = cv2.resize(
        mask.astype(np.float32),
        (w, h),
        interpolation=cv2.INTER_NEAREST
    )

    return (
        mask > 0.5
    ).astype(np.uint8)


def mask_iou(mask_a, mask_b):

    intersection = np.logical_and(
        mask_a > 0,
        mask_b > 0
    ).sum()

    union = np.logical_or(
        mask_a > 0,
        mask_b > 0
    ).sum()

    if union == 0:
        return 1.0

    return float(
        intersection / union
    )


def containment_ratio(
    lesion,
    processing
):

    lesion_area = (
        lesion > 0
    ).sum()

    if lesion_area == 0:
        return None

    inside = np.logical_and(
        lesion > 0,
        processing > 0
    ).sum()

    return float(
        inside / lesion_area
    )


# ============================================================
# LOAD MANIFEST
# ============================================================

manifest = pd.read_csv(
    MANIFEST_PATH
)

train_manifest = manifest[
    manifest["final_split"] == "train"
].copy()

source_lookup = {
    row["final_image"]:
        row["source_category"]
    for _, row
    in train_manifest.iterrows()
}


# ============================================================
# IMAGE LIST
# ============================================================

train_images = sorted([
    p
    for p in TRAIN_IMG_DIR.iterdir()
    if p.is_file()
])


assert len(train_images) == 865, (
    f"❌ Expected 865 train images, "
    f"found {len(train_images)}"
)


# ============================================================
# LOAD MODEL
# ============================================================

print("\n" + "=" * 100)
print("LOADING AI_Y2")
print("=" * 100)

model = YOLO(
    str(MODEL_PATH)
)

model_names = {
    int(k): str(v)
    for k, v in model.names.items()
}

assert model_names == EXPECTED_NAMES

assert str(model.task) == "segment"


print(
    f"Classes: {model_names}"
)

print(
    f"Task   : {model.task}"
)


# ============================================================
# STREAMING INFERENCE
# ============================================================

print("\n" + "=" * 100)
print("TRAIN-ONLY HARD MINING")
print("=" * 100)

print(f"""
Train images           : {len(train_images)}

batch                  : {PRED_BATCH}
stream                 : True
half                   : True
retina_masks           : False

Validation used        : NO
Test used              : NO
""")


prediction_stream = model.predict(

    source=[
        str(p)
        for p in train_images
    ],

    imgsz=IMGSZ,

    conf=CONF,
    iou=IOU,

    device=0,

    batch=PRED_BATCH,

    # FP16 dramatically reduces inference memory
    half=True,

    # Important:
    # full-resolution retina masks are unnecessary here
    retina_masks=False,

    stream=True,

    verbose=False,

    save=False
)


# ============================================================
# PROCESS ONE RESULT AT A TIME
# ============================================================

rows = []


for idx, (img_path, result) in enumerate(
    zip(
        train_images,
        prediction_stream
    ),
    start=1
):

    image = cv2.imread(
        str(img_path)
    )

    if image is None:

        raise RuntimeError(
            f"Unreadable image:\n{img_path}"
        )

    h, w = image.shape[:2]


    label_path = (
        TRAIN_LBL_DIR /
        f"{img_path.stem}.txt"
    )


    gt = parse_gt(
        label_path,
        w,
        h
    )


    gt_processing = union_masks(
        gt[0],
        h,
        w
    )

    gt_lesion = union_masks(
        gt[1],
        h,
        w
    )


    pred_processing_masks = []
    pred_lesion_masks = []


    # --------------------------------------------------------
    # MOVE OUTPUTS TO CPU IMMEDIATELY
    # --------------------------------------------------------

    if (
        result.boxes is not None
        and
        len(result.boxes) > 0
        and
        result.masks is not None
    ):

        classes = (
            result.boxes.cls
            .detach()
            .cpu()
            .numpy()
            .astype(np.int32)
        )

        raw_masks = (
            result.masks.data
            .detach()
            .float()
            .cpu()
            .numpy()
        )


        for cls_id, raw_mask in zip(
            classes,
            raw_masks
        ):

            mask = resize_mask(
                raw_mask,
                w,
                h
            )

            if cls_id == 0:

                pred_processing_masks.append(
                    mask
                )

            elif cls_id == 1:

                pred_lesion_masks.append(
                    mask
                )


        del classes
        del raw_masks


    pred_processing = union_masks(
        pred_processing_masks,
        h,
        w
    )

    pred_lesion = union_masks(
        pred_lesion_masks,
        h,
        w
    )


    # ========================================================
    # METRICS
    # ========================================================

    gt_has_lesion = (
        len(gt[1]) > 0
    )

    pred_has_lesion = (
        len(pred_lesion_masks) > 0
    )


    pa_iou = mask_iou(
        gt_processing,
        pred_processing
    )


    lesion_iou = (
        mask_iou(
            gt_lesion,
            pred_lesion
        )
        if gt_has_lesion
        else None
    )


    pred_containment = (
        containment_ratio(
            pred_lesion,
            pred_processing
        )
        if pred_has_lesion
        else None
    )


    missed_lesion = (
        gt_has_lesion
        and
        not pred_has_lesion
    )


    false_lesion = (
        not gt_has_lesion
        and
        pred_has_lesion
    )


    low_lesion_iou = (
        gt_has_lesion
        and
        lesion_iou < LESION_IOU_HARD
    )


    low_pa_iou = (
        pa_iou < PA_IOU_HARD
    )


    poor_containment = (
        pred_has_lesion
        and
        pred_containment is not None
        and
        pred_containment
        < CONTAINMENT_HARD
    )


    hard = (
        missed_lesion
        or
        false_lesion
        or
        low_lesion_iou
        or
        low_pa_iou
        or
        poor_containment
    )


    # ========================================================
    # HARDNESS SCORE
    # ========================================================

    hard_score = 0.0


    if missed_lesion:
        hard_score += 100.0


    if false_lesion:
        hard_score += 75.0


    if gt_has_lesion:

        hard_score += (
            50.0
            * (
                1.0
                -
                lesion_iou
            )
        )


    hard_score += (
        30.0
        * (
            1.0
            -
            pa_iou
        )
    )


    if poor_containment:

        hard_score += (
            25.0
            * (
                1.0
                -
                pred_containment
            )
        )


    rows.append({

        "filename":
            img_path.name,

        "source":
            source_lookup.get(
                img_path.name,
                "unknown"
            ),

        "gt_has_lesion":
            gt_has_lesion,

        "missed_lesion":
            missed_lesion,

        "false_lesion":
            false_lesion,

        "lesion_iou":
            lesion_iou,

        "pa_iou":
            pa_iou,

        "pred_containment":
            pred_containment,

        "low_lesion_iou":
            low_lesion_iou,

        "low_pa_iou":
            low_pa_iou,

        "poor_containment":
            poor_containment,

        "hard":
            hard,

        "hard_score":
            hard_score,
    })


    # ========================================================
    # RELEASE EVERYTHING FOR THIS IMAGE
    # ========================================================

    del result
    del image

    del gt

    del gt_processing
    del gt_lesion

    del pred_processing
    del pred_lesion

    del pred_processing_masks
    del pred_lesion_masks


    if idx % 50 == 0:

        gc.collect()

        torch.cuda.empty_cache()

        print(
            f"Processed "
            f"{idx}/{len(train_images)}"
        )


# ============================================================
# END INFERENCE / FREE MODEL
# ============================================================

del prediction_stream
del model

gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()


print(
    f"\nGPU allocated after mining: "
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)


# ============================================================
# HARD-MINING REPORT
# ============================================================

hard_df = pd.DataFrame(
    rows
)


hard_df.to_csv(
    HARD_REPORT,
    index=False
)


selected = hard_df[
    hard_df["hard"] == True
].copy()


print("\n" + "=" * 100)
print("TRAIN HARD-MINING SUMMARY")
print("=" * 100)


print(
    f"Train images            : {len(hard_df)}"
)

print(
    f"Hard images selected    : {len(selected)}"
)

print(
    f"Hard fraction           : "
    f"{100*len(selected)/len(hard_df):.2f}%"
)

print(
    f"Missed lesions          : "
    f"{int(hard_df['missed_lesion'].sum())}"
)

print(
    f"False lesion images     : "
    f"{int(hard_df['false_lesion'].sum())}"
)

print(
    f"Low lesion IoU          : "
    f"{int(hard_df['low_lesion_iou'].sum())}"
)

print(
    f"Low processing IoU      : "
    f"{int(hard_df['low_pa_iou'].sum())}"
)

print(
    f"Poor containment        : "
    f"{int(hard_df['poor_containment'].sum())}"
)


# ============================================================
# SOURCE SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("HARD CASES BY SOURCE")
print("=" * 100)


source_summary = (
    selected[
        "source"
    ]
    .value_counts()
)


print(
    source_summary.to_string()
)


# ============================================================
# BUILD BOOSTED TRAINING DATASET
# ============================================================

print("\n" + "=" * 100)
print("BUILDING HARD-MINED DATASET")
print("=" * 100)


if BOOST_ROOT.exists():

    shutil.rmtree(
        BOOST_ROOT
    )


for split in [
    "train",
    "valid",
    "test"
]:

    (
        BOOST_ROOT /
        split /
        "images"
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    (
        BOOST_ROOT /
        split /
        "labels"
    ).mkdir(
        parents=True,
        exist_ok=True
    )


# Original dataset copied exactly
for split in [
    "train",
    "valid",
    "test"
]:

    src_img_dir = (
        ORIGINAL_ROOT /
        split /
        "images"
    )

    src_lbl_dir = (
        ORIGINAL_ROOT /
        split /
        "labels"
    )

    dst_img_dir = (
        BOOST_ROOT /
        split /
        "images"
    )

    dst_lbl_dir = (
        BOOST_ROOT /
        split /
        "labels"
    )


    for p in src_img_dir.iterdir():

        if p.is_file():

            shutil.copy2(
                p,
                dst_img_dir / p.name
            )


    for p in src_lbl_dir.iterdir():

        if p.is_file():

            shutil.copy2(
                p,
                dst_lbl_dir / p.name
            )


# ============================================================
# ADD ONE EXTRA COPY OF EACH HARD TRAIN IMAGE
# ============================================================

boost_count = 0


for _, row in selected.iterrows():

    filename = row[
        "filename"
    ]

    src_img = (
        TRAIN_IMG_DIR /
        filename
    )

    src_lbl = (
        TRAIN_LBL_DIR /
        f"{Path(filename).stem}.txt"
    )


    suffix = "__hardboost1"


    new_image_name = (
        f"{Path(filename).stem}"
        f"{suffix}"
        f"{Path(filename).suffix}"
    )

    new_label_name = (
        f"{Path(filename).stem}"
        f"{suffix}.txt"
    )


    shutil.copy2(
        src_img,
        BOOST_ROOT /
        "train" /
        "images" /
        new_image_name
    )


    shutil.copy2(
        src_lbl,
        BOOST_ROOT /
        "train" /
        "labels" /
        new_label_name
    )


    boost_count += 1


# ============================================================
# DATA.YAML
# ============================================================

data_yaml = f"""path: {BOOST_ROOT}
train: train/images
val: valid/images
test: test/images
nc: 2
names:
  - processing_area
  - lesion
"""


(
    BOOST_ROOT /
    "data.yaml"
).write_text(
    data_yaml,
    encoding="utf-8"
)


# ============================================================
# VERIFY FINAL DATASET
# ============================================================

final_counts = {}


for split in [
    "train",
    "valid",
    "test"
]:

    img_count = len([
        p
        for p in (
            BOOST_ROOT /
            split /
            "images"
        ).iterdir()
        if p.is_file()
    ])

    lbl_count = len(
        list(
            (
                BOOST_ROOT /
                split /
                "labels"
            ).glob("*.txt")
        )
    )


    final_counts[
        split
    ] = {
        "images":
            img_count,

        "labels":
            lbl_count,
    }


    assert (
        img_count
        ==
        lbl_count
    )


assert (
    final_counts[
        "valid"
    ][
        "images"
    ]
    == 173
)


assert (
    final_counts[
        "test"
    ][
        "images"
    ]
    == 115
)


assert (
    final_counts[
        "train"
    ][
        "images"
    ]
    ==
    865 + boost_count
)


# ============================================================
# SAVE SUMMARY
# ============================================================

summary = {

    "original_train":
        865,

    "hard_selected":
        int(
            len(selected)
        ),

    "boost_copies":
        int(
            boost_count
        ),

    "final_train":
        int(
            final_counts[
                "train"
            ][
                "images"
            ]
        ),

    "valid":
        173,

    "test":
        115,

    "hard_by_source":
        {
            str(k): int(v)
            for k, v
            in source_summary.items()
        },

    "dataset":
        str(
            BOOST_ROOT
        ),
}


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("CELL 7 FINAL SUMMARY")
print("=" * 100)

print(f"""
HARD-MINED DATASET:
{BOOST_ROOT}

CLASS CONTRACT:
0 = processing_area
1 = lesion

ORIGINAL TRAIN:
865

HARD SELECTED:
{len(selected)}

BOOST COPIES:
{boost_count}

FINAL TRAIN:
{final_counts['train']['images']}

VALID:
173
UNCHANGED

TEST:
115
UNCHANGED

VALIDATION USED FOR MINING:
NO

TEST USED FOR MINING:
NO

REPORT:
{HARD_REPORT}

SUMMARY:
{SUMMARY_JSON}

✅ Y1/Y2 compatibility preserved
✅ train-only mining
✅ no validation leakage
✅ no test leakage
✅ ready for refinement training
""")

print("=" * 100)
print("✅ MODULE 2 CELL 7 COMPLETE")
print("=" * 100)

AI_Y2 CELL 7 — FRESH-RUNTIME GPU CHECK
GPU       : NVIDIA A100-SXM4-80GB
Allocated : 56.55 GB
Reserved  : 60.82 GB


RuntimeError: 
❌ GPU still has >5 GB allocated.
Colab Runtime → Restart session karo, then ONLY this cell run karo.

In [ ]:
# ============================================================
# AI_Y2 — FULL COLAB BACKUP
#
# Purpose:
# Pack ALL useful AI_Y2 work from /content into one ZIP
# before deleting/resetting the Colab runtime.
#
# Includes when present:
# - AI_Y2_FINAL dataset
# - Module 1 work/raw data
# - Module 2 runs/reports/candidates
# - ai_y2.pt
# - hard-mined dataset (if partially/fully created)
# - original source dataset ZIP
# - YOLO26-M pretrained checkpoint
# - JSON / CSV / YAML / TXT reports
# - environment + SHA256 manifest
#
# Then starts browser download.
# ============================================================

from pathlib import Path
import os
import sys
import json
import hashlib
import zipfile
import platform
import subprocess
from datetime import datetime

from google.colab import files


# ============================================================
# PATHS
# ============================================================

CONTENT = Path("/content")

BACKUP_STAGE = CONTENT / "AI_Y2_FULL_BACKUP_STAGE"
ZIP_PATH = CONTENT / "AI_Y2_FULL_BACKUP.zip"

# clean old temporary backup if present
if BACKUP_STAGE.exists():
    import shutil
    shutil.rmtree(BACKUP_STAGE)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

BACKUP_STAGE.mkdir(parents=True, exist_ok=True)


# ============================================================
# PROJECT ITEMS TO INCLUDE
# ============================================================

explicit_items = [

    # -----------------------
    # FINAL DATASET
    # -----------------------
    CONTENT / "AI_Y2_FINAL",

    # -----------------------
    # MODULE 1
    # -----------------------
    CONTENT / "AI_Y2_MODULE1_RAW",
    CONTENT / "AI_Y2_MODULE1_WORK",

    # -----------------------
    # MODULE 2
    # -----------------------
    CONTENT / "AI_Y2_MODULE2",

    # -----------------------
    # HARD-MINING OUTPUT
    # May or may not exist yet.
    # -----------------------
    CONTENT / "AI_Y2_HARDMINED",

    # -----------------------
    # FINAL/CANDIDATE MODEL
    # -----------------------
    CONTENT / "ai_y2.pt",

    # -----------------------
    # BASE MODEL
    # Helpful in fresh runtime.
    # -----------------------
    CONTENT / "yolo26m-seg.pt",

    # -----------------------
    # ORIGINAL DATASET ZIP
    # -----------------------
    CONTENT / "yolo_dataset-20261001T214553Z-1-001.zip",
]


# Also catch useful AI_Y2 files/directories
# that may exist under /content but were not listed above.
auto_candidates = []

for p in CONTENT.iterdir():

    name_lower = p.name.lower()

    if (
        p.name.startswith("AI_Y2")
        or
        p.name.startswith("ai_y2")
        or
        "oralscan" in name_lower
    ):
        auto_candidates.append(p)


# combine without duplicates
all_items = []

seen = set()

for p in explicit_items + auto_candidates:

    p = Path(p)

    if p.exists():

        resolved = str(p.resolve())

        if resolved not in seen:

            seen.add(resolved)
            all_items.append(p)


# ============================================================
# COPY PROJECT CONTENT INTO STAGING FOLDER
# ============================================================

import shutil

copied = []

print("=" * 100)
print("AI_Y2 FULL BACKUP")
print("=" * 100)

print("\nItems found:\n")

for src in all_items:

    dst = BACKUP_STAGE / src.name

    print(f"FOUND : {src}")

    if src.is_dir():

        shutil.copytree(
            src,
            dst,
            dirs_exist_ok=True
        )

    else:

        shutil.copy2(
            src,
            dst
        )

    copied.append(dst)


# ============================================================
# SAVE ENVIRONMENT INFO
# ============================================================

env_txt = BACKUP_STAGE / "ENVIRONMENT_INFO.txt"

env_lines = []

env_lines.append(
    f"Backup created: {datetime.now().isoformat()}"
)

env_lines.append(
    f"Python: {sys.version}"
)

env_lines.append(
    f"Platform: {platform.platform()}"
)

try:
    import torch

    env_lines.append(
        f"PyTorch: {torch.__version__}"
    )

    env_lines.append(
        f"CUDA available: {torch.cuda.is_available()}"
    )

    if torch.cuda.is_available():

        env_lines.append(
            f"GPU: {torch.cuda.get_device_name(0)}"
        )

        env_lines.append(
            f"CUDA allocated GB: "
            f"{torch.cuda.memory_allocated()/1024**3:.3f}"
        )

        env_lines.append(
            f"CUDA reserved GB: "
            f"{torch.cuda.memory_reserved()/1024**3:.3f}"
        )

except Exception as e:

    env_lines.append(
        f"PyTorch info error: {e}"
    )


try:
    import ultralytics

    env_lines.append(
        f"Ultralytics: {ultralytics.__version__}"
    )

except Exception as e:

    env_lines.append(
        f"Ultralytics info error: {e}"
    )


env_txt.write_text(
    "\n".join(env_lines),
    encoding="utf-8"
)


# ============================================================
# PIP FREEZE
# ============================================================

try:

    freeze_output = subprocess.check_output(
        [
            sys.executable,
            "-m",
            "pip",
            "freeze"
        ],
        text=True
    )

    (
        BACKUP_STAGE /
        "pip_freeze.txt"
    ).write_text(
        freeze_output,
        encoding="utf-8"
    )

except Exception as e:

    (
        BACKUP_STAGE /
        "pip_freeze.txt"
    ).write_text(
        f"Could not generate pip freeze:\n{e}",
        encoding="utf-8"
    )


# ============================================================
# SHA256 MANIFEST
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


manifest_records = []

all_backup_files = sorted([
    p
    for p in BACKUP_STAGE.rglob("*")
    if p.is_file()
])


print("\nComputing SHA256 manifest...")

for idx, p in enumerate(
    all_backup_files,
    start=1
):

    rel = p.relative_to(
        BACKUP_STAGE
    )

    digest = sha256_file(p)

    manifest_records.append({

        "path":
            str(rel),

        "size_bytes":
            p.stat().st_size,

        "sha256":
            digest,
    })

    if idx % 250 == 0:

        print(
            f"Hashed {idx}/{len(all_backup_files)} files"
        )


manifest_json = (
    BACKUP_STAGE /
    "BACKUP_MANIFEST.json"
)

manifest_json.write_text(
    json.dumps(
        {
            "created_at":
                datetime.now().isoformat(),

            "file_count":
                len(manifest_records),

            "files":
                manifest_records,
        },
        indent=2
    ),
    encoding="utf-8"
)


# Human-readable SHA256 list
sha_txt = (
    BACKUP_STAGE /
    "SHA256SUMS.txt"
)

sha_txt.write_text(
    "\n".join(
        f"{x['sha256']}  {x['path']}"
        for x in manifest_records
    ),
    encoding="utf-8"
)


# ============================================================
# SAVE RESTORE README
# ============================================================

readme = BACKUP_STAGE / "README_RESTORE.txt"

readme.write_text(
"""AI_Y2 FULL BACKUP
===================

This archive contains the current AI_Y2 Colab project state.

Important expected items may include:

AI_Y2_FINAL/
    Frozen clean Y2 dataset
    train / valid / test
    data.yaml
    dataset_manifest.csv
    AI_Y2_DATASET_FREEZE.json

AI_Y2_MODULE1_RAW/
AI_Y2_MODULE1_WORK/
    Module-1 intermediate/raw/remapping files if still present.

AI_Y2_MODULE2/
    Training runs
    reports
    candidate_y2_phase1.pt
    candidate_y2_phase2.pt
    validation/failure-analysis outputs

ai_y2.pt
    Current compatibility-passed AI_Y2 model.

AI_Y2_HARDMINED/
    May be absent or incomplete because hard mining had not
    successfully completed before this backup.

yolo26m-seg.pt
    Base YOLO26-M segmentation checkpoint if present.

BACKUP_MANIFEST.json
SHA256SUMS.txt
    Used to verify extracted files after restoration.

IMPORTANT CONTRACT
------------------
Task:
    segmentation

Input:
    640 x 640

Class IDs:
    0 = processing_area
    1 = lesion

Expected Y1-compatible output interface:
    result.boxes.cls
    result.boxes.conf
    result.boxes.xyxy
    result.masks.data
    result.masks.xy

CURRENT STATUS
--------------
Module 1:
    completed/frozen.

Module 2:
    baseline training completed.
    Phase 1 selected.
    ai_y2.pt created.
    compatibility passed.
    validation/test analysis completed.

Next planned step:
    restart with clean GPU,
    restore archive,
    perform TRAIN-ONLY hard-example mining,
    then boundary-focused refinement.

Do NOT train on validation or test data.
""",
    encoding="utf-8"
)


# ============================================================
# CREATE ZIP64 ARCHIVE
# ============================================================

print("\n" + "=" * 100)
print("CREATING ZIP")
print("=" * 100)


with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True
) as zf:

    staged_files = sorted([
        p
        for p in BACKUP_STAGE.rglob("*")
        if p.is_file()
    ])

    for idx, file_path in enumerate(
        staged_files,
        start=1
    ):

        arcname = Path(
            "AI_Y2_FULL_BACKUP"
        ) / file_path.relative_to(
            BACKUP_STAGE
        )

        zf.write(
            file_path,
            arcname=str(arcname)
        )

        if idx % 250 == 0:

            print(
                f"Packed {idx}/{len(staged_files)} files"
            )


# ============================================================
# VERIFY ZIP
# ============================================================

print("\nVerifying ZIP integrity...")

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zf:

    bad_file = zf.testzip()


if bad_file is not None:

    raise RuntimeError(
        f"❌ ZIP integrity failed at: {bad_file}"
    )


zip_size_gb = (
    ZIP_PATH.stat().st_size
    /
    (1024 ** 3)
)


print("\n" + "=" * 100)
print("BACKUP COMPLETE")
print("=" * 100)

print(
    f"ZIP:\n{ZIP_PATH}"
)

print(
    f"\nZIP size: "
    f"{zip_size_gb:.3f} GB"
)

print(
    f"\nFiles packed: "
    f"{len(staged_files)}"
)

print(
    "\nZIP integrity: ✅ PASS"
)

print(
    "\nSHA256 manifest: ✅ CREATED"
)

print("""
IMPORTANT:
Browser download complete hone se pehle
runtime ko disconnect/delete MAT karna.
""")


# ============================================================
# START DOWNLOAD
# ============================================================

print(
    "Starting browser download..."
)

files.download(
    str(ZIP_PATH)
)

AI_Y2 FULL BACKUP

Items found:

FOUND : /content/AI_Y2_FINAL
FOUND : /content/AI_Y2_MODULE1_RAW
FOUND : /content/AI_Y2_MODULE1_WORK
FOUND : /content/AI_Y2_MODULE2
FOUND : /content/ai_y2.pt
FOUND : /content/yolo26m-seg.pt
FOUND : /content/yolo_dataset-20261001T214553Z-1-001.zip
FOUND : /content/AI_Y2_FULL_BACKUP_STAGE

Computing SHA256 manifest...
Hashed 250/9450 files
Hashed 500/9450 files
Hashed 750/9450 files
Hashed 1000/9450 files
Hashed 1250/9450 files
Hashed 1500/9450 files
Hashed 1750/9450 files
Hashed 2000/9450 files
Hashed 2250/9450 files
Hashed 2500/9450 files
Hashed 2750/9450 files
Hashed 3000/9450 files
Hashed 3250/9450 files
Hashed 3500/9450 files
Hashed 3750/9450 files
Hashed 4000/9450 files
Hashed 4250/9450 files
Hashed 4500/9450 files
Hashed 4750/9450 files
Hashed 5000/9450 files
Hashed 5250/9450 files
Hashed 5500/9450 files
Hashed 5750/9450 files
Hashed 6000/9450 files
Hashed 6250/9450 files
Hashed 6500/9450 files
Hashed 6750/9450 files
Hashed 7000/9450 files
Hashed 72

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# ============================================================
# AI_Y2 — FRESH RUNTIME RESTORE + FULL VERIFICATION
#
# ZIP:
# /content/AI_Y2_FULL_BACKUP.zip
#
# Does:
# 1. ZIP integrity check
# 2. Extract backup
# 3. Verify SHA256 manifest
# 4. Restore project into /content
# 5. Verify frozen dataset fingerprint
# 6. Verify dataset counts / classes
# 7. Verify AI_Y2 model contract
# 8. Verify fresh GPU memory
# ============================================================

from pathlib import Path
import os
import sys
import gc
import json
import yaml
import shutil
import hashlib
import zipfile
import subprocess


# ============================================================
# PATHS
# ============================================================

ZIP_PATH = Path(
    "/content/AI_Y2_FULL_BACKUP.zip"
)

RESTORE_TMP = Path(
    "/content/_AI_Y2_RESTORE_TMP"
)

EXPECTED_DATASET_SHA256 = (
    "fcf15becb5390bc23d79c81d47e0cf45"
    "fe386a74217872c3788ffcf6e3544e6e"
)

EXPECTED_CLASSES = {
    0: "processing_area",
    1: "lesion",
}


# ============================================================
# START
# ============================================================

print("=" * 100)
print("AI_Y2 — FRESH RUNTIME RESTORE")
print("=" * 100)

if not ZIP_PATH.exists():

    raise FileNotFoundError(
        f"❌ ZIP not found:\n{ZIP_PATH}"
    )


print(
    f"\nZIP found:"
    f"\n{ZIP_PATH}"
)

print(
    f"ZIP size: "
    f"{ZIP_PATH.stat().st_size / 1024**3:.3f} GB"
)


# ============================================================
# ZIP INTEGRITY
# ============================================================

print("\n" + "=" * 100)
print("1. ZIP INTEGRITY CHECK")
print("=" * 100)


with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zf:

    bad = zf.testzip()

    names = zf.namelist()


if bad is not None:

    raise RuntimeError(
        f"❌ ZIP corrupt at:\n{bad}"
    )


print("✅ ZIP integrity PASS")

print(
    f"Archive entries: {len(names)}"
)


# ============================================================
# CLEAN TEMP EXTRACTION ONLY
# ============================================================

if RESTORE_TMP.exists():

    shutil.rmtree(
        RESTORE_TMP
    )


RESTORE_TMP.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# EXTRACT
# ============================================================

print("\n" + "=" * 100)
print("2. EXTRACTING BACKUP")
print("=" * 100)


with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zf:

    zf.extractall(
        RESTORE_TMP
    )


BACKUP_ROOT = (
    RESTORE_TMP /
    "AI_Y2_FULL_BACKUP"
)


if not BACKUP_ROOT.exists():

    raise RuntimeError(
        "❌ Expected backup root missing:\n"
        f"{BACKUP_ROOT}"
    )


print(
    f"✅ Extracted to:\n"
    f"{BACKUP_ROOT}"
)


# ============================================================
# VERIFY BACKUP MANIFEST
# ============================================================

print("\n" + "=" * 100)
print("3. SHA256 BACKUP VERIFICATION")
print("=" * 100)


MANIFEST_JSON = (
    BACKUP_ROOT /
    "BACKUP_MANIFEST.json"
)


if not MANIFEST_JSON.exists():

    raise FileNotFoundError(
        f"❌ Missing backup manifest:\n"
        f"{MANIFEST_JSON}"
    )


with open(
    MANIFEST_JSON,
    "r",
    encoding="utf-8"
) as f:

    backup_manifest = json.load(f)


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


bad_files = []
missing_files = []

records = backup_manifest[
    "files"
]


for idx, record in enumerate(
    records,
    start=1
):

    path = (
        BACKUP_ROOT /
        record["path"]
    )

    if not path.exists():

        missing_files.append(
            record["path"]
        )

        continue


    current_hash = sha256_file(
        path
    )


    if current_hash != record["sha256"]:

        bad_files.append({
            "path":
                record["path"],

            "expected":
                record["sha256"],

            "actual":
                current_hash,
        })


    if idx % 500 == 0:

        print(
            f"Verified "
            f"{idx}/{len(records)} files"
        )


print(
    f"\nManifest files : "
    f"{len(records)}"
)

print(
    f"Missing        : "
    f"{len(missing_files)}"
)

print(
    f"Hash mismatch  : "
    f"{len(bad_files)}"
)


if missing_files:

    print(
        "\nMissing examples:"
    )

    print(
        "\n".join(
            missing_files[:10]
        )
    )


if bad_files:

    print(
        "\nHash mismatch examples:"
    )

    for x in bad_files[:10]:

        print(
            x["path"]
        )


if missing_files or bad_files:

    raise RuntimeError(
        "❌ Backup SHA256 verification FAILED."
    )


print(
    "✅ Every manifest-tracked file "
    "matches the original backup."
)


# ============================================================
# RESTORE INTO /content
# ============================================================

print("\n" + "=" * 100)
print("4. RESTORING PROJECT TO /content")
print("=" * 100)


# Items we actually need for continuation.
restore_names = [

    "AI_Y2_FINAL",

    "AI_Y2_MODULE1_RAW",
    "AI_Y2_MODULE1_WORK",

    "AI_Y2_MODULE2",

    "AI_Y2_HARDMINED",

    "ai_y2.pt",

    "yolo26m-seg.pt",

    "yolo_dataset-20261001T214553Z-1-001.zip",
]


restored = []


for name in restore_names:

    src = (
        BACKUP_ROOT /
        name
    )

    if not src.exists():

        print(
            f"SKIP (not in backup): {name}"
        )

        continue


    dst = Path(
        "/content"
    ) / name


    # Fresh runtime normally has no conflict.
    # If something exists, replace it with verified backup.
    if dst.exists():

        if dst.is_dir():

            shutil.rmtree(
                dst
            )

        else:

            dst.unlink()


    if src.is_dir():

        shutil.copytree(
            src,
            dst
        )

    else:

        shutil.copy2(
            src,
            dst
        )


    restored.append(
        str(dst)
    )

    print(
        f"RESTORED: {dst}"
    )


# ============================================================
# REQUIRED CORE FILES
# ============================================================

DATASET_ROOT = Path(
    "/content/AI_Y2_FINAL"
)

DATA_YAML = (
    DATASET_ROOT /
    "data.yaml"
)

FREEZE_JSON = (
    DATASET_ROOT /
    "AI_Y2_DATASET_FREEZE.json"
)

DATASET_MANIFEST = (
    DATASET_ROOT /
    "dataset_manifest.csv"
)

Y2_MODEL = Path(
    "/content/ai_y2.pt"
)


required = [

    DATASET_ROOT,
    DATA_YAML,
    FREEZE_JSON,
    DATASET_MANIFEST,
    Y2_MODEL,
]


for p in required:

    if not p.exists():

        raise FileNotFoundError(
            f"❌ Required restored item missing:\n"
            f"{p}"
        )


print(
    "\n✅ Core Y2 files restored."
)


# ============================================================
# DATASET YAML / CLASS CONTRACT
# ============================================================

print("\n" + "=" * 100)
print("5. DATASET CLASS CONTRACT")
print("=" * 100)


with open(
    DATA_YAML,
    "r",
    encoding="utf-8"
) as f:

    cfg = yaml.safe_load(f)


print(
    f"nc    : {cfg.get('nc')}"
)

print(
    f"names : {cfg.get('names')}"
)


if cfg.get("nc") != 2:

    raise RuntimeError(
        "❌ Expected nc=2."
    )


if cfg.get("names") != [
    "processing_area",
    "lesion"
]:

    raise RuntimeError(
        "❌ Y1-compatible class order changed."
    )


print(
    "✅ Class IDs/names preserved."
)


# ============================================================
# RECOMPUTE EXACT FROZEN DATASET FINGERPRINT
# ============================================================

print("\n" + "=" * 100)
print("6. FROZEN DATASET FINGERPRINT")
print("=" * 100)


def dataset_fingerprint(root):

    lines = []

    for split in [
        "train",
        "valid",
        "test"
    ]:

        img_dir = (
            root /
            split /
            "images"
        )

        lbl_dir = (
            root /
            split /
            "labels"
        )


        for img in sorted(
            img_dir.iterdir(),
            key=lambda p: p.name
        ):

            if not img.is_file():
                continue


            lbl = (
                lbl_dir /
                f"{img.stem}.txt"
            )


            if not lbl.exists():

                raise RuntimeError(
                    f"Missing label:\n{lbl}"
                )


            lines.append(
                "|".join([
                    split,
                    img.name,
                    sha256_file(img),
                    sha256_file(lbl),
                ])
            )


    return hashlib.sha256(
        "\n".join(
            lines
        ).encode(
            "utf-8"
        )
    ).hexdigest()


current_dataset_hash = (
    dataset_fingerprint(
        DATASET_ROOT
    )
)


print(
    f"Expected:\n"
    f"{EXPECTED_DATASET_SHA256}"
)

print(
    f"\nCurrent:\n"
    f"{current_dataset_hash}"
)


if (
    current_dataset_hash
    != EXPECTED_DATASET_SHA256
):

    raise RuntimeError(
        "❌ Frozen dataset fingerprint mismatch."
    )


print(
    "\n✅ EXACT Module-1 frozen dataset restored."
)


# ============================================================
# SPLIT COUNTS
# ============================================================

print("\n" + "=" * 100)
print("7. DATASET SPLIT COUNTS")
print("=" * 100)


expected_counts = {

    "train":
        865,

    "valid":
        173,

    "test":
        115,
}


for split, expected in expected_counts.items():

    imgs = [
        p
        for p in (
            DATASET_ROOT /
            split /
            "images"
        ).iterdir()
        if p.is_file()
    ]

    labels = list(
        (
            DATASET_ROOT /
            split /
            "labels"
        ).glob(
            "*.txt"
        )
    )


    print(
        f"{split.upper():5} | "
        f"images={len(imgs)} | "
        f"labels={len(labels)}"
    )


    if (
        len(imgs) != expected
        or
        len(labels) != expected
    ):

        raise RuntimeError(
            f"❌ {split} count mismatch."
        )


print(
    "✅ 865 / 173 / 115 restored exactly."
)


# ============================================================
# INSTALL ULTRALYTICS ONLY IF NEEDED
# ============================================================

try:

    import ultralytics

except ImportError:

    print(
        "\nInstalling Ultralytics..."
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics==8.4.171"
    ])


import ultralytics
import torch

from ultralytics import YOLO


# ============================================================
# Y2 MODEL CONTRACT
# ============================================================

print("\n" + "=" * 100)
print("8. RESTORED Y2 MODEL CONTRACT")
print("=" * 100)


model = YOLO(
    str(
        Y2_MODEL
    )
)


model_names = {
    int(k): str(v)
    for k, v
    in model.names.items()
}


print(
    f"Ultralytics : "
    f"{ultralytics.__version__}"
)

print(
    f"Task        : "
    f"{model.task}"
)

print(
    f"Classes     : "
    f"{model_names}"
)

print(
    f"Model SHA256:\n"
    f"{sha256_file(Y2_MODEL)}"
)


if str(
    model.task
) != "segment":

    raise RuntimeError(
        "❌ Restored AI_Y2 is not segmentation."
    )


if model_names != EXPECTED_CLASSES:

    raise RuntimeError(
        "❌ Restored AI_Y2 class contract mismatch."
    )


print(
    "✅ Y1-compatible AI_Y2 model restored."
)


# Release model BEFORE checking memory
del model

gc.collect()

torch.cuda.empty_cache()


# ============================================================
# GPU STATE
# ============================================================

print("\n" + "=" * 100)
print("9. FRESH GPU STATE")
print("=" * 100)


if not torch.cuda.is_available():

    raise RuntimeError(
        "❌ CUDA unavailable."
    )


allocated = (
    torch.cuda.memory_allocated()
    /
    1024**3
)

reserved = (
    torch.cuda.memory_reserved()
    /
    1024**3
)


print(
    f"GPU       : "
    f"{torch.cuda.get_device_name(0)}"
)

print(
    f"Allocated : "
    f"{allocated:.3f} GB"
)

print(
    f"Reserved  : "
    f"{reserved:.3f} GB"
)


# ============================================================
# CURRENT MODULE 2 ARTIFACTS
# ============================================================

print("\n" + "=" * 100)
print("10. MODULE 2 RESTORED ARTIFACTS")
print("=" * 100)


artifacts = [

    Path(
        "/content/AI_Y2_MODULE2/"
        "candidate_y2_phase1.pt"
    ),

    Path(
        "/content/AI_Y2_MODULE2/"
        "candidate_y2_phase2.pt"
    ),

    Path(
        "/content/AI_Y2_MODULE2/reports/"
        "validation_candidate_comparison.json"
    ),

    Path(
        "/content/AI_Y2_MODULE2/reports/"
        "AI_Y2_final_test_report.json"
    ),

    Path(
        "/content/AI_Y2_MODULE2/reports/"
        "validation_failure_analysis.csv"
    ),

    Path(
        "/content/AI_Y2_MODULE2/reports/"
        "validation_hard_cases.csv"
    ),

    Path(
        "/content/AI_Y2_MODULE2/reports/"
        "cell6_geometry_summary.json"
    ),
]


for p in artifacts:

    print(
        f"{'✅' if p.exists() else '—'} "
        f"{p}"
    )


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("RESTORE COMPLETE")
print("=" * 100)

print(f"""
Backup integrity            : PASS
Dataset fingerprint         : PASS

Dataset:
train                       : 865
valid                       : 173
test                        : 115

Class 0                     : processing_area
Class 1                     : lesion

AI_Y2 task                  : segment

AI_Y2 model                 :
{Y2_MODEL}

Dataset                     :
{DATASET_ROOT}

GPU allocated after cleanup : {allocated:.3f} GB
GPU reserved after cleanup  : {reserved:.3f} GB

NEXT:
TRAIN-ONLY hard-example mining (Cell 7)
on this clean runtime.

Validation will NOT be mined.
Test will NOT be mined.
""")

print("=" * 100)
print("✅ AI_Y2 RESTORE VERIFIED")
print("=" * 100)

AI_Y2 — FRESH RUNTIME RESTORE

ZIP found:
/content/AI_Y2_FULL_BACKUP.zip
ZIP size: 1.034 GB

1. ZIP INTEGRITY CHECK
✅ ZIP integrity PASS
Archive entries: 9453

2. EXTRACTING BACKUP
✅ Extracted to:
/content/_AI_Y2_RESTORE_TMP/AI_Y2_FULL_BACKUP

3. SHA256 BACKUP VERIFICATION
Verified 500/9450 files
Verified 1000/9450 files
Verified 1500/9450 files
Verified 2000/9450 files
Verified 2500/9450 files
Verified 3000/9450 files
Verified 3500/9450 files
Verified 4000/9450 files
Verified 4500/9450 files
Verified 5000/9450 files
Verified 5500/9450 files
Verified 6000/9450 files
Verified 6500/9450 files
Verified 7000/9450 files
Verified 7500/9450 files
Verified 8000/9450 files
Verified 8500/9450 files
Verified 9000/9450 files

Manifest files : 9450
Missing        : 0
Hash mismatch  : 0
✅ Every manifest-tracked file matches the original backup.

4. RESTORING PROJECT TO /content
RESTORED: /content/AI_Y2_FINAL
RESTORED: /content/AI_Y2_MODULE1_RAW
RESTORED: /content/AI_Y2_MODULE1_WORK
RESTORED: /conten

In [2]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 7
# FRESH-RUNTIME TRAIN-ONLY HARD EXAMPLE MINING
# ============================================================

from pathlib import Path
import gc
import json
import shutil

import cv2
import numpy as np
import pandas as pd
import torch

from ultralytics import YOLO


# ============================================================
# PATHS
# ============================================================

MODEL_PATH = Path("/content/ai_y2.pt")

ORIGINAL_ROOT = Path("/content/AI_Y2_FINAL")

TRAIN_IMG_DIR = ORIGINAL_ROOT / "train" / "images"
TRAIN_LBL_DIR = ORIGINAL_ROOT / "train" / "labels"

MANIFEST_PATH = ORIGINAL_ROOT / "dataset_manifest.csv"

BOOST_ROOT = Path("/content/AI_Y2_HARDMINED")

REPORT_DIR = Path("/content/AI_Y2_MODULE2/reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

HARD_REPORT = REPORT_DIR / "train_hard_mining.csv"
SUMMARY_JSON = REPORT_DIR / "train_hard_mining_summary.json"


# ============================================================
# CONFIG
# ============================================================

IMGSZ = 640
CONF = 0.25
IOU = 0.70

PRED_BATCH = 1

EXPECTED_NAMES = {
    0: "processing_area",
    1: "lesion",
}

LESION_IOU_HARD = 0.50
PA_IOU_HARD = 0.50
CONTAINMENT_HARD = 0.50

BOOST_COPIES = 1


# ============================================================
# GPU CHECK
# ============================================================

print("=" * 100)
print("AI_Y2 MODULE 2 — CELL 7")
print("TRAIN-ONLY HARD EXAMPLE MINING")
print("=" * 100)

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3

print(f"GPU       : {torch.cuda.get_device_name(0)}")
print(f"Allocated : {allocated:.3f} GB")
print(f"Reserved  : {reserved:.3f} GB")

if allocated > 5:
    raise RuntimeError(
        "❌ GPU memory unexpectedly high. Stop here."
    )


# ============================================================
# VERIFY CORE FILES
# ============================================================

for p in [
    MODEL_PATH,
    TRAIN_IMG_DIR,
    TRAIN_LBL_DIR,
    MANIFEST_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(f"❌ Missing: {p}")


# ============================================================
# HELPERS
# ============================================================

def polygon_to_mask(points, w, h):

    mask = np.zeros((h, w), dtype=np.uint8)

    pts = points.copy()
    pts[:, 0] *= w
    pts[:, 1] *= h

    pts = np.round(pts).astype(np.int32)

    cv2.fillPoly(
        mask,
        [pts],
        1
    )

    return mask


def parse_gt(label_path, w, h):

    out = {
        0: [],
        1: [],
    }

    text = label_path.read_text(
        encoding="utf-8"
    ).strip()

    for line in text.splitlines():

        parts = line.split()

        cls_id = int(parts[0])

        coords = np.array(
            [float(v) for v in parts[1:]],
            dtype=np.float32
        ).reshape(-1, 2)

        out[cls_id].append(
            polygon_to_mask(
                coords,
                w,
                h
            )
        )

    return out


def union_masks(masks, h, w):

    out = np.zeros(
        (h, w),
        dtype=np.uint8
    )

    for m in masks:
        out = np.maximum(
            out,
            m.astype(np.uint8)
        )

    return out


def resize_mask(mask, w, h):

    resized = cv2.resize(
        mask.astype(np.float32),
        (w, h),
        interpolation=cv2.INTER_NEAREST
    )

    return (
        resized > 0.5
    ).astype(np.uint8)


def mask_iou(a, b):

    inter = np.logical_and(
        a > 0,
        b > 0
    ).sum()

    union = np.logical_or(
        a > 0,
        b > 0
    ).sum()

    if union == 0:
        return 1.0

    return float(
        inter / union
    )


def containment_ratio(
    lesion,
    processing
):

    lesion_area = (
        lesion > 0
    ).sum()

    if lesion_area == 0:
        return None

    inside = np.logical_and(
        lesion > 0,
        processing > 0
    ).sum()

    return float(
        inside / lesion_area
    )


# ============================================================
# LOAD MANIFEST
# ============================================================

manifest = pd.read_csv(
    MANIFEST_PATH
)

train_manifest = manifest[
    manifest["final_split"] == "train"
].copy()

source_lookup = {
    row["final_image"]:
        row["source_category"]
    for _, row in train_manifest.iterrows()
}


# ============================================================
# TRAIN IMAGE LIST
# ============================================================

train_images = sorted([
    p
    for p in TRAIN_IMG_DIR.iterdir()
    if p.is_file()
])

assert len(train_images) == 865, (
    f"Expected 865 train images, got {len(train_images)}"
)


# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO(
    str(MODEL_PATH)
)

model_names = {
    int(k): str(v)
    for k, v in model.names.items()
}

assert model_names == EXPECTED_NAMES
assert str(model.task) == "segment"

print(f"\nClasses : {model_names}")
print(f"Task    : {model.task}")

print(f"""
Train images           : {len(train_images)}
Batch                  : {PRED_BATCH}
Stream                 : True
Half precision         : True
Retina masks           : False

Validation used mining : NO
Test used mining       : NO
""")


# ============================================================
# STREAMING INFERENCE
# ============================================================

prediction_stream = model.predict(

    source=[
        str(p)
        for p in train_images
    ],

    imgsz=IMGSZ,
    conf=CONF,
    iou=IOU,

    device=0,
    batch=PRED_BATCH,

    half=True,
    retina_masks=False,

    stream=True,
    verbose=False,
    save=False
)


# ============================================================
# ANALYZE ONE IMAGE AT A TIME
# ============================================================

rows = []


for idx, (img_path, result) in enumerate(
    zip(
        train_images,
        prediction_stream
    ),
    start=1
):

    image = cv2.imread(
        str(img_path)
    )

    if image is None:
        raise RuntimeError(
            f"Unreadable image: {img_path}"
        )

    h, w = image.shape[:2]

    label_path = (
        TRAIN_LBL_DIR /
        f"{img_path.stem}.txt"
    )

    gt = parse_gt(
        label_path,
        w,
        h
    )

    gt_pa = union_masks(
        gt[0],
        h,
        w
    )

    gt_lesion = union_masks(
        gt[1],
        h,
        w
    )


    pred_pa_masks = []
    pred_lesion_masks = []


    if (
        result.boxes is not None
        and
        len(result.boxes) > 0
        and
        result.masks is not None
    ):

        classes = (
            result.boxes.cls
            .detach()
            .cpu()
            .numpy()
            .astype(np.int32)
        )

        masks = (
            result.masks.data
            .detach()
            .float()
            .cpu()
            .numpy()
        )

        for cls_id, raw_mask in zip(
            classes,
            masks
        ):

            m = resize_mask(
                raw_mask,
                w,
                h
            )

            if cls_id == 0:
                pred_pa_masks.append(m)

            elif cls_id == 1:
                pred_lesion_masks.append(m)

        del classes
        del masks


    pred_pa = union_masks(
        pred_pa_masks,
        h,
        w
    )

    pred_lesion = union_masks(
        pred_lesion_masks,
        h,
        w
    )


    gt_has_lesion = (
        len(gt[1]) > 0
    )

    pred_has_lesion = (
        len(pred_lesion_masks) > 0
    )


    pa_iou = mask_iou(
        gt_pa,
        pred_pa
    )


    lesion_iou = (
        mask_iou(
            gt_lesion,
            pred_lesion
        )
        if gt_has_lesion
        else None
    )


    pred_containment = (
        containment_ratio(
            pred_lesion,
            pred_pa
        )
        if pred_has_lesion
        else None
    )


    missed_lesion = (
        gt_has_lesion
        and
        not pred_has_lesion
    )


    false_lesion = (
        not gt_has_lesion
        and
        pred_has_lesion
    )


    low_lesion_iou = (
        gt_has_lesion
        and
        lesion_iou < LESION_IOU_HARD
    )


    low_pa_iou = (
        pa_iou < PA_IOU_HARD
    )


    poor_containment = (
        pred_has_lesion
        and
        pred_containment is not None
        and
        pred_containment < CONTAINMENT_HARD
    )


    hard = (
        missed_lesion
        or
        false_lesion
        or
        low_lesion_iou
        or
        low_pa_iou
        or
        poor_containment
    )


    # ========================================================
    # HARD SCORE
    # ========================================================

    hard_score = 0.0


    if missed_lesion:
        hard_score += 100.0


    if false_lesion:
        hard_score += 75.0


    if gt_has_lesion:

        hard_score += (
            50.0 *
            (
                1.0
                -
                lesion_iou
            )
        )


    hard_score += (
        30.0 *
        (
            1.0
            -
            pa_iou
        )
    )


    if poor_containment:

        hard_score += (
            25.0 *
            (
                1.0
                -
                pred_containment
            )
        )


    rows.append({

        "filename":
            img_path.name,

        "source":
            source_lookup.get(
                img_path.name,
                "unknown"
            ),

        "gt_has_lesion":
            gt_has_lesion,

        "missed_lesion":
            missed_lesion,

        "false_lesion":
            false_lesion,

        "lesion_iou":
            lesion_iou,

        "pa_iou":
            pa_iou,

        "pred_containment":
            pred_containment,

        "low_lesion_iou":
            low_lesion_iou,

        "low_pa_iou":
            low_pa_iou,

        "poor_containment":
            poor_containment,

        "hard":
            hard,

        "hard_score":
            hard_score,
    })


    # release tensors
    del result
    del image
    del gt
    del gt_pa
    del gt_lesion
    del pred_pa
    del pred_lesion
    del pred_pa_masks
    del pred_lesion_masks


    if idx % 50 == 0:

        gc.collect()
        torch.cuda.empty_cache()

        allocated = (
            torch.cuda.memory_allocated()
            /
            1024**3
        )

        print(
            f"Processed {idx}/{len(train_images)} "
            f"| GPU allocated={allocated:.2f} GB"
        )


# ============================================================
# CLEAN GPU
# ============================================================

del prediction_stream
del model

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


print(
    f"\nGPU after mining: "
    f"{torch.cuda.memory_allocated()/1024**3:.3f} GB"
)


# ============================================================
# SAVE HARD REPORT
# ============================================================

hard_df = pd.DataFrame(
    rows
)

hard_df.to_csv(
    HARD_REPORT,
    index=False
)

selected = hard_df[
    hard_df["hard"] == True
].copy()


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("TRAIN HARD-MINING SUMMARY")
print("=" * 100)


print(
    f"Train images            : {len(hard_df)}"
)

print(
    f"Hard images selected    : {len(selected)}"
)

print(
    f"Hard fraction           : "
    f"{100*len(selected)/len(hard_df):.2f}%"
)

print(
    f"Missed lesions          : "
    f"{int(hard_df['missed_lesion'].sum())}"
)

print(
    f"False lesion images     : "
    f"{int(hard_df['false_lesion'].sum())}"
)

print(
    f"Low lesion IoU          : "
    f"{int(hard_df['low_lesion_iou'].sum())}"
)

print(
    f"Low processing IoU      : "
    f"{int(hard_df['low_pa_iou'].sum())}"
)

print(
    f"Poor containment        : "
    f"{int(hard_df['poor_containment'].sum())}"
)


# ============================================================
# SOURCE SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("HARD CASES BY SOURCE")
print("=" * 100)


source_summary = (
    selected[
        "source"
    ]
    .value_counts()
)


print(
    source_summary.to_string()
)


# ============================================================
# BUILD BOOSTED DATASET
# ============================================================

print("\n" + "=" * 100)
print("BUILDING HARD-MINED DATASET")
print("=" * 100)


if BOOST_ROOT.exists():

    shutil.rmtree(
        BOOST_ROOT
    )


for split in [
    "train",
    "valid",
    "test"
]:

    (
        BOOST_ROOT /
        split /
        "images"
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    (
        BOOST_ROOT /
        split /
        "labels"
    ).mkdir(
        parents=True,
        exist_ok=True
    )


# Copy original dataset exactly
for split in [
    "train",
    "valid",
    "test"
]:

    src_img_dir = (
        ORIGINAL_ROOT /
        split /
        "images"
    )

    src_lbl_dir = (
        ORIGINAL_ROOT /
        split /
        "labels"
    )

    dst_img_dir = (
        BOOST_ROOT /
        split /
        "images"
    )

    dst_lbl_dir = (
        BOOST_ROOT /
        split /
        "labels"
    )


    for p in src_img_dir.iterdir():

        if p.is_file():

            shutil.copy2(
                p,
                dst_img_dir / p.name
            )


    for p in src_lbl_dir.iterdir():

        if p.is_file():

            shutil.copy2(
                p,
                dst_lbl_dir / p.name
            )


# Add ONE duplicate of every selected hard train image
boost_count = 0


for _, row in selected.iterrows():

    filename = row["filename"]

    src_img = (
        TRAIN_IMG_DIR /
        filename
    )

    src_lbl = (
        TRAIN_LBL_DIR /
        f"{Path(filename).stem}.txt"
    )

    suffix = "__hardboost1"

    new_img = (
        f"{Path(filename).stem}"
        f"{suffix}"
        f"{Path(filename).suffix}"
    )

    new_lbl = (
        f"{Path(filename).stem}"
        f"{suffix}.txt"
    )


    shutil.copy2(
        src_img,
        BOOST_ROOT /
        "train" /
        "images" /
        new_img
    )

    shutil.copy2(
        src_lbl,
        BOOST_ROOT /
        "train" /
        "labels" /
        new_lbl
    )

    boost_count += 1


# ============================================================
# DATA YAML
# ============================================================

data_yaml = f"""path: {BOOST_ROOT}
train: train/images
val: valid/images
test: test/images
nc: 2
names:
  - processing_area
  - lesion
"""


(
    BOOST_ROOT /
    "data.yaml"
).write_text(
    data_yaml,
    encoding="utf-8"
)


# ============================================================
# VERIFY COUNTS
# ============================================================

counts = {}


for split in [
    "train",
    "valid",
    "test"
]:

    img_count = len([
        p
        for p in (
            BOOST_ROOT /
            split /
            "images"
        ).iterdir()
        if p.is_file()
    ])

    lbl_count = len(
        list(
            (
                BOOST_ROOT /
                split /
                "labels"
            ).glob("*.txt")
        )
    )

    counts[split] = {
        "images":
            img_count,

        "labels":
            lbl_count,
    }

    assert img_count == lbl_count


assert counts["valid"]["images"] == 173
assert counts["test"]["images"] == 115

assert (
    counts["train"]["images"]
    ==
    865 + boost_count
)


# ============================================================
# SAVE SUMMARY
# ============================================================

summary = {

    "original_train":
        865,

    "hard_selected":
        int(
            len(selected)
        ),

    "boost_copies":
        int(
            boost_count
        ),

    "final_train":
        int(
            counts["train"]["images"]
        ),

    "valid":
        173,

    "test":
        115,

    "hard_by_source":
        {
            str(k): int(v)
            for k, v
            in source_summary.items()
        },

    "dataset":
        str(
            BOOST_ROOT
        ),
}


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("CELL 7 FINAL SUMMARY")
print("=" * 100)

print(f"""
Hard-mined dataset:
{BOOST_ROOT}

Original train:
865

Hard selected:
{len(selected)}

Boost copies:
{boost_count}

Final train:
{counts['train']['images']}

Validation:
173 — UNCHANGED

Test:
115 — UNCHANGED

Classes:
0 = processing_area
1 = lesion

Validation mining:
NO

Test mining:
NO

Hard report:
{HARD_REPORT}

Summary:
{SUMMARY_JSON}

✅ Train-only hard mining
✅ No validation leakage
✅ No test leakage
✅ Y1 class contract preserved
""")

print("=" * 100)
print("✅ MODULE 2 CELL 7 COMPLETE")
print("=" * 100)

AI_Y2 MODULE 2 — CELL 7
TRAIN-ONLY HARD EXAMPLE MINING
GPU       : NVIDIA A100-SXM4-80GB
Allocated : 0.000 GB
Reserved  : 0.000 GB

Classes : {0: 'processing_area', 1: 'lesion'}
Task    : segment

Train images           : 865
Batch                  : 1
Stream                 : True
Half precision         : True
Retina masks           : False

Validation used mining : NO
Test used mining       : NO

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Processed 50/865 | GPU allocated=9.37 GB
Processed 100/865 | GPU allocated=9.37 GB
Processed 150/865 | GPU allocated=9.37 GB
Processed 200/865 | GPU allocated=9.37 GB
Processed 250/865 | GPU allocated=9.37 GB
Processed 300/865 | GPU allocated=9.37 GB
Processed 350/865 | GPU allocated=9.37 GB
Processed 400/865 | GPU allocated=9.37 GB
Processed 450/865 | GPU allocated=9.37 GB
Processed 500/865 | GPU allocated=9.37 GB
Processed 550/865 | GPU allocated=9.37 GB
Processed 600/865 | GPU allocated=9.37 GB
Proc

In [3]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 7B
#
# REFINED HARD-MINED DATASET
#
# Uses existing:
# /content/AI_Y2_MODULE2/reports/train_hard_mining.csv
#
# NO new inference.
#
# Selection:
# - missed lesion
# - false lesion
# - lesion IoU < 0.50
# - processing-area IoU < 0.50
#
# Poor containment is NOT used as a duplication criterion.
# ============================================================

from pathlib import Path
import shutil
import json
import pandas as pd


# ============================================================
# PATHS
# ============================================================

ORIGINAL_ROOT = Path(
    "/content/AI_Y2_FINAL"
)

TRAIN_IMG_DIR = (
    ORIGINAL_ROOT /
    "train" /
    "images"
)

TRAIN_LBL_DIR = (
    ORIGINAL_ROOT /
    "train" /
    "labels"
)

HARD_REPORT = Path(
    "/content/AI_Y2_MODULE2/reports/train_hard_mining.csv"
)

REFINED_ROOT = Path(
    "/content/AI_Y2_REFINED_HARDMINED"
)

REPORT_DIR = Path(
    "/content/AI_Y2_MODULE2/reports"
)

REFINED_CSV = (
    REPORT_DIR /
    "refined_hard_train_selection.csv"
)

REFINED_JSON = (
    REPORT_DIR /
    "refined_hard_train_summary.json"
)


# ============================================================
# LOAD
# ============================================================

assert HARD_REPORT.exists(), HARD_REPORT

df = pd.read_csv(
    HARD_REPORT
)


# ============================================================
# GENUINE HARD CASES
# ============================================================

df["genuine_hard"] = (

    df["missed_lesion"].astype(bool)

    |

    df["false_lesion"].astype(bool)

    |

    (
        df["gt_has_lesion"].astype(bool)
        &
        (df["lesion_iou"] < 0.50)
    )

    |

    (
        df["pa_iou"] < 0.50
    )
)


selected = df[
    df["genuine_hard"] == True
].copy()


# ============================================================
# REFINED SEVERITY SCORE
# ============================================================

def score(row):

    s = 0.0

    if bool(row["missed_lesion"]):
        s += 100.0

    if bool(row["false_lesion"]):
        s += 80.0

    if bool(row["gt_has_lesion"]):

        liou = row["lesion_iou"]

        if pd.notna(liou):

            if liou < 0.25:
                s += 60.0

            elif liou < 0.50:
                s += 40.0

            else:
                s += 10.0

    pa = row["pa_iou"]

    if pd.notna(pa):

        if pa < 0.25:
            s += 35.0

        elif pa < 0.50:
            s += 20.0

    return s


selected["refined_score"] = selected.apply(
    score,
    axis=1
)


# ============================================================
# SOURCE-AWARE CAP
#
# Prevent one source from dominating.
# Keep strongest cases per source.
# ============================================================

SOURCE_CAPS = {

    "cancer": 80,

    "leukoplakia": 100,

    "leukoplakia_like": 100,

    "ulcer": 80,

    "processing_area": 60,
}


final_parts = []


for source, group in selected.groupby(
    "source"
):

    group = group.sort_values(
        "refined_score",
        ascending=False
    )

    cap = SOURCE_CAPS.get(
        source,
        len(group)
    )

    final_parts.append(
        group.head(cap)
    )


final_sel = pd.concat(
    final_parts,
    ignore_index=True
)


final_sel = final_sel.sort_values(
    "refined_score",
    ascending=False
).reset_index(
    drop=True
)


# ============================================================
# SAVE SELECTION
# ============================================================

final_sel.to_csv(
    REFINED_CSV,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print("=" * 100)
print("AI_Y2 MODULE 2 — CELL 7B")
print("REFINED HARD-SELECTION")
print("=" * 100)

print(
    f"Original mining rows     : {len(df)}"
)

print(
    f"Genuine hard candidates  : {len(selected)}"
)

print(
    f"Final selected after caps: {len(final_sel)}"
)

print("\nSelected by source:")

print(
    final_sel[
        "source"
    ]
    .value_counts()
    .to_string()
)


print("\nFailure composition:")

print(
    f"Missed lesion   : "
    f"{int(final_sel['missed_lesion'].sum())}"
)

print(
    f"False lesion    : "
    f"{int(final_sel['false_lesion'].sum())}"
)

print(
    f"Low lesion IoU : "
    f"{int(final_sel['low_lesion_iou'].sum())}"
)

print(
    f"Low PA IoU     : "
    f"{int(final_sel['low_pa_iou'].sum())}"
)


# ============================================================
# BUILD REFINED DATASET
# ============================================================

if REFINED_ROOT.exists():

    shutil.rmtree(
        REFINED_ROOT
    )


for split in [
    "train",
    "valid",
    "test"
]:

    (
        REFINED_ROOT /
        split /
        "images"
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    (
        REFINED_ROOT /
        split /
        "labels"
    ).mkdir(
        parents=True,
        exist_ok=True
    )


# copy original dataset unchanged
for split in [
    "train",
    "valid",
    "test"
]:

    src_img = (
        ORIGINAL_ROOT /
        split /
        "images"
    )

    src_lbl = (
        ORIGINAL_ROOT /
        split /
        "labels"
    )

    dst_img = (
        REFINED_ROOT /
        split /
        "images"
    )

    dst_lbl = (
        REFINED_ROOT /
        split /
        "labels"
    )


    for p in src_img.iterdir():

        if p.is_file():
            shutil.copy2(
                p,
                dst_img / p.name
            )


    for p in src_lbl.iterdir():

        if p.is_file():
            shutil.copy2(
                p,
                dst_lbl / p.name
            )


# ============================================================
# DUPLICATE REFINED HARD TRAIN CASES ONCE
# ============================================================

boost_count = 0


for _, row in final_sel.iterrows():

    filename = row["filename"]

    src_img = (
        TRAIN_IMG_DIR /
        filename
    )

    src_lbl = (
        TRAIN_LBL_DIR /
        f"{Path(filename).stem}.txt"
    )

    suffix = "__refinedhard1"

    new_img_name = (
        f"{Path(filename).stem}"
        f"{suffix}"
        f"{Path(filename).suffix}"
    )

    new_lbl_name = (
        f"{Path(filename).stem}"
        f"{suffix}.txt"
    )


    shutil.copy2(
        src_img,
        REFINED_ROOT /
        "train" /
        "images" /
        new_img_name
    )

    shutil.copy2(
        src_lbl,
        REFINED_ROOT /
        "train" /
        "labels" /
        new_lbl_name
    )

    boost_count += 1


# ============================================================
# DATA.YAML
# ============================================================

yaml_text = f"""path: {REFINED_ROOT}
train: train/images
val: valid/images
test: test/images
nc: 2
names:
  - processing_area
  - lesion
"""


(
    REFINED_ROOT /
    "data.yaml"
).write_text(
    yaml_text,
    encoding="utf-8"
)


# ============================================================
# VERIFY COUNTS
# ============================================================

counts = {}


for split in [
    "train",
    "valid",
    "test"
]:

    imgs = [
        p
        for p in (
            REFINED_ROOT /
            split /
            "images"
        ).iterdir()
        if p.is_file()
    ]

    lbls = list(
        (
            REFINED_ROOT /
            split /
            "labels"
        ).glob("*.txt")
    )

    counts[split] = (
        len(imgs),
        len(lbls)
    )

    assert len(imgs) == len(lbls)


assert counts["valid"][0] == 173
assert counts["test"][0] == 115

assert (
    counts["train"][0]
    ==
    865 + boost_count
)


# ============================================================
# SAVE SUMMARY
# ============================================================

summary = {

    "original_train":
        865,

    "genuine_hard_candidates":
        int(len(selected)),

    "final_selected":
        int(len(final_sel)),

    "boost_count":
        int(boost_count),

    "final_train":
        int(counts["train"][0]),

    "valid":
        173,

    "test":
        115,

    "selected_by_source":
        {
            str(k): int(v)
            for k, v in
            final_sel[
                "source"
            ].value_counts().items()
        },

    "dataset":
        str(
            REFINED_ROOT
        ),
}


with open(
    REFINED_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("CELL 7B FINAL SUMMARY")
print("=" * 100)

print(f"""
Dataset:
{REFINED_ROOT}

Original train:
865

Refined hard selected:
{len(final_sel)}

Final train:
{counts['train'][0]}

Validation:
173 — unchanged

Test:
115 — unchanged

Classes:
0 = processing_area
1 = lesion

Selection CSV:
{REFINED_CSV}

Summary JSON:
{REFINED_JSON}

✅ containment NOT used for oversampling
✅ validation untouched
✅ test untouched
✅ Y1 class contract preserved
""")

print("=" * 100)
print("✅ CELL 7B COMPLETE")
print("=" * 100)

AI_Y2 MODULE 2 — CELL 7B
REFINED HARD-SELECTION
Original mining rows     : 865
Genuine hard candidates  : 219
Final selected after caps: 204

Selected by source:
source
leukoplakia_like    100
cancer               62
leukoplakia          39
ulcer                 3

Failure composition:
Missed lesion   : 1
False lesion    : 0
Low lesion IoU : 5
Low PA IoU     : 200

CELL 7B FINAL SUMMARY

Dataset:
/content/AI_Y2_REFINED_HARDMINED

Original train:
865

Refined hard selected:
204

Final train:
1069

Validation:
173 — unchanged

Test:
115 — unchanged

Classes:
0 = processing_area
1 = lesion

Selection CSV:
/content/AI_Y2_MODULE2/reports/refined_hard_train_selection.csv

Summary JSON:
/content/AI_Y2_MODULE2/reports/refined_hard_train_summary.json

✅ containment NOT used for oversampling
✅ validation untouched
✅ test untouched
✅ Y1 class contract preserved

✅ CELL 7B COMPLETE


In [4]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 8
#
# BOUNDARY-FOCUSED REFINEMENT TRAINING
#
# START:
#   /content/ai_y2.pt
#
# TRAIN:
#   /content/AI_Y2_REFINED_HARDMINED/train
#
# VALIDATION:
#   same original 173 validation images
#
# TEST:
#   NOT USED
#
# IMPORTANT:
#   Current /content/ai_y2.pt is NOT overwritten.
#   This cell creates a new candidate only.
# ============================================================

from pathlib import Path
import gc
import json
import hashlib
import shutil

import torch
from ultralytics import YOLO


# ============================================================
# PATHS
# ============================================================

START_MODEL = Path(
    "/content/ai_y2.pt"
)

DATA_ROOT = Path(
    "/content/AI_Y2_REFINED_HARDMINED"
)

DATA_YAML = (
    DATA_ROOT /
    "data.yaml"
)

RUN_ROOT = Path(
    "/content/AI_Y2_MODULE2/runs"
)

REPORT_DIR = Path(
    "/content/AI_Y2_MODULE2/reports"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CANDIDATE_OUT = Path(
    "/content/AI_Y2_MODULE2/"
    "candidate_y2_refined_mask1.pt"
)

REPORT_JSON = (
    REPORT_DIR /
    "AI_Y2_refined_mask1_validation.json"
)


# ============================================================
# CONTRACT
# ============================================================

EXPECTED_NAMES = {
    0: "processing_area",
    1: "lesion"
}


# ============================================================
# CHECKS
# ============================================================

assert START_MODEL.exists(), START_MODEL
assert DATA_YAML.exists(), DATA_YAML

for split, expected in [
    ("train", 1069),
    ("valid", 173),
    ("test", 115),
]:

    img_dir = (
        DATA_ROOT /
        split /
        "images"
    )

    lbl_dir = (
        DATA_ROOT /
        split /
        "labels"
    )

    n_img = len([
        p for p in img_dir.iterdir()
        if p.is_file()
    ])

    n_lbl = len(
        list(
            lbl_dir.glob("*.txt")
        )
    )

    print(
        f"{split:<5}: "
        f"images={n_img} labels={n_lbl}"
    )

    assert n_img == expected
    assert n_lbl == expected


# ============================================================
# GPU CLEANUP
# ============================================================

gc.collect()

torch.cuda.empty_cache()

if torch.cuda.is_available():
    torch.cuda.ipc_collect()

print("\n" + "=" * 100)
print("AI_Y2 MODULE 2 — CELL 8")
print("BOUNDARY-FOCUSED REFINEMENT")
print("=" * 100)

print(
    f"GPU       : "
    f"{torch.cuda.get_device_name(0)}"
)

print(
    f"Allocated : "
    f"{torch.cuda.memory_allocated()/1024**3:.3f} GB"
)

print(
    f"Reserved  : "
    f"{torch.cuda.memory_reserved()/1024**3:.3f} GB"
)


# ============================================================
# VERIFY STARTING MODEL
# ============================================================

check_model = YOLO(
    str(START_MODEL)
)

names = {
    int(k): str(v)
    for k, v in check_model.names.items()
}

assert names == EXPECTED_NAMES
assert str(check_model.task) == "segment"

print(f"\nStart model : {START_MODEL}")
print(f"Task        : {check_model.task}")
print(f"Classes     : {names}")

del check_model

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# TRAIN FUNCTION
# ============================================================

def run_refinement(batch_size):

    run_name = (
        f"AI_Y2_phase3_refined_mask1_b{batch_size}"
    )

    run_dir = (
        RUN_ROOT /
        run_name
    )

    # prevent accidental reuse of an old partial run
    if run_dir.exists():

        shutil.rmtree(
            run_dir
        )

    print("\n" + "=" * 100)

    print(
        f"STARTING REFINEMENT TRAINING "
        f"(batch={batch_size})"
    )

    print("=" * 100)


    model = YOLO(
        str(START_MODEL)
    )


    model.train(

        # ----------------------------------------
        # DATA / HARDWARE
        # ----------------------------------------
        data=str(DATA_YAML),

        imgsz=640,

        batch=batch_size,

        device=0,

        workers=8,

        cache="ram",

        amp=True,


        # ----------------------------------------
        # TRAINING LENGTH
        # ----------------------------------------
        epochs=80,

        patience=18,


        # ----------------------------------------
        # OPTIMIZER
        # ----------------------------------------
        optimizer="AdamW",

        lr0=0.00005,

        lrf=0.10,

        weight_decay=0.0005,

        cos_lr=True,

        warmup_epochs=1.5,


        # ----------------------------------------
        # MASK REFINEMENT
        #
        # KEY CHANGE:
        # mask_ratio=1 instead of 2
        # ----------------------------------------
        mask_ratio=1,

        overlap_mask=True,


        # ----------------------------------------
        # VERY MILD AUGMENTATION
        # ----------------------------------------
        degrees=2.0,

        translate=0.01,

        scale=0.03,

        shear=0.0,

        perspective=0.0,

        flipud=0.0,

        fliplr=0.50,

        hsv_h=0.004,

        hsv_s=0.08,

        hsv_v=0.08,


        # ----------------------------------------
        # NO STRONG COMPOSITE AUGMENTATION
        # ----------------------------------------
        mosaic=0.0,

        close_mosaic=0,

        mixup=0.0,

        copy_paste=0.0,


        # ----------------------------------------
        # REPRODUCIBILITY
        # ----------------------------------------
        seed=42,

        deterministic=True,


        # ----------------------------------------
        # OUTPUT
        # ----------------------------------------
        project=str(RUN_ROOT),

        name=run_name,

        exist_ok=True,

        plots=False,

        save=True,

        save_period=-1,

        verbose=True
    )


    best_path = (
        run_dir /
        "weights" /
        "best.pt"
    )


    if not best_path.exists():

        raise RuntimeError(
            f"❌ best.pt not found:\n{best_path}"
        )


    del model

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


    return best_path, run_dir


# ============================================================
# TRAIN
#
# Start with batch=4.
# If mask_ratio=1 causes CUDA OOM,
# automatically retry cleanly at batch=2.
# ============================================================

used_batch = None


try:

    best_path, run_dir = (
        run_refinement(
            batch_size=4
        )
    )

    used_batch = 4


except torch.OutOfMemoryError:

    print("\n" + "!" * 100)

    print(
        "CUDA OOM at batch=4."
    )

    print(
        "Cleaning GPU and retrying "
        "from ORIGINAL ai_y2.pt with batch=2..."
    )

    print("!" * 100)

    gc.collect()

    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


    best_path, run_dir = (
        run_refinement(
            batch_size=2
        )
    )

    used_batch = 2


except RuntimeError as e:

    if (
        "out of memory"
        in str(e).lower()
    ):

        print("\n" + "!" * 100)

        print(
            "CUDA OOM at batch=4."
        )

        print(
            "Retrying cleanly with batch=2..."
        )

        print("!" * 100)

        gc.collect()

        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


        best_path, run_dir = (
            run_refinement(
                batch_size=2
            )
        )

        used_batch = 2

    else:

        raise


# ============================================================
# COPY BEST TO STABLE CANDIDATE PATH
# ============================================================

shutil.copy2(
    best_path,
    CANDIDATE_OUT
)


print("\n" + "=" * 100)
print("TRAINING COMPLETE")
print("=" * 100)

print(f"Used batch : {used_batch}")
print(f"Run dir    : {run_dir}")
print(f"Best       : {best_path}")
print(f"Candidate  : {CANDIDATE_OUT}")


# ============================================================
# VALIDATION ONLY
#
# IMPORTANT:
# same 173-image validation split.
# Test is untouched.
# ============================================================

gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()


candidate = YOLO(
    str(CANDIDATE_OUT)
)


candidate_names = {
    int(k): str(v)
    for k, v in candidate.names.items()
}

assert candidate_names == EXPECTED_NAMES
assert str(candidate.task) == "segment"


print("\n" + "=" * 100)
print("VALIDATING REFINED CANDIDATE")
print("=" * 100)

print("""
Split:
VALIDATION

Images:
173

conf:
0.25

iou:
0.70

imgsz:
640

TEST:
NOT USED
""")


metrics = candidate.val(

    data=str(DATA_YAML),

    split="val",

    imgsz=640,

    conf=0.25,

    iou=0.70,

    batch=4,

    device=0,

    plots=False,

    save_json=False,

    verbose=False
)


# ============================================================
# EXTRACT METRICS
# ============================================================

report = {

    "candidate":
        str(CANDIDATE_OUT),

    "starting_model":
        str(START_MODEL),

    "batch_used":
        int(used_batch),

    "mask_ratio":
        1,

    "validation_images":
        173,

    "test_used":
        False,


    # ----------------------------------------
    # MASK
    # ----------------------------------------
    "mask_precision":
        float(metrics.seg.mp),

    "mask_recall":
        float(metrics.seg.mr),

    "mask_map50":
        float(metrics.seg.map50),

    "mask_map50_95":
        float(metrics.seg.map),


    # ----------------------------------------
    # BOX
    # ----------------------------------------
    "box_precision":
        float(metrics.box.mp),

    "box_recall":
        float(metrics.box.mr),

    "box_map50":
        float(metrics.box.map50),

    "box_map50_95":
        float(metrics.box.map),
}


# Per-class mask mAP50-95
seg_maps = list(
    metrics.seg.maps
)

if len(seg_maps) >= 2:

    report[
        "processing_mask_map50_95"
    ] = float(
        seg_maps[0]
    )

    report[
        "lesion_mask_map50_95"
    ] = float(
        seg_maps[1]
    )


# Per-class box mAP50-95
box_maps = list(
    metrics.box.maps
)

if len(box_maps) >= 2:

    report[
        "processing_box_map50_95"
    ] = float(
        box_maps[0]
    )

    report[
        "lesion_box_map50_95"
    ] = float(
        box_maps[1]
    )


# ============================================================
# SHA256
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


report[
    "candidate_sha256"
] = sha256_file(
    CANDIDATE_OUT
)


# ============================================================
# SAVE REPORT
# ============================================================

with open(
    REPORT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2
    )


# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 100)
print("REFINED CANDIDATE — VALIDATION METRICS")
print("=" * 100)


for k, v in report.items():

    if isinstance(
        v,
        float
    ):

        print(
            f"{k:<32}: {v:.6f}"
        )

    else:

        print(
            f"{k:<32}: {v}"
        )


# ============================================================
# CLEANUP
# ============================================================

del candidate
del metrics

gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()


print("\n" + "=" * 100)
print("CELL 8 COMPLETE")
print("=" * 100)

print(f"""
Refined candidate:
{CANDIDATE_OUT}

Validation report:
{REPORT_JSON}

Current production candidate:
 /content/ai_y2.pt

IMPORTANT:
The old ai_y2.pt has NOT been overwritten.

TEST SET:
NOT USED.

Next step:
Direct validation comparison:
CURRENT Y2
vs
REFINED mask_ratio=1 Y2

Only after that comparison will we decide
whether the refined candidate deserves to
become the new ai_y2.pt.
""")

print("=" * 100)
print("✅ MODULE 2 CELL 8 COMPLETE")
print("=" * 100)

train: images=1069 labels=1069
valid: images=173 labels=173
test : images=115 labels=115

AI_Y2 MODULE 2 — CELL 8
BOUNDARY-FOCUSED REFINEMENT
GPU       : NVIDIA A100-SXM4-80GB
Allocated : 0.031 GB
Reserved  : 0.990 GB

Start model : /content/ai_y2.pt
Task        : segment
Classes     : {0: 'processing_area', 1: 'lesion'}

STARTING REFINEMENT TRAINING (batch=4)
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/AI_Y2_REFINED_HARDMINED/data.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs

In [5]:

# ============================================================
# AI_Y2 — MODULE 2 / CELL 9
#
# TRUE OOF FOLD CONSTRUCTION
#
# IMPORTANT:
# - Uses ONLY original TRAIN split (865 images)
# - Validation 173 NOT touched
# - Test 115 NOT touched
#
# Creates 5 source-stratified, lineage-group-safe folds.
#
# Fold models in next cell MUST start from:
# /content/yolo26m-seg.pt
#
# NOT from /content/ai_y2.pt
# ============================================================

from pathlib import Path
import re
import json
import shutil

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold


# ============================================================
# PATHS
# ============================================================

DATA_ROOT = Path(
    "/content/AI_Y2_FINAL"
)

TRAIN_IMG_DIR = (
    DATA_ROOT /
    "train" /
    "images"
)

TRAIN_LBL_DIR = (
    DATA_ROOT /
    "train" /
    "labels"
)

MANIFEST_PATH = (
    DATA_ROOT /
    "dataset_manifest.csv"
)

BASE_MODEL = Path(
    "/content/yolo26m-seg.pt"
)

OOF_ROOT = Path(
    "/content/AI_Y2_OOF_FOLDS"
)

REPORT_DIR = Path(
    "/content/AI_Y2_MODULE2/reports"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ASSIGNMENT_CSV = (
    REPORT_DIR /
    "AI_Y2_oof_fold_assignments.csv"
)

SUMMARY_JSON = (
    REPORT_DIR /
    "AI_Y2_oof_fold_summary.json"
)


# ============================================================
# CHECKS
# ============================================================

assert DATA_ROOT.exists()
assert TRAIN_IMG_DIR.exists()
assert TRAIN_LBL_DIR.exists()
assert MANIFEST_PATH.exists()
assert BASE_MODEL.exists()


# ============================================================
# CLEAN OLD FOLD DEFINITIONS
# ============================================================

if OOF_ROOT.exists():

    shutil.rmtree(
        OOF_ROOT
    )

OOF_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# LOAD MANIFEST
# ============================================================

manifest = pd.read_csv(
    MANIFEST_PATH
)

train_manifest = manifest[
    manifest["final_split"] == "train"
].copy()


assert len(train_manifest) == 865


# ============================================================
# HELPERS
# ============================================================

def extract_group_id(filename):

    """
    Final Y2 filenames contain:
    __Y2Gxxxxx__

    This is the lineage/group identifier created during
    cleaned dataset construction.

    Images from same Y2G group MUST remain in same OOF fold.
    """

    m = re.search(
        r"__Y2G(\d+)__",
        filename
    )

    if not m:

        raise RuntimeError(
            f"Could not extract Y2 lineage group from:\n"
            f"{filename}"
        )

    return f"Y2G{m.group(1)}"


def label_has_lesion(label_path):

    text = label_path.read_text(
        encoding="utf-8"
    ).strip()

    if not text:
        return False

    for line in text.splitlines():

        parts = line.split()

        if int(parts[0]) == 1:
            return True

    return False


# ============================================================
# BUILD TRAIN TABLE
# ============================================================

rows = []


for _, row in train_manifest.iterrows():

    filename = row["final_image"]

    img_path = (
        TRAIN_IMG_DIR /
        filename
    )

    lbl_path = (
        TRAIN_LBL_DIR /
        f"{Path(filename).stem}.txt"
    )


    assert img_path.exists(), img_path
    assert lbl_path.exists(), lbl_path


    source = str(
        row["source_category"]
    )


    group_id = extract_group_id(
        filename
    )


    has_lesion = label_has_lesion(
        lbl_path
    )


    rows.append({

        "filename":
            filename,

        "image_path":
            str(img_path),

        "label_path":
            str(lbl_path),

        "source":
            source,

        "group_id":
            group_id,

        "has_lesion":
            bool(has_lesion),
    })


df = pd.DataFrame(
    rows
)


assert len(df) == 865


# ============================================================
# STRATIFICATION LABEL
#
# source + lesion presence
#
# e.g.
# cancer|1
# leukoplakia|1
# processing_area|0
# ============================================================

df[
    "stratum"
] = (
    df["source"].astype(str)
    +
    "|lesion="
    +
    df["has_lesion"].astype(int).astype(str)
)


# ============================================================
# BASIC AUDIT
# ============================================================

print("=" * 100)
print("AI_Y2 MODULE 2 — CELL 9")
print("TRUE OOF FOLD CONSTRUCTION")
print("=" * 100)


print(
    f"Train images       : {len(df)}"
)

print(
    f"Unique lineage IDs : {df['group_id'].nunique()}"
)

print(
    f"Lesion positive    : {int(df['has_lesion'].sum())}"
)

print(
    f"Processing-only    : {int((~df['has_lesion']).sum())}"
)


print("\nSource distribution:")

print(
    df[
        "source"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# VERIFY EACH LINEAGE GROUP HAS SINGLE SOURCE
# ============================================================

group_source_counts = (
    df.groupby(
        "group_id"
    )[
        "source"
    ]
    .nunique()
)


bad_group_sources = (
    group_source_counts[
        group_source_counts > 1
    ]
)


if len(bad_group_sources):

    raise RuntimeError(
        "❌ Some lineage groups contain "
        "multiple source categories."
    )


# ============================================================
# 5-FOLD STRATIFIED GROUP SPLIT
# ============================================================

N_SPLITS = 5

splitter = StratifiedGroupKFold(

    n_splits=N_SPLITS,

    shuffle=True,

    random_state=42
)


df[
    "oof_fold"
] = -1


X = np.zeros(
    len(df),
    dtype=np.uint8
)

y = df[
    "stratum"
].values

groups = df[
    "group_id"
].values


for fold_idx, (
    train_idx,
    holdout_idx
) in enumerate(

    splitter.split(
        X,
        y,
        groups
    )

):

    df.loc[
        df.index[
            holdout_idx
        ],
        "oof_fold"
    ] = fold_idx


assert (
    df["oof_fold"] >= 0
).all()


# ============================================================
# GLOBAL LEAKAGE AUDIT
# ============================================================

print("\n" + "=" * 100)
print("OOF FOLD AUDIT")
print("=" * 100)


# Every group must map to exactly one fold
group_fold_counts = (
    df.groupby(
        "group_id"
    )[
        "oof_fold"
    ]
    .nunique()
)


bad_groups = (
    group_fold_counts[
        group_fold_counts != 1
    ]
)


if len(bad_groups):

    raise RuntimeError(
        "❌ LINEAGE LEAKAGE detected "
        "across OOF folds."
    )


print(
    "✅ Every lineage group belongs to exactly one holdout fold."
)


# ============================================================
# CREATE FOLD FILES
#
# We DO NOT copy image data.
#
# Ultralytics can train using image-list .txt files.
# This saves disk and preserves exact original labels.
# ============================================================

fold_summaries = []


for fold in range(
    N_SPLITS
):

    fold_root = (
        OOF_ROOT /
        f"fold_{fold}"
    )

    fold_root.mkdir(
        parents=True,
        exist_ok=True
    )


    holdout_df = df[
        df["oof_fold"] == fold
    ].copy()


    fold_train_df = df[
        df["oof_fold"] != fold
    ].copy()


    # --------------------------------------------
    # Group leakage verification
    # --------------------------------------------

    train_groups = set(
        fold_train_df[
            "group_id"
        ]
    )

    holdout_groups = set(
        holdout_df[
            "group_id"
        ]
    )


    overlap = (
        train_groups
        &
        holdout_groups
    )


    assert len(overlap) == 0


    # --------------------------------------------
    # WRITE IMAGE LISTS
    # --------------------------------------------

    train_txt = (
        fold_root /
        "train.txt"
    )

    holdout_txt = (
        fold_root /
        "holdout.txt"
    )


    train_txt.write_text(
        "\n".join(
            fold_train_df[
                "image_path"
            ].tolist()
        )
        + "\n",
        encoding="utf-8"
    )


    holdout_txt.write_text(
        "\n".join(
            holdout_df[
                "image_path"
            ].tolist()
        )
        + "\n",
        encoding="utf-8"
    )


    # --------------------------------------------
    # DATA YAML
    # --------------------------------------------

    data_yaml = f"""path: {DATA_ROOT}
train: {train_txt}
val: {holdout_txt}
nc: 2
names:
  - processing_area
  - lesion
"""


    (
        fold_root /
        "data.yaml"
    ).write_text(
        data_yaml,
        encoding="utf-8"
    )


    # --------------------------------------------
    # FOLD SUMMARY
    # --------------------------------------------

    summary = {

        "fold":
            int(fold),

        "train_images":
            int(
                len(
                    fold_train_df
                )
            ),

        "holdout_images":
            int(
                len(
                    holdout_df
                )
            ),

        "train_groups":
            int(
                fold_train_df[
                    "group_id"
                ].nunique()
            ),

        "holdout_groups":
            int(
                holdout_df[
                    "group_id"
                ].nunique()
            ),

        "group_overlap":
            int(
                len(overlap)
            ),

        "holdout_lesion_positive":
            int(
                holdout_df[
                    "has_lesion"
                ].sum()
            ),

        "holdout_processing_only":
            int(
                (
                    ~holdout_df[
                        "has_lesion"
                    ]
                ).sum()
            ),
    }


    fold_summaries.append(
        summary
    )


    print("\n" + "-" * 100)

    print(
        f"FOLD {fold}"
    )

    print(
        f"train   : {len(fold_train_df)}"
    )

    print(
        f"holdout : {len(holdout_df)}"
    )

    print(
        f"groups  : "
        f"{fold_train_df['group_id'].nunique()} train / "
        f"{holdout_df['group_id'].nunique()} holdout"
    )

    print(
        f"leakage : {len(overlap)}"
    )


    print(
        "\nHoldout source distribution:"
    )

    print(
        holdout_df[
            "source"
        ]
        .value_counts()
        .sort_index()
        .to_string()
    )


# ============================================================
# VERIFY EACH IMAGE APPEARS EXACTLY ONCE AS OOF HOLDOUT
# ============================================================

fold_counts = (
    df[
        "oof_fold"
    ]
    .value_counts()
    .sort_index()
)


assert (
    fold_counts.sum()
    ==
    865
)


assert (
    df[
        "filename"
    ].nunique()
    ==
    865
)


# ============================================================
# SAVE ASSIGNMENT TABLE
# ============================================================

df.sort_values(
    [
        "oof_fold",
        "source",
        "group_id",
        "filename",
    ]
).to_csv(
    ASSIGNMENT_CSV,
    index=False
)


# ============================================================
# SUMMARY JSON
# ============================================================

summary_json = {

    "method":
        "5-fold StratifiedGroupKFold",

    "seed":
        42,

    "base_model_required":
        str(
            BASE_MODEL
        ),

    "current_y2_allowed_as_fold_start":
        False,

    "total_train_images":
        865,

    "unique_lineage_groups":
        int(
            df[
                "group_id"
            ].nunique()
        ),

    "validation_used":
        False,

    "test_used":
        False,

    "folds":
        fold_summaries,
}


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary_json,
        f,
        indent=2
    )


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("CELL 9 FINAL SUMMARY")
print("=" * 100)


print(
    f"Total OOF images       : {len(df)}"
)

print(
    f"Unique lineage groups  : "
    f"{df['group_id'].nunique()}"
)

print(
    f"Number of folds        : {N_SPLITS}"
)

print(
    f"Fold sizes             : "
    f"{fold_counts.to_dict()}"
)


print(f"""
Fold definitions:
{OOF_ROOT}

Assignments:
{ASSIGNMENT_CSV}

Summary:
{SUMMARY_JSON}

FOLD TRAINING START MODEL:
{BASE_MODEL}

DO NOT START FOLD MODELS FROM:
/content/ai_y2.pt

Why:
Current ai_y2.pt has already seen all 865 training images,
so it cannot produce genuine out-of-fold predictions.

Validation 173:
NOT USED

Test 115:
NOT USED

✅ 5-fold source-stratified
✅ lineage-group-safe
✅ no validation leakage
✅ no test leakage
✅ ready for true OOF training
""")


print("=" * 100)
print("✅ MODULE 2 CELL 9 COMPLETE")
print("=" * 100)

AI_Y2 MODULE 2 — CELL 9
TRUE OOF FOLD CONSTRUCTION
Train images       : 865
Unique lineage IDs : 845
Lesion positive    : 667
Processing-only    : 198

Source distribution:
source
processing_area     198
leukoplakia_like    187
leukoplakia         184
cancer              150
ulcer               146

OOF FOLD AUDIT
✅ Every lineage group belongs to exactly one holdout fold.

----------------------------------------------------------------------------------------------------
FOLD 0
train   : 692
holdout : 173
groups  : 676 train / 169 holdout
leakage : 0

Holdout source distribution:
source
cancer              31
leukoplakia         37
leukoplakia_like    38
processing_area     39
ulcer               28

----------------------------------------------------------------------------------------------------
FOLD 1
train   : 694
holdout : 171
groups  : 676 train / 169 holdout
leakage : 0

Holdout source distribution:
source
cancer              31
leukoplakia         37
leukoplakia_like    34
p

In [7]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 10
#
# TRUE 5-FOLD OOF MODEL TRAINING
#
# Each fold:
#   train = ~80% of ORIGINAL TRAIN
#   val   = remaining ~20% ORIGINAL TRAIN
#
# IMPORTANT:
# - Every fold starts independently from yolo26m-seg.pt
# - Current ai_y2.pt is NEVER used as starting weights
# - Original VALIDATION 173 is NOT used
# - Original TEST 115 is NOT used
#
# Output:
# /content/AI_Y2_MODULE2/oof_models/fold_X_best.pt
# ============================================================

from pathlib import Path
import gc
import json
import shutil
import hashlib
import time

import torch
from ultralytics import YOLO


# ============================================================
# PATHS
# ============================================================

BASE_MODEL = Path(
    "/content/yolo26m-seg.pt"
)

OOF_ROOT = Path(
    "/content/AI_Y2_OOF_FOLDS"
)

RUN_ROOT = Path(
    "/content/AI_Y2_MODULE2/oof_runs"
)

MODEL_OUT_ROOT = Path(
    "/content/AI_Y2_MODULE2/oof_models"
)

REPORT_DIR = Path(
    "/content/AI_Y2_MODULE2/reports"
)

REPORT_JSON = (
    REPORT_DIR /
    "AI_Y2_oof_training_summary.json"
)


# ============================================================
# CONFIG
# ============================================================

N_FOLDS = 5

EXPECTED_NAMES = {
    0: "processing_area",
    1: "lesion",
}


# ============================================================
# CHECKS
# ============================================================

assert BASE_MODEL.exists(), BASE_MODEL
assert OOF_ROOT.exists(), OOF_ROOT

RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def cleanup_gpu():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# ============================================================
# GPU
# ============================================================

print("=" * 100)
print("AI_Y2 MODULE 2 — CELL 10")
print("TRUE 5-FOLD OOF TRAINING")
print("=" * 100)

if not torch.cuda.is_available():
    raise RuntimeError("❌ CUDA GPU unavailable.")

print(
    f"GPU       : "
    f"{torch.cuda.get_device_name(0)}"
)

print(
    f"Allocated : "
    f"{torch.cuda.memory_allocated()/1024**3:.3f} GB"
)

print(
    f"Reserved  : "
    f"{torch.cuda.memory_reserved()/1024**3:.3f} GB"
)


# ============================================================
# VERIFY GENERIC PRETRAINED BASE MODEL
#
# IMPORTANT:
# yolo26m-seg.pt is a GENERIC segmentation checkpoint.
# It is NOT expected to already have OralScan class names.
#
# OralScan's 2-class head is configured from each fold's
# data.yaml during training.
# ============================================================

base_check = YOLO(
    str(BASE_MODEL)
)

assert str(base_check.task) == "segment", (
    f"❌ Expected segmentation checkpoint, got: "
    f"{base_check.task}"
)

base_names = {
    int(k): str(v)
    for k, v in base_check.names.items()
}

print(f"\nBase model : {BASE_MODEL}")
print(f"Task       : {base_check.task}")
print(
    f"Base classes before OralScan training: "
    f"{len(base_names)}"
)

print(
    "✅ Generic pretrained segmentation checkpoint accepted."
)

print(
    "✅ OralScan nc=2 + class names will come from fold data.yaml."
)

del base_check

cleanup_gpu()


# ============================================================
# TRAIN EACH FOLD
# ============================================================

fold_reports = []

global_start = time.time()


for fold in range(
    N_FOLDS
):

    print("\n\n" + "#" * 100)
    print(f"# OOF FOLD {fold}")
    print("#" * 100)


    fold_root = (
        OOF_ROOT /
        f"fold_{fold}"
    )

    data_yaml = (
        fold_root /
        "data.yaml"
    )

    train_txt = (
        fold_root /
        "train.txt"
    )

    holdout_txt = (
        fold_root /
        "holdout.txt"
    )


    assert data_yaml.exists(), data_yaml
    assert train_txt.exists(), train_txt
    assert holdout_txt.exists(), holdout_txt


    train_count = len([
        x
        for x in train_txt.read_text(
            encoding="utf-8"
        ).splitlines()
        if x.strip()
    ])

    holdout_count = len([
        x
        for x in holdout_txt.read_text(
            encoding="utf-8"
        ).splitlines()
        if x.strip()
    ])


    print(
        f"Fold train   : {train_count}"
    )

    print(
        f"Fold holdout : {holdout_count}"
    )


    # --------------------------------------------------------
    # Stable output paths
    # --------------------------------------------------------

    run_name = (
        f"AI_Y2_OOF_fold_{fold}"
    )

    run_dir = (
        RUN_ROOT /
        run_name
    )

    stable_best = (
        MODEL_OUT_ROOT /
        f"fold_{fold}_best.pt"
    )


    # --------------------------------------------------------
    # RESUME-SAFE:
    # If stable best already exists and contract is valid,
    # skip retraining that fold.
    # --------------------------------------------------------

    if stable_best.exists():

        print(
            f"\nExisting model found:\n"
            f"{stable_best}"
        )

        check = YOLO(
            str(stable_best)
        )

        names = {
            int(k): str(v)
            for k, v in check.names.items()
        }

        if (
            names == EXPECTED_NAMES
            and
            str(check.task) == "segment"
        ):

            print(
                "✅ Existing fold model contract valid."
            )

            print(
                "Skipping retraining for this fold."
            )

            fold_reports.append({

                "fold":
                    int(fold),

                "train_images":
                    int(train_count),

                "holdout_images":
                    int(holdout_count),

                "model":
                    str(stable_best),

                "sha256":
                    sha256_file(
                        stable_best
                    ),

                "status":
                    "existing_valid_model",
            })

            del check

            cleanup_gpu()

            continue

        else:

            del check

            print(
                "Existing fold model invalid — retraining."
            )

            stable_best.unlink()


    # --------------------------------------------------------
    # Remove partial/old run
    # --------------------------------------------------------

    if run_dir.exists():

        shutil.rmtree(
            run_dir
        )


    cleanup_gpu()


    # --------------------------------------------------------
    # LOAD FRESH PRETRAINED BASE
    # --------------------------------------------------------

    model = YOLO(
        str(BASE_MODEL)
    )


    fold_start = time.time()


    # ========================================================
    # TRAIN
    #
    # OOF purpose:
    # credible unseen predictions, not final production model.
    # ========================================================

    model.train(

        # ----------------------------------------
        # DATA
        # ----------------------------------------
        data=str(
            data_yaml
        ),

        imgsz=640,

        batch=16,

        device=0,

        workers=8,

        cache="ram",

        amp=True,


        # ----------------------------------------
        # TRAINING LENGTH
        # ----------------------------------------
        epochs=180,

        patience=35,


        # ----------------------------------------
        # OPTIMIZER
        # ----------------------------------------
        optimizer="AdamW",

        lr0=0.0004,

        lrf=0.01,

        weight_decay=0.0005,

        cos_lr=True,

        warmup_epochs=4.0,


        # ----------------------------------------
        # SEGMENTATION
        # ----------------------------------------
        mask_ratio=2,

        overlap_mask=True,


        # ----------------------------------------
        # AUGMENTATION
        # ----------------------------------------
        degrees=8.0,

        translate=0.04,

        scale=0.12,

        shear=0.0,

        perspective=0.0,

        flipud=0.0,

        fliplr=0.50,

        hsv_h=0.008,

        hsv_s=0.16,

        hsv_v=0.18,


        # ----------------------------------------
        # LIGHT COMPOSITE AUGMENTATION
        # ----------------------------------------
        mosaic=0.10,

        close_mosaic=25,

        mixup=0.0,

        copy_paste=0.0,


        # ----------------------------------------
        # REPRODUCIBILITY
        # ----------------------------------------
        seed=(
            42 + fold
        ),

        deterministic=True,


        # ----------------------------------------
        # OUTPUT
        # ----------------------------------------
        project=str(
            RUN_ROOT
        ),

        name=run_name,

        exist_ok=True,

        plots=False,

        save=True,

        save_period=-1,

        verbose=True
    )


    # ========================================================
    # BEST MODEL
    # ========================================================

    best_path = (
        run_dir /
        "weights" /
        "best.pt"
    )


    if not best_path.exists():

        raise RuntimeError(
            f"❌ Fold {fold} best.pt missing:\n"
            f"{best_path}"
        )


    shutil.copy2(
        best_path,
        stable_best
    )


    # ========================================================
    # CONTRACT CHECK AFTER ORALSCAN TRAINING
    # ========================================================

    fold_model = YOLO(
        str(stable_best)
    )

    fold_names = {
        int(k): str(v)
        for k, v
        in fold_model.names.items()
    }

    assert fold_names == EXPECTED_NAMES, (
        f"❌ Fold {fold} class contract mismatch:\n"
        f"{fold_names}"
    )

    assert str(
        fold_model.task
    ) == "segment"


    # ========================================================
    # HOLDOUT VALIDATION METRICS
    # ========================================================

    metrics = fold_model.val(

        data=str(
            data_yaml
        ),

        split="val",

        imgsz=640,

        conf=0.25,

        iou=0.70,

        batch=8,

        device=0,

        plots=False,

        save_json=False,

        verbose=False
    )


    fold_minutes = (
        time.time()
        -
        fold_start
    ) / 60.0


    report = {

        "fold":
            int(fold),

        "train_images":
            int(train_count),

        "holdout_images":
            int(holdout_count),

        "training_minutes":
            float(fold_minutes),

        "model":
            str(stable_best),

        "sha256":
            sha256_file(
                stable_best
            ),

        "mask_precision":
            float(
                metrics.seg.mp
            ),

        "mask_recall":
            float(
                metrics.seg.mr
            ),

        "mask_map50":
            float(
                metrics.seg.map50
            ),

        "mask_map50_95":
            float(
                metrics.seg.map
            ),

        "box_precision":
            float(
                metrics.box.mp
            ),

        "box_recall":
            float(
                metrics.box.mr
            ),

        "box_map50":
            float(
                metrics.box.map50
            ),

        "box_map50_95":
            float(
                metrics.box.map
            ),

        "processing_mask_map50_95":
            float(
                metrics.seg.maps[0]
            ),

        "lesion_mask_map50_95":
            float(
                metrics.seg.maps[1]
            ),

        "status":
            "trained",
    }


    fold_reports.append(
        report
    )


    print("\n" + "=" * 100)
    print(f"FOLD {fold} COMPLETE")
    print("=" * 100)

    for k, v in report.items():

        if isinstance(
            v,
            float
        ):

            print(
                f"{k:<30}: "
                f"{v:.6f}"
            )

        else:

            print(
                f"{k:<30}: "
                f"{v}"
            )


    # ========================================================
    # CLEAN BETWEEN FOLDS
    # ========================================================

    del metrics
    del fold_model
    del model

    cleanup_gpu()

    print(
        f"\nGPU after fold cleanup: "
        f"{torch.cuda.memory_allocated()/1024**3:.3f} GB"
    )


# ============================================================
# GLOBAL SUMMARY
# ============================================================

total_minutes = (
    time.time()
    -
    global_start
) / 60.0


summary = {

    "base_model":
        str(
            BASE_MODEL
        ),

    "current_y2_used_as_start":
        False,

    "num_folds":
        N_FOLDS,

    "original_validation_used":
        False,

    "original_test_used":
        False,

    "total_runtime_minutes":
        float(
            total_minutes
        ),

    "folds":
        fold_reports,
}


with open(
    REPORT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ============================================================
# SUMMARY TABLE
# ============================================================

print("\n\n" + "=" * 100)
print("OOF TRAINING — FINAL SUMMARY")
print("=" * 100)


trained_reports = [
    x
    for x in fold_reports
    if "mask_map50_95" in x
]


for r in fold_reports:

    print(
        f"\nFold {r['fold']}"
        f"\n  train    : {r['train_images']}"
        f"\n  holdout  : {r['holdout_images']}"
        f"\n  model    : {r['model']}"
        f"\n  status   : {r['status']}"
    )

    if "mask_map50_95" in r:

        print(
            f"  mask mAP50-95        : "
            f"{r['mask_map50_95']:.4f}"
        )

        print(
            f"  processing mask mAP  : "
            f"{r['processing_mask_map50_95']:.4f}"
        )

        print(
            f"  lesion mask mAP      : "
            f"{r['lesion_mask_map50_95']:.4f}"
        )


if trained_reports:

    mean_mask = sum(
        x["mask_map50_95"]
        for x in trained_reports
    ) / len(
        trained_reports
    )

    mean_pa = sum(
        x[
            "processing_mask_map50_95"
        ]
        for x in trained_reports
    ) / len(
        trained_reports
    )

    mean_lesion = sum(
        x[
            "lesion_mask_map50_95"
        ]
        for x in trained_reports
    ) / len(
        trained_reports
    )


    print("\n" + "-" * 100)

    print(
        f"Mean OOF mask mAP50-95       : "
        f"{mean_mask:.4f}"
    )

    print(
        f"Mean processing mask mAP50-95: "
        f"{mean_pa:.4f}"
    )

    print(
        f"Mean lesion mask mAP50-95    : "
        f"{mean_lesion:.4f}"
    )


print(f"""
Total runtime:
{total_minutes:.2f} minutes

Models:
{MODEL_OUT_ROOT}

Training summary:
{REPORT_JSON}

Original validation 173:
NOT USED

Original test 115:
NOT USED

Current ai_y2.pt:
NOT USED AS FOLD START

NEXT:
Run every image through ONLY its own unseen-fold model
and build true OOF failure/hardness table.
""")


print("=" * 100)
print("✅ MODULE 2 CELL 10 COMPLETE")
print("=" * 100)

AI_Y2 MODULE 2 — CELL 10
TRUE 5-FOLD OOF TRAINING
GPU       : NVIDIA A100-SXM4-80GB
Allocated : 0.062 GB
Reserved  : 1.186 GB

Base model : /content/yolo26m-seg.pt
Task       : segment
Base classes before OralScan training: 80
✅ Generic pretrained segmentation checkpoint accepted.
✅ OralScan nc=2 + class names will come from fold data.yaml.


####################################################################################################
# OOF FOLD 0
####################################################################################################
Fold train   : 692
Fold holdout : 173
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=25, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr

In [8]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 11
#
# TRUE OOF HARD-CASE MINING
#
# Every original TRAIN image is predicted ONLY by the fold
# model that NEVER saw that image during training.
#
# Uses:
#   /content/AI_Y2_MODULE2/oof_models/fold_X_best.pt
#   /content/AI_Y2_MODULE2/reports/AI_Y2_oof_fold_assignments.csv
#
# Validation 173: NOT USED
# Test 115: NOT USED
# ============================================================

from pathlib import Path
import gc
import json

import cv2
import numpy as np
import pandas as pd
import torch

from ultralytics import YOLO


# ============================================================
# PATHS
# ============================================================

DATA_ROOT = Path(
    "/content/AI_Y2_FINAL"
)

TRAIN_IMG_DIR = (
    DATA_ROOT /
    "train" /
    "images"
)

TRAIN_LBL_DIR = (
    DATA_ROOT /
    "train" /
    "labels"
)

ASSIGNMENT_CSV = Path(
    "/content/AI_Y2_MODULE2/reports/"
    "AI_Y2_oof_fold_assignments.csv"
)

MODEL_ROOT = Path(
    "/content/AI_Y2_MODULE2/oof_models"
)

REPORT_DIR = Path(
    "/content/AI_Y2_MODULE2/reports"
)

OOF_FAILURE_CSV = (
    REPORT_DIR /
    "AI_Y2_true_oof_failure_analysis.csv"
)

OOF_HARD_CSV = (
    REPORT_DIR /
    "AI_Y2_true_oof_hard_cases.csv"
)

SUMMARY_JSON = (
    REPORT_DIR /
    "AI_Y2_true_oof_hard_summary.json"
)


# ============================================================
# CONFIG
# ============================================================

N_FOLDS = 5

IMGSZ = 640
CONF = 0.25
IOU = 0.70

EXPECTED_NAMES = {
    0: "processing_area",
    1: "lesion"
}

LESION_IOU_HARD = 0.50
LESION_IOU_SEVERE = 0.25
PA_IOU_HARD = 0.50


# ============================================================
# HELPERS
# ============================================================

def cleanup_gpu():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def polygon_to_mask(points, w, h):

    mask = np.zeros(
        (h, w),
        dtype=np.uint8
    )

    pts = points.copy()

    pts[:, 0] *= w
    pts[:, 1] *= h

    pts = np.round(
        pts
    ).astype(np.int32)

    cv2.fillPoly(
        mask,
        [pts],
        1
    )

    return mask


def parse_gt(label_path, w, h):

    out = {
        0: [],
        1: [],
    }

    text = label_path.read_text(
        encoding="utf-8"
    ).strip()

    for line in text.splitlines():

        parts = line.split()

        cls_id = int(parts[0])

        coords = np.array(
            [float(x) for x in parts[1:]],
            dtype=np.float32
        ).reshape(-1, 2)

        out[cls_id].append(
            polygon_to_mask(
                coords,
                w,
                h
            )
        )

    return out


def union_masks(masks, h, w):

    out = np.zeros(
        (h, w),
        dtype=np.uint8
    )

    for m in masks:

        out = np.maximum(
            out,
            m.astype(np.uint8)
        )

    return out


def resize_mask(mask, w, h):

    resized = cv2.resize(
        mask.astype(np.float32),
        (w, h),
        interpolation=cv2.INTER_NEAREST
    )

    return (
        resized > 0.5
    ).astype(np.uint8)


def mask_iou(a, b):

    inter = np.logical_and(
        a > 0,
        b > 0
    ).sum()

    union = np.logical_or(
        a > 0,
        b > 0
    ).sum()

    if union == 0:
        return 1.0

    return float(
        inter / union
    )


# ============================================================
# LOAD ASSIGNMENTS
# ============================================================

assert ASSIGNMENT_CSV.exists()

assignments = pd.read_csv(
    ASSIGNMENT_CSV
)

assert len(assignments) == 865

print("=" * 100)
print("AI_Y2 MODULE 2 — CELL 11")
print("TRUE OOF HARD-CASE MINING")
print("=" * 100)

print(
    f"OOF assignment rows : {len(assignments)}"
)

print(
    f"Folds               : "
    f"{sorted(assignments['oof_fold'].unique().tolist())}"
)

print("""
Validation used:
NO

Test used:
NO
""")


# ============================================================
# PROCESS EACH FOLD
# ============================================================

all_rows = []


for fold in range(N_FOLDS):

    print("\n" + "#" * 100)
    print(f"# OOF PREDICTION FOLD {fold}")
    print("#" * 100)

    fold_df = assignments[
        assignments["oof_fold"] == fold
    ].copy()

    model_path = (
        MODEL_ROOT /
        f"fold_{fold}_best.pt"
    )

    assert model_path.exists(), model_path

    print(
        f"Holdout images : {len(fold_df)}"
    )

    print(
        f"Model          : {model_path}"
    )

    cleanup_gpu()

    model = YOLO(
        str(model_path)
    )

    names = {
        int(k): str(v)
        for k, v in model.names.items()
    }

    assert names == EXPECTED_NAMES
    assert str(model.task) == "segment"


    image_paths = [
        str(
            TRAIN_IMG_DIR /
            filename
        )
        for filename
        in fold_df["filename"].tolist()
    ]


    prediction_stream = model.predict(

        source=image_paths,

        imgsz=IMGSZ,

        conf=CONF,
        iou=IOU,

        device=0,

        batch=1,

        stream=True,

        retina_masks=False,

        verbose=False,
        save=False
    )


    for (
        (_, row),
        result
    ) in zip(
        fold_df.iterrows(),
        prediction_stream
    ):

        filename = row["filename"]

        img_path = (
            TRAIN_IMG_DIR /
            filename
        )

        lbl_path = (
            TRAIN_LBL_DIR /
            f"{Path(filename).stem}.txt"
        )


        image = cv2.imread(
            str(img_path)
        )

        if image is None:
            raise RuntimeError(
                f"Unreadable image: {img_path}"
            )

        h, w = image.shape[:2]


        # ----------------------------------------
        # GT
        # ----------------------------------------

        gt = parse_gt(
            lbl_path,
            w,
            h
        )

        gt_pa = union_masks(
            gt[0],
            h,
            w
        )

        gt_lesion = union_masks(
            gt[1],
            h,
            w
        )

        gt_has_lesion = (
            len(gt[1]) > 0
        )


        # ----------------------------------------
        # PREDICTIONS
        # ----------------------------------------

        pred_pa_masks = []
        pred_lesion_masks = []

        lesion_confs = []
        pa_confs = []


        if (
            result.boxes is not None
            and
            len(result.boxes) > 0
            and
            result.masks is not None
        ):

            classes = (
                result.boxes.cls
                .detach()
                .cpu()
                .numpy()
                .astype(np.int32)
            )

            confs = (
                result.boxes.conf
                .detach()
                .cpu()
                .numpy()
            )

            masks = (
                result.masks.data
                .detach()
                .float()
                .cpu()
                .numpy()
            )


            for cls_id, conf, raw_mask in zip(
                classes,
                confs,
                masks
            ):

                m = resize_mask(
                    raw_mask,
                    w,
                    h
                )

                if cls_id == 0:

                    pred_pa_masks.append(m)

                    pa_confs.append(
                        float(conf)
                    )

                elif cls_id == 1:

                    pred_lesion_masks.append(m)

                    lesion_confs.append(
                        float(conf)
                    )


        pred_pa = union_masks(
            pred_pa_masks,
            h,
            w
        )

        pred_lesion = union_masks(
            pred_lesion_masks,
            h,
            w
        )


        pred_has_lesion = (
            len(pred_lesion_masks) > 0
        )


        # ----------------------------------------
        # METRICS
        # ----------------------------------------

        pa_iou = mask_iou(
            gt_pa,
            pred_pa
        )


        lesion_iou = (
            mask_iou(
                gt_lesion,
                pred_lesion
            )
            if gt_has_lesion
            else None
        )


        missed_lesion = (
            gt_has_lesion
            and
            not pred_has_lesion
        )


        false_lesion = (
            not gt_has_lesion
            and
            pred_has_lesion
        )


        severe_lesion_iou = (
            gt_has_lesion
            and
            lesion_iou < LESION_IOU_SEVERE
        )


        low_lesion_iou = (
            gt_has_lesion
            and
            lesion_iou < LESION_IOU_HARD
        )


        low_pa_iou = (
            pa_iou < PA_IOU_HARD
        )


        # ----------------------------------------
        # TRUE HARDNESS SCORE
        # ----------------------------------------

        hard_score = 0.0


        if missed_lesion:
            hard_score += 100.0


        if false_lesion:
            hard_score += 80.0


        if gt_has_lesion:

            if lesion_iou < 0.25:
                hard_score += 60.0

            elif lesion_iou < 0.50:
                hard_score += 40.0

            elif lesion_iou < 0.70:
                hard_score += 15.0


        if pa_iou < 0.25:
            hard_score += 35.0

        elif pa_iou < 0.50:
            hard_score += 20.0


        genuine_hard = (
            missed_lesion
            or
            false_lesion
            or
            low_lesion_iou
            or
            low_pa_iou
        )


        all_rows.append({

            "filename":
                filename,

            "source":
                row["source"],

            "oof_fold":
                int(fold),

            "gt_has_lesion":
                bool(gt_has_lesion),

            "pred_has_lesion":
                bool(pred_has_lesion),

            "gt_lesion_instances":
                int(len(gt[1])),

            "pred_lesion_instances":
                int(
                    len(
                        pred_lesion_masks
                    )
                ),

            "pred_pa_instances":
                int(
                    len(
                        pred_pa_masks
                    )
                ),

            "missed_lesion":
                bool(missed_lesion),

            "false_lesion":
                bool(false_lesion),

            "lesion_iou":
                lesion_iou,

            "pa_iou":
                float(pa_iou),

            "severe_lesion_iou":
                bool(
                    severe_lesion_iou
                ),

            "low_lesion_iou":
                bool(
                    low_lesion_iou
                ),

            "low_pa_iou":
                bool(
                    low_pa_iou
                ),

            "mean_lesion_conf":
                (
                    float(
                        np.mean(
                            lesion_confs
                        )
                    )
                    if lesion_confs
                    else None
                ),

            "mean_pa_conf":
                (
                    float(
                        np.mean(
                            pa_confs
                        )
                    )
                    if pa_confs
                    else None
                ),

            "genuine_hard":
                bool(
                    genuine_hard
                ),

            "hard_score":
                float(
                    hard_score
                ),
        })


        del result
        del image
        del gt
        del gt_pa
        del gt_lesion
        del pred_pa
        del pred_lesion


    del prediction_stream
    del model

    cleanup_gpu()

    print(
        f"✅ Fold {fold} predictions complete."
    )


# ============================================================
# COMBINE
# ============================================================

df = pd.DataFrame(
    all_rows
)

assert len(df) == 865
assert df["filename"].nunique() == 865


df.to_csv(
    OOF_FAILURE_CSV,
    index=False
)


hard_df = df[
    df["genuine_hard"] == True
].copy()

hard_df = hard_df.sort_values(
    "hard_score",
    ascending=False
).reset_index(
    drop=True
)

hard_df.to_csv(
    OOF_HARD_CSV,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

lesion_df = df[
    df["gt_has_lesion"] == True
]

processing_only_df = df[
    df["gt_has_lesion"] == False
]


summary = {

    "total_images":
        int(len(df)),

    "lesion_positive":
        int(
            len(
                lesion_df
            )
        ),

    "processing_only":
        int(
            len(
                processing_only_df
            )
        ),

    "missed_lesion_images":
        int(
            df[
                "missed_lesion"
            ].sum()
        ),

    "false_lesion_images":
        int(
            df[
                "false_lesion"
            ].sum()
        ),

    "severe_lesion_iou_images":
        int(
            df[
                "severe_lesion_iou"
            ].sum()
        ),

    "low_lesion_iou_images":
        int(
            df[
                "low_lesion_iou"
            ].sum()
        ),

    "low_pa_iou_images":
        int(
            df[
                "low_pa_iou"
            ].sum()
        ),

    "genuine_hard_images":
        int(
            df[
                "genuine_hard"
            ].sum()
        ),

    "mean_lesion_iou":
        float(
            lesion_df[
                "lesion_iou"
            ].mean()
        ),

    "median_lesion_iou":
        float(
            lesion_df[
                "lesion_iou"
            ].median()
        ),

    "mean_pa_iou":
        float(
            df[
                "pa_iou"
            ].mean()
        ),
}


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 100)
print("TRUE OOF HARD-MINING SUMMARY")
print("=" * 100)

for k, v in summary.items():
    print(
        f"{k:<34}: {v}"
    )


# ============================================================
# SOURCE-WISE SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("SOURCE-WISE TRUE OOF FAILURES")
print("=" * 100)


source_rows = []


for source in sorted(
    df["source"].unique()
):

    s = df[
        df["source"] == source
    ]

    lesion_s = s[
        s["gt_has_lesion"] == True
    ]


    source_rows.append({

        "source":
            source,

        "images":
            len(s),

        "lesion_images":
            len(
                lesion_s
            ),

        "missed":
            int(
                s[
                    "missed_lesion"
                ].sum()
            ),

        "false_lesion":
            int(
                s[
                    "false_lesion"
                ].sum()
            ),

        "low_lesion_iou":
            int(
                s[
                    "low_lesion_iou"
                ].sum()
            ),

        "low_pa_iou":
            int(
                s[
                    "low_pa_iou"
                ].sum()
            ),

        "mean_lesion_iou":
            (
                lesion_s[
                    "lesion_iou"
                ].mean()
                if len(
                    lesion_s
                )
                else None
            ),

        "mean_pa_iou":
            s[
                "pa_iou"
            ].mean(),

        "hard_images":
            int(
                s[
                    "genuine_hard"
                ].sum()
            ),
    })


source_df = pd.DataFrame(
    source_rows
)


print(
    source_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# FOLD-WISE SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("FOLD-WISE TRUE OOF FAILURES")
print("=" * 100)


fold_rows = []


for fold in range(
    N_FOLDS
):

    s = df[
        df["oof_fold"] == fold
    ]

    lesion_s = s[
        s["gt_has_lesion"] == True
    ]


    fold_rows.append({

        "fold":
            fold,

        "images":
            len(s),

        "missed":
            int(
                s[
                    "missed_lesion"
                ].sum()
            ),

        "false_lesion":
            int(
                s[
                    "false_lesion"
                ].sum()
            ),

        "low_lesion_iou":
            int(
                s[
                    "low_lesion_iou"
                ].sum()
            ),

        "low_pa_iou":
            int(
                s[
                    "low_pa_iou"
                ].sum()
            ),

        "mean_lesion_iou":
            lesion_s[
                "lesion_iou"
            ].mean(),

        "mean_pa_iou":
            s[
                "pa_iou"
            ].mean(),

        "hard_images":
            int(
                s[
                    "genuine_hard"
                ].sum()
            ),
    })


fold_df = pd.DataFrame(
    fold_rows
)


print(
    fold_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# TOP HARD CASES
# ============================================================

print("\n" + "=" * 100)
print("TOP 30 TRUE OOF HARD CASES")
print("=" * 100)


cols = [

    "filename",

    "source",

    "oof_fold",

    "missed_lesion",

    "false_lesion",

    "lesion_iou",

    "pa_iou",

    "hard_score",
]


print(
    hard_df[
        cols
    ]
    .head(30)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("CELL 11 COMPLETE")
print("=" * 100)

print(f"""
All OOF predictions:
{OOF_FAILURE_CSV}

True hard cases:
{OOF_HARD_CSV}

Summary:
{SUMMARY_JSON}

Original validation:
NOT USED

Original test:
NOT USED

Every training image was evaluated by a model
that did NOT train on that image.

NEXT:
Use these genuine OOF failures to build the
final Y2 training strategy.
""")

print("=" * 100)
print("✅ MODULE 2 CELL 11 COMPLETE")
print("=" * 100)

AI_Y2 MODULE 2 — CELL 11
TRUE OOF HARD-CASE MINING
OOF assignment rows : 865
Folds               : [0, 1, 2, 3, 4]

Validation used:
NO

Test used:
NO


####################################################################################################
# OOF PREDICTION FOLD 0
####################################################################################################
Holdout images : 173
Model          : /content/AI_Y2_MODULE2/oof_models/fold_0_best.pt
✅ Fold 0 predictions complete.

####################################################################################################
# OOF PREDICTION FOLD 1
####################################################################################################
Holdout images : 171
Model          : /content/AI_Y2_MODULE2/oof_models/fold_1_best.pt
✅ Fold 1 predictions complete.

####################################################################################################
# OOF PREDICTION FOLD 2
################################

In [9]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 12
#
# FINAL OOF-WEIGHTED TRAIN DATASET
#
# Uses true OOF failures from Cell 11.
#
# Keeps:
# - all original 865 train images
# - one extra copy of selected genuinely hard images
#
# Validation 173 unchanged
# Test 115 unchanged
#
# Selection priority:
# 1. all missed lesions
# 2. all false-lesion processing-only images
# 3. all severe lesion IoU < 0.25
# 4. highest remaining hard-score cases with source caps
# ============================================================

from pathlib import Path
import shutil
import json
import pandas as pd


# ============================================================
# PATHS
# ============================================================

ORIGINAL_ROOT = Path(
    "/content/AI_Y2_FINAL"
)

OOF_CSV = Path(
    "/content/AI_Y2_MODULE2/reports/"
    "AI_Y2_true_oof_failure_analysis.csv"
)

FINAL_ROOT = Path(
    "/content/AI_Y2_FINAL_OOF_WEIGHTED"
)

REPORT_DIR = Path(
    "/content/AI_Y2_MODULE2/reports"
)

SELECTION_CSV = (
    REPORT_DIR /
    "AI_Y2_final_oof_weight_selection.csv"
)

SUMMARY_JSON = (
    REPORT_DIR /
    "AI_Y2_final_oof_weight_summary.json"
)


# ============================================================
# LOAD
# ============================================================

assert OOF_CSV.exists(), OOF_CSV

df = pd.read_csv(
    OOF_CSV
)

assert len(df) == 865


# ============================================================
# CRITICAL CASES — ALWAYS KEEP
# ============================================================

critical = df[
    (df["missed_lesion"] == True)
    |
    (df["false_lesion"] == True)
    |
    (df["severe_lesion_iou"] == True)
].copy()


critical_names = set(
    critical["filename"]
)


# ============================================================
# REMAINING GENUINE HARD CASES
# ============================================================

remaining = df[
    (df["genuine_hard"] == True)
    &
    (~df["filename"].isin(
        critical_names
    ))
].copy()


remaining = remaining.sort_values(
    "hard_score",
    ascending=False
)


# ============================================================
# SOURCE-AWARE CAPS
#
# Extra non-critical examples only.
#
# We intentionally emphasize the weaker domains:
# leukoplakia_like
# leukoplakia
# cancer
#
# ulcer already generalizes better.
# processing_area mostly clean.
# ============================================================

EXTRA_CAPS = {
    "leukoplakia_like": 55,
    "leukoplakia": 50,
    "cancer": 45,
    "ulcer": 25,
    "processing_area": 15,
}


extra_parts = []


for source, cap in EXTRA_CAPS.items():

    s = remaining[
        remaining["source"] == source
    ].copy()

    s = s.sort_values(
        "hard_score",
        ascending=False
    )

    extra_parts.append(
        s.head(cap)
    )


extra = pd.concat(
    extra_parts,
    ignore_index=True
)


# ============================================================
# FINAL SELECTION
# ============================================================

selected = pd.concat(
    [
        critical,
        extra
    ],
    ignore_index=True
)


selected = (
    selected
    .drop_duplicates(
        subset=["filename"]
    )
    .sort_values(
        [
            "hard_score",
            "source"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# AUDIT
# ============================================================

print("=" * 100)
print("AI_Y2 MODULE 2 — CELL 12")
print("FINAL TRUE-OOF WEIGHTED DATASET")
print("=" * 100)

print(
    f"Original train images      : {len(df)}"
)

print(
    f"True OOF hard images       : "
    f"{int(df['genuine_hard'].sum())}"
)

print(
    f"Critical selected          : "
    f"{len(critical)}"
)

print(
    f"Additional source-balanced : "
    f"{len(extra)}"
)

print(
    f"Final unique boost set     : "
    f"{len(selected)}"
)


print("\nFinal selected by source:")

print(
    selected[
        "source"
    ]
    .value_counts()
    .to_string()
)


print("\nCritical failure coverage:")

print(
    f"Missed lesions in full OOF : "
    f"{int(df['missed_lesion'].sum())}"
)

print(
    f"Missed selected            : "
    f"{int(selected['missed_lesion'].sum())}"
)

print(
    f"False lesions in full OOF  : "
    f"{int(df['false_lesion'].sum())}"
)

print(
    f"False selected             : "
    f"{int(selected['false_lesion'].sum())}"
)

print(
    f"Severe lesion IoU full     : "
    f"{int(df['severe_lesion_iou'].sum())}"
)

print(
    f"Severe selected            : "
    f"{int(selected['severe_lesion_iou'].sum())}"
)


# ============================================================
# SAFETY ASSERTIONS
# ============================================================

assert (
    int(selected["missed_lesion"].sum())
    ==
    int(df["missed_lesion"].sum())
)

assert (
    int(selected["false_lesion"].sum())
    ==
    int(df["false_lesion"].sum())
)

assert (
    int(selected["severe_lesion_iou"].sum())
    ==
    int(df["severe_lesion_iou"].sum())
)


# ============================================================
# SAVE SELECTION
# ============================================================

selected.to_csv(
    SELECTION_CSV,
    index=False
)


# ============================================================
# BUILD DATASET
# ============================================================

if FINAL_ROOT.exists():

    shutil.rmtree(
        FINAL_ROOT
    )


for split in [
    "train",
    "valid",
    "test"
]:

    (
        FINAL_ROOT /
        split /
        "images"
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    (
        FINAL_ROOT /
        split /
        "labels"
    ).mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# COPY ORIGINAL DATASET EXACTLY
# ============================================================

for split in [
    "train",
    "valid",
    "test"
]:

    src_img = (
        ORIGINAL_ROOT /
        split /
        "images"
    )

    src_lbl = (
        ORIGINAL_ROOT /
        split /
        "labels"
    )

    dst_img = (
        FINAL_ROOT /
        split /
        "images"
    )

    dst_lbl = (
        FINAL_ROOT /
        split /
        "labels"
    )


    for p in src_img.iterdir():

        if p.is_file():

            shutil.copy2(
                p,
                dst_img / p.name
            )


    for p in src_lbl.iterdir():

        if p.is_file():

            shutil.copy2(
                p,
                dst_lbl / p.name
            )


# ============================================================
# ADD ONE EXTRA COPY OF SELECTED TRAIN CASES
# ============================================================

TRAIN_IMG_DIR = (
    ORIGINAL_ROOT /
    "train" /
    "images"
)

TRAIN_LBL_DIR = (
    ORIGINAL_ROOT /
    "train" /
    "labels"
)


boost_count = 0


for _, row in selected.iterrows():

    filename = row[
        "filename"
    ]

    src_img = (
        TRAIN_IMG_DIR /
        filename
    )

    src_lbl = (
        TRAIN_LBL_DIR /
        f"{Path(filename).stem}.txt"
    )

    assert src_img.exists(), src_img
    assert src_lbl.exists(), src_lbl


    suffix = "__oofhard1"


    new_img_name = (
        f"{Path(filename).stem}"
        f"{suffix}"
        f"{Path(filename).suffix}"
    )

    new_lbl_name = (
        f"{Path(filename).stem}"
        f"{suffix}.txt"
    )


    shutil.copy2(
        src_img,
        FINAL_ROOT /
        "train" /
        "images" /
        new_img_name
    )

    shutil.copy2(
        src_lbl,
        FINAL_ROOT /
        "train" /
        "labels" /
        new_lbl_name
    )


    boost_count += 1


# ============================================================
# DATA.YAML
# ============================================================

yaml_text = f"""path: {FINAL_ROOT}
train: train/images
val: valid/images
test: test/images
nc: 2
names:
  - processing_area
  - lesion
"""


(
    FINAL_ROOT /
    "data.yaml"
).write_text(
    yaml_text,
    encoding="utf-8"
)


# ============================================================
# FINAL COUNTS
# ============================================================

counts = {}


for split in [
    "train",
    "valid",
    "test"
]:

    imgs = [
        p
        for p in (
            FINAL_ROOT /
            split /
            "images"
        ).iterdir()
        if p.is_file()
    ]

    lbls = list(
        (
            FINAL_ROOT /
            split /
            "labels"
        ).glob("*.txt")
    )


    assert len(imgs) == len(lbls)


    counts[
        split
    ] = len(imgs)


assert counts["valid"] == 173
assert counts["test"] == 115

assert (
    counts["train"]
    ==
    865 + boost_count
)


# ============================================================
# SUMMARY JSON
# ============================================================

summary = {

    "original_train":
        865,

    "true_oof_hard":
        int(
            df[
                "genuine_hard"
            ].sum()
        ),

    "critical_selected":
        int(
            len(
                critical
            )
        ),

    "extra_selected":
        int(
            len(
                extra
            )
        ),

    "final_boost_count":
        int(
            boost_count
        ),

    "final_train":
        int(
            counts[
                "train"
            ]
        ),

    "valid":
        173,

    "test":
        115,

    "selected_by_source":
        {
            str(k): int(v)
            for k, v in
            selected[
                "source"
            ]
            .value_counts()
            .items()
        },

    "dataset":
        str(
            FINAL_ROOT
        ),
}


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("CELL 12 FINAL SUMMARY")
print("=" * 100)

print(f"""
Dataset:
{FINAL_ROOT}

Original train:
865

OOF-selected boost:
{boost_count}

Final train:
{counts['train']}

Validation:
173 — unchanged

Test:
115 — unchanged

Classes:
0 = processing_area
1 = lesion

Selection CSV:
{SELECTION_CSV}

Summary JSON:
{SUMMARY_JSON}

✅ all OOF lesion misses included
✅ all OOF false lesions included
✅ all severe lesion-IoU cases included
✅ weaker domains emphasized
✅ validation untouched
✅ test untouched
✅ Y1 compatibility preserved
""")

print("=" * 100)
print("✅ MODULE 2 CELL 12 COMPLETE")
print("=" * 100)

AI_Y2 MODULE 2 — CELL 12
FINAL TRUE-OOF WEIGHTED DATASET
Original train images      : 865
True OOF hard images       : 473
Critical selected          : 166
Additional source-balanced : 179
Final unique boost set     : 345

Final selected by source:
source
leukoplakia_like    107
leukoplakia          97
cancer               78
ulcer                46
processing_area      17

Critical failure coverage:
Missed lesions in full OOF : 113
Missed selected            : 113
False lesions in full OOF  : 13
False selected             : 13
Severe lesion IoU full     : 153
Severe selected            : 153

CELL 12 FINAL SUMMARY

Dataset:
/content/AI_Y2_FINAL_OOF_WEIGHTED

Original train:
865

OOF-selected boost:
345

Final train:
1210

Validation:
173 — unchanged

Test:
115 — unchanged

Classes:
0 = processing_area
1 = lesion

Selection CSV:
/content/AI_Y2_MODULE2/reports/AI_Y2_final_oof_weight_selection.csv

Summary JSON:
/content/AI_Y2_MODULE2/reports/AI_Y2_final_oof_weight_summary.json

✅ all OO

In [10]:
# ============================================================
# AI_Y2 — MODULE 2 / CELL 13
#
# FINAL OOF-WEIGHTED Y2 CANDIDATE TRAINING
#
# START MODEL:
#   /content/ai_y2.pt
#
# TRAIN:
#   /content/AI_Y2_FINAL_OOF_WEIGHTED/train
#
# VALIDATION:
#   original unchanged 173 images
#
# TEST:
#   NOT USED
#
# IMPORTANT:
# - current ai_y2.pt will NOT be overwritten
# - mask_ratio stays 2
# - low-LR controlled fine-tuning
# ============================================================

from pathlib import Path
import gc
import json
import hashlib
import shutil
import torch

from ultralytics import YOLO


# ============================================================
# PATHS
# ============================================================

START_MODEL = Path(
    "/content/ai_y2.pt"
)

DATA_ROOT = Path(
    "/content/AI_Y2_FINAL_OOF_WEIGHTED"
)

DATA_YAML = (
    DATA_ROOT /
    "data.yaml"
)

RUN_ROOT = Path(
    "/content/AI_Y2_MODULE2/runs"
)

RUN_NAME = (
    "AI_Y2_final_oof_weighted"
)

RUN_DIR = (
    RUN_ROOT /
    RUN_NAME
)

CANDIDATE_OUT = Path(
    "/content/AI_Y2_MODULE2/"
    "candidate_y2_oof_weighted.pt"
)

REPORT_DIR = Path(
    "/content/AI_Y2_MODULE2/reports"
)

REPORT_JSON = (
    REPORT_DIR /
    "AI_Y2_oof_weighted_candidate_validation.json"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CONTRACT
# ============================================================

EXPECTED_NAMES = {
    0: "processing_area",
    1: "lesion"
}


# ============================================================
# HELPERS
# ============================================================

def cleanup_gpu():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


# ============================================================
# VERIFY DATASET
# ============================================================

assert START_MODEL.exists(), START_MODEL
assert DATA_YAML.exists(), DATA_YAML


expected_counts = {

    "train": 1210,
    "valid": 173,
    "test": 115,
}


print("=" * 100)
print("AI_Y2 MODULE 2 — CELL 13")
print("FINAL OOF-WEIGHTED CANDIDATE TRAINING")
print("=" * 100)


for split, expected in expected_counts.items():

    image_dir = (
        DATA_ROOT /
        split /
        "images"
    )

    label_dir = (
        DATA_ROOT /
        split /
        "labels"
    )

    images = [
        p
        for p in image_dir.iterdir()
        if p.is_file()
    ]

    labels = list(
        label_dir.glob(
            "*.txt"
        )
    )

    print(
        f"{split:<5} : "
        f"images={len(images)} "
        f"labels={len(labels)}"
    )

    assert len(images) == expected
    assert len(labels) == expected


# ============================================================
# GPU
# ============================================================

cleanup_gpu()

assert torch.cuda.is_available()

print(
    f"\nGPU       : "
    f"{torch.cuda.get_device_name(0)}"
)

print(
    f"Allocated : "
    f"{torch.cuda.memory_allocated()/1024**3:.3f} GB"
)

print(
    f"Reserved  : "
    f"{torch.cuda.memory_reserved()/1024**3:.3f} GB"
)


# ============================================================
# VERIFY START MODEL
# ============================================================

check = YOLO(
    str(START_MODEL)
)

names = {
    int(k): str(v)
    for k, v in check.names.items()
}

assert names == EXPECTED_NAMES
assert str(check.task) == "segment"

print(
    f"\nStart model : {START_MODEL}"
)

print(
    f"Task        : {check.task}"
)

print(
    f"Classes     : {names}"
)

del check

cleanup_gpu()


# ============================================================
# REMOVE ONLY OLD PARTIAL RUN OF THIS EXACT EXPERIMENT
# ============================================================

if RUN_DIR.exists():

    print(
        "\nRemoving previous partial Cell-13 run..."
    )

    shutil.rmtree(
        RUN_DIR
    )


# ============================================================
# LOAD CURRENT STRONG Y2
# ============================================================

model = YOLO(
    str(START_MODEL)
)


# ============================================================
# TRAIN
# ============================================================

print("\n" + "=" * 100)
print("STARTING FINAL OOF-WEIGHTED REFINEMENT")
print("=" * 100)


model.train(

    # --------------------------------------------------------
    # DATA / HARDWARE
    # --------------------------------------------------------

    data=str(
        DATA_YAML
    ),

    imgsz=640,

    batch=16,

    device=0,

    workers=8,

    cache="ram",

    amp=True,


    # --------------------------------------------------------
    # CONTROLLED FINETUNING
    # --------------------------------------------------------

    epochs=120,

    patience=25,


    # --------------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------------

    optimizer="AdamW",

    lr0=0.00010,

    lrf=0.05,

    weight_decay=0.0005,

    cos_lr=True,

    warmup_epochs=2.0,


    # --------------------------------------------------------
    # SEGMENTATION
    #
    # IMPORTANT:
    # mask_ratio=1 experiment was worse.
    # Keep proven ratio=2.
    # --------------------------------------------------------

    mask_ratio=2,

    overlap_mask=True,


    # --------------------------------------------------------
    # MILD CLINICAL IMAGE AUGMENTATION
    # --------------------------------------------------------

    degrees=4.0,

    translate=0.02,

    scale=0.06,

    shear=0.0,

    perspective=0.0,

    flipud=0.0,

    fliplr=0.50,

    hsv_h=0.005,

    hsv_s=0.10,

    hsv_v=0.12,


    # --------------------------------------------------------
    # VERY LIGHT MOSAIC
    # --------------------------------------------------------

    mosaic=0.05,

    close_mosaic=20,

    mixup=0.0,

    copy_paste=0.0,


    # --------------------------------------------------------
    # REPRODUCIBILITY
    # --------------------------------------------------------

    seed=42,

    deterministic=True,


    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    project=str(
        RUN_ROOT
    ),

    name=RUN_NAME,

    exist_ok=True,

    plots=False,

    save=True,

    save_period=-1,

    verbose=True
)


# ============================================================
# BEST MODEL
# ============================================================

best_path = (
    RUN_DIR /
    "weights" /
    "best.pt"
)

assert best_path.exists(), (
    f"best.pt missing:\n"
    f"{best_path}"
)


shutil.copy2(
    best_path,
    CANDIDATE_OUT
)


del model

cleanup_gpu()


# ============================================================
# VERIFY FINAL CANDIDATE CONTRACT
# ============================================================

candidate = YOLO(
    str(CANDIDATE_OUT)
)

candidate_names = {
    int(k): str(v)
    for k, v in candidate.names.items()
}

assert candidate_names == EXPECTED_NAMES
assert str(candidate.task) == "segment"


print("\n" + "=" * 100)
print("CANDIDATE CREATED")
print("=" * 100)

print(
    f"Candidate : "
    f"{CANDIDATE_OUT}"
)

print(
    f"Classes   : "
    f"{candidate_names}"
)

print(
    f"Task      : "
    f"{candidate.task}"
)


# ============================================================
# EXACT APPLES-TO-APPLES VALIDATION
#
# Evaluate BOTH:
# 1. existing current Y2
# 2. new OOF candidate
#
# SAME:
# - 173 validation images
# - imgsz=640
# - conf=.25
# - iou=.70
#
# TEST NOT USED.
# ============================================================

print("\n" + "=" * 100)
print("APPLE-TO-APPLE VALIDATION")
print("=" * 100)

print("""
Validation images : 173
Image size        : 640
Confidence        : 0.25
IoU               : 0.70

TEST USED:
NO
""")


# ------------------------------------------------------------
# BASELINE
# ------------------------------------------------------------

baseline = YOLO(
    str(START_MODEL)
)

baseline_metrics = baseline.val(

    data=str(
        DATA_YAML
    ),

    split="val",

    imgsz=640,

    conf=0.25,

    iou=0.70,

    batch=8,

    device=0,

    plots=False,

    save_json=False,

    verbose=False
)


# ------------------------------------------------------------
# CANDIDATE
# ------------------------------------------------------------

candidate_metrics = candidate.val(

    data=str(
        DATA_YAML
    ),

    split="val",

    imgsz=640,

    conf=0.25,

    iou=0.70,

    batch=8,

    device=0,

    plots=False,

    save_json=False,

    verbose=False
)


# ============================================================
# EXTRACT
# ============================================================

def extract_metrics(m):

    return {

        "mask_precision":
            float(
                m.seg.mp
            ),

        "mask_recall":
            float(
                m.seg.mr
            ),

        "mask_map50":
            float(
                m.seg.map50
            ),

        "mask_map50_95":
            float(
                m.seg.map
            ),

        "processing_mask_map50_95":
            float(
                m.seg.maps[0]
            ),

        "lesion_mask_map50_95":
            float(
                m.seg.maps[1]
            ),

        "box_precision":
            float(
                m.box.mp
            ),

        "box_recall":
            float(
                m.box.mr
            ),

        "box_map50":
            float(
                m.box.map50
            ),

        "box_map50_95":
            float(
                m.box.map
            ),

        "processing_box_map50_95":
            float(
                m.box.maps[0]
            ),

        "lesion_box_map50_95":
            float(
                m.box.maps[1]
            ),
    }


base_report = extract_metrics(
    baseline_metrics
)

candidate_report = extract_metrics(
    candidate_metrics
)


# ============================================================
# DELTAS
# ============================================================

deltas = {

    k:
        candidate_report[k]
        -
        base_report[k]

    for k in base_report.keys()
}


# ============================================================
# REPORT
# ============================================================

report = {

    "baseline_model":
        str(
            START_MODEL
        ),

    "candidate_model":
        str(
            CANDIDATE_OUT
        ),

    "validation_images":
        173,

    "test_used":
        False,

    "candidate_sha256":
        sha256_file(
            CANDIDATE_OUT
        ),

    "baseline":
        base_report,

    "candidate":
        candidate_report,

    "delta_candidate_minus_baseline":
        deltas,
}


with open(
    REPORT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2
    )


# ============================================================
# DISPLAY COMPARISON
# ============================================================

print("\n" + "=" * 100)
print("CURRENT Y2 vs OOF-WEIGHTED CANDIDATE")
print("=" * 100)

print(
    f"{'METRIC':<32}"
    f"{'CURRENT':>12}"
    f"{'CANDIDATE':>14}"
    f"{'DELTA':>12}"
)

print("-" * 100)


for metric in [

    "mask_precision",
    "mask_recall",
    "mask_map50",
    "mask_map50_95",

    "processing_mask_map50_95",
    "lesion_mask_map50_95",

    "box_precision",
    "box_recall",
    "box_map50",
    "box_map50_95",

    "processing_box_map50_95",
    "lesion_box_map50_95",

]:

    print(

        f"{metric:<32}"

        f"{base_report[metric]:>12.6f}"

        f"{candidate_report[metric]:>14.6f}"

        f"{deltas[metric]:>+12.6f}"
    )


# ============================================================
# FINAL INFO
# ============================================================

print("\n" + "=" * 100)
print("CELL 13 FINAL SUMMARY")
print("=" * 100)

print(f"""
Current Y2:
{START_MODEL}

New OOF candidate:
{CANDIDATE_OUT}

Validation report:
{REPORT_JSON}

Train:
1210 weighted images

Validation:
173 unchanged

Test:
115 untouched

Class contract:
0 = processing_area
1 = lesion

Current ai_y2.pt overwritten:
NO

TEST USED:
NO

IMPORTANT:
Do NOT promote candidate automatically.

Next:
Check lesion detection + false positives +
boundary quality on the same validation set,
then decide final Y2.
""")

print("=" * 100)
print("✅ MODULE 2 CELL 13 COMPLETE")
print("=" * 100)


# ============================================================
# CLEANUP
# ============================================================

del baseline
del candidate
del baseline_metrics
del candidate_metrics

cleanup_gpu()

AI_Y2 MODULE 2 — CELL 13
FINAL OOF-WEIGHTED CANDIDATE TRAINING
train : images=1210 labels=1210
valid : images=173 labels=173
test  : images=115 labels=115

GPU       : NVIDIA A100-SXM4-80GB
Allocated : 0.062 GB
Reserved  : 1.186 GB

Start model : /content/ai_y2.pt
Task        : segment
Classes     : {0: 'processing_area', 1: 'lesion'}

STARTING FINAL OOF-WEIGHTED REFINEMENT
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/AI_Y2_FINAL_OOF_WEIGHTED/data.yaml, degrees=4.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, e

In [11]:
from google.colab import files

MODEL_PATH = "/content/ai_y2.pt"

files.download(MODEL_PATH)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>